<a href="https://colab.research.google.com/gist/pctablet505/e5bdc764cc465bd992cc2d4a56242992/fedrated_learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!git clone https://github.com/pctablet505/BalanceFL.git balancedfl_repo

Cloning into 'balancedfl_repo'...
remote: Enumerating objects: 152, done.
remote: Counting objects: 100% (152/152), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 152 (delta 79), reused 135 (delta 71), pack-reused 0 (from 0)
Receiving objects: 100% (152/152), 719.69 KiB | 948.00 KiB/s, done.
Resolving deltas: 100% (79/79), done.


In [2]:
%cd /content/balancedfl_repo/
!git pull

/content/balancedfl_repo
Already up to date.


In [3]:
# 2. Get training command for your DeepFed model
!python CIFAR10/simple_train.py --config CIFAR10/config/deepfed.yaml --model deepfed

Training deepfed with config CIFAR10/config/deepfed.yaml
Experiment: simple_exp
Config loaded:
- Dataset: DeepFed
- Model: DeepFed
- Classes: 8
- Output: ./runs_simple/simple_exp
- Device: cuda
- Mode: Federated (5 clients, 100 rounds)

TRAINING STARTED
For federated training, run the original scripts:
python train_ours.py --cfg CIFAR10/config/deepfed.yaml --exp_name simple_exp
python train_fedavg.py --cfg CIFAR10/config/deepfed.yaml --exp_name simple_exp
python train_fedprox.py --cfg CIFAR10/config/deepfed.yaml --exp_name simple_exp
Use the commands above to start actual training!

Next steps:
1. Check the suggested command above
2. Results will be saved in: ./runs_simple/simple_exp
3. Use tensorboard to view training: tensorboard --logdir ./runs_simple/simple_exp


In [4]:
!python CIFAR10/train_ours.py --cfg CIFAR10/config/deepfed.yaml --exp_name simple_exp

2025-11-10 03:53:41.200448: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762746821.221386    5586 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762746821.227436    5586 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1762746821.243791    5586 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762746821.243821    5586 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1762746821.243826    5586 computation_placer.cc:177] computation placer alr

In [5]:
!head -n 80 CIFAR10/train_ours.py

import os
import numpy as np
import argparse
import yaml
import shutil
from pathlib import Path

import torch

# torch.backends.cudnn.benchmark = True
from torch.utils.tensorboard import SummaryWriter

from utils.misc import update_config, deterministic
from utils.logger import Logger, print_write, write_summary
from data import dataloader
from models.utils import *
from fed import Fed_client, Fed_server


"""
Script for original federated training (train only a global model). 
Support different datasets/losses/samplings. 
"""


if __name__ == "__main__":

    data_root_dict = {
        "CIFAR10": "../dataset/cifar_10",
    }

    """
    Parameters
    """
    # important parameters
    parser = argparse.ArgumentParser()
    parser.add_argument("--cfg", default="config/fedlt.yaml", type=str)
    parser.add_argument("--exp_name", default="ours", type=str, help="exp name")
    parser.add_argument(
        "--non_iidness", default=1, type=int, help="non-iid degree of distributed data"
  

In [ ]:
!sed -i '/imb_ratio:/a\  tao_ratio: 0.0' CIFAR10/config/deepfed.yaml

In [ ]:
!sed -i '/exp_name:/a\  work_dir: ./runs_simple/simple_exp' CIFAR10/config/deepfed.yaml

In [ ]:
!sed -i '/metainfo:/a\  exp_name: simple_exp' CIFAR10/config/deepfed.yaml

In [ ]:
!cat CIFAR10/config/deepfed.yaml

fl_opt:
  rounds: 100
  num_clients: 5
  frac: 1.0         # the fraction of clients in each FL round
  local_ep: 10      # local epoch
  local_bs: 32      # local batch size
  aggregation: fedavg  # fedavg, fedavgm, fedbn, fedfs

criterions:
  def_file: ./loss/KDLoss.py    # for cross entropy
  loss_params: {Temp: 2, lamda: 0, loss_cls: ce, loss_kd: kl}

networks:
  feat_model:
    def_file: DeepFed
    params: 
      input_shape: [26, 1]
      gru_hidden_size: 128
    optim_params: {lr: 0.001, momentum: 0.9, weight_decay: 0.0001}
    feat_dim: 512
    fix: false
  classifier:
    def_file: DotProductClassifier
    params: {num_classes: 8, l2_norm: false, bias: true, scale: 1}   
    optim_params: {lr: 0.001, momentum: 0.9, weight_decay: 0.0001} 
    fix: false
    
dataset:
  name: DeepFed
  num_classes: 8
  shot_few: 0   # if shot_few>0, (iid + non-iid) mixed mode; else, non-mixed mode
  non_iidness: 1
  imb_ratio: 1.0  # balanced data for intrusion detection
  tao_ratio: 0.0

train

In [ ]:
!pip install pyarrow

In [ ]:
!rm -rf /content/balancedfl_repo/cache

In [ ]:
"""
DeepFed: Intrusion Detection with Imbalanced Dataset Handling
==============================================================

Research-focused implementation comparing different approaches to handle
highly imbalanced network attack datasets (majority Normal, minority Attacks).

COMPARISON APPROACHES:
1. Baseline: Train on imbalanced data (no handling)
2. Class Weights: Penalize misclassification of minority classes
3. SMOTE Oversampling: Synthetic minority oversampling technique
4. Random Undersampling: Random undersampling of majority class
5. Combined: SMOTE + Tomek links cleaning

Dataset: Edge-IIoTset - Cyber Security Dataset of IoT & IIoT
Model: DeepFed (GRU + CNN parallel architecture)

Based on DeepFed paper: "DeepFed: Federated Deep Learning for Intrusion Detection
in Industrial Cyber-Physical Systems"

INSTALLATION:
-------------
For full functionality (all 5 approaches), install imbalanced-learn:
    pip install imbalanced-learn

Without imbalanced-learn, only approaches 1 and 2 will run.

USAGE:
------
python deepfed_train.py

The script will:
- Load and preprocess Edge-IIoTset dataset
- Train 5 models with different imbalance handling approaches
- Evaluate and compare all approaches
- Generate comparison visualizations
- Save results to models/deepfed/ and visualizations/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import subprocess
import sys
import pickle
from collections import Counter
import json

# Use Keras 3 (not tensorflow.keras)
import keras
from keras import layers, models, callbacks, ops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OrdinalEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for handling imbalanced datasets
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine import SMOTETomek
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("WARNING: imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("         Only baseline and class weights approaches will be available.")
    IMBLEARN_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

# Check for required packages
try:
    import tables  # For HDF5 support
except ImportError:
    print("ERROR: pytables is required for HDF5 export. Please install with `pip install tables`.")
    sys.exit(1)

# Configuration
DATASET_NAME = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
DATA_DIR = "./data/edge_iiot"
MODEL_DIR = "./models/deepfed"
VIS_DIR = "./visualizations"
CACHE_DIR = "./cache"
PREPROCESSED_DIR = "./cache/preprocessed"  # For efficient binary format
HDF5_DATASET = Path(PREPROCESSED_DIR) / "dataset.h5"
BATCH_SIZE = 128
EPOCHS = 50
LEARNING_RATE = 0.001
RANDOM_STATE = 42
SEQUENCE_LENGTH = 128  # Time steps for time-series sequences (set None for dynamic windows)
WINDOW_STRIDE = 64  # Stride for sliding window sequence creation
VALIDATION_SPLIT = 0.2
SAMPLE_SIZE = 100000  # Use a small sample for debugging; set to None for full dataset
USE_MULTICLASS = True  # Use multi-class attack type classification
USE_CACHED_DATA = True  # Use preprocessed binary data if available

# Set random seeds
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Create directories
for dir_path in [DATA_DIR, MODEL_DIR, VIS_DIR, CACHE_DIR, PREPROCESSED_DIR]:
    os.makedirs(dir_path, exist_ok=True)


def download_dataset():
    """Download Edge-IIoTset dataset from Kaggle"""
    print("\n" + "=" * 80)
    print("DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE")
    print("=" * 80)

    # Setup Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        print("✓ Running in Google Colab - using secrets")

        # Get credentials from Colab secrets
        kaggle_username = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')

        if not kaggle_username or not kaggle_key:
            raise ValueError("KAGGLE_USERNAME and KAGGLE_KEY must be set in Colab secrets")

        # Set environment variables for Kaggle API
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key

        print(f"  • Username: {kaggle_username}")
        print(f"  • API Key: {'*' * len(kaggle_key)}")

    except ImportError:
        print("✓ Not running in Colab - using default kaggle.json authentication")
    except Exception as e:
        print(f"✗ Error setting up Kaggle credentials: {e}")
        print("\nPlease add these secrets in Colab:")
        print("  1. Click the key icon (🔑) in the left sidebar")
        print("  2. Add secret: KAGGLE_USERNAME")
        print("  3. Add secret: KAGGLE_KEY")
        print("\nGet your credentials from: https://www.kaggle.com/settings/account")
        raise

    try:
        import kaggle
    except ImportError:
        print("Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
        import kaggle

    try:
        print(f"\nDownloading {DATASET_NAME}...")
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET_NAME, "-p", DATA_DIR
        ], check=True)

        # Extract zip files
        for zip_file in Path(DATA_DIR).glob("*.zip"):
            print(f"Extracting {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            zip_file.unlink()

        print("✓ Dataset downloaded and extracted successfully!")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        print(f"\nPlease download manually from:")
        print(f"https://www.kaggle.com/datasets/{DATASET_NAME}")
        return False


def convert_csv_to_binary():
    """
    Convert CSV files to a consolidated Parquet dataset while preserving all features.
    Adds source metadata and derived temporal features for downstream processing.
    """
    preprocessed_file = HDF5_DATASET

    if preprocessed_file.exists() and USE_CACHED_DATA:
        print("\n" + "=" * 80)
        print("✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING")
        print("=" * 80)
        print(f"Using cached file: {preprocessed_file}")
        print(f"Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")
        return preprocessed_file

    print("\n" + "=" * 80)
    print("CONVERTING CSV TO EFFICIENT BINARY FORMAT")
    print("=" * 80)

    # Find all CSV files
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV files found!")

    print(f"\n✓ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

    # Load and combine all CSVs
    if SAMPLE_SIZE:
        print(f"\nLoading sample data (max {SAMPLE_SIZE:,} rows per file)...")
    else:
        print(f"\nLoading FULL dataset (this may take a while)...")

    dfs = []
    manifest = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, nrows=SAMPLE_SIZE, low_memory=False)
            original_rows = len(df)

            # Normalize string columns to avoid mixed dtype issues
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            for col in object_cols:
                df[col] = df[col].astype(str).str.strip().fillna('__NA__')

            # Attach source metadata
            df['source_file'] = csv_file.name
            df['source_path'] = str(csv_file.relative_to(DATA_DIR))
            df['source_category'] = csv_file.parent.name

            # Derive attack type from filename
            filename = csv_file.name
            if '_attack.csv' in filename:
                attack_type = filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                attack_type = 'Mixed'  # These contain multiple attack types
            else:
                attack_type = 'Normal'  # Sensor data files
            df['Attack_type'] = attack_type

            # Build temporal features without dropping original column
            if 'frame.time' in df.columns:
                time_str = df['frame.time'].astype(str).str.strip()
                parsed_time = pd.to_datetime(time_str, format='%Y %H:%M:%S.%f', errors='coerce')
                if parsed_time.isna().all():
                    parsed_time = pd.to_datetime(time_str, errors='coerce')
                parsed_time = parsed_time.fillna(method='ffill').fillna(method='bfill')
                df['frame_time_datetime'] = parsed_time
                base_time = parsed_time.iloc[0]
                rel_seconds = (parsed_time - base_time).dt.total_seconds()
                df['frame_time_relative_sec'] = rel_seconds.astype('float64')

            dfs.append(df)
            duration = float(df.get('frame_time_relative_sec', pd.Series([0])).max()) if len(df) else 0.0
            manifest.append({
                'file': str(csv_file.relative_to(DATA_DIR)),
                'rows_loaded': int(original_rows),
                'duration_seconds': duration
            })
            print(f"  ✓ {csv_file.name}: {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading {csv_file.name}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*80}")
    print(f"Combined dataset: {len(df):,} rows × {len(df.columns)} columns")
    print(f"{'='*80}")

    print("\nPreparing data for HDF5 storage...")
    df_filtered = df.copy()
    print(f"Final dataset shape: {df_filtered.shape}")
    print(f"Memory usage: {df_filtered.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    preprocessed_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle HDF5 file locking issues (common in Colab)
    if preprocessed_file.exists():
        print(f"  → HDF5 file exists, removing to recreate...")
        try:
            preprocessed_file.unlink()  # Remove the existing file
            print("  ✓ Removed existing HDF5 file")
        except Exception as e:
            print(f"  ✗ Could not remove existing file: {e}")
            print("  → Please manually delete the file and try again")
            raise RuntimeError(f"Cannot remove existing HDF5 file {preprocessed_file}. Please delete it manually.")

    # Save to HDF5
    df_filtered.to_hdf(preprocessed_file, key='data', mode='w', index=False)
    print(f"✓ Saved: {preprocessed_file}")
    print(f"  Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")

    total_csv_size = sum(f.stat().st_size for f in csv_files)
    compression_ratio = (1 - preprocessed_file.stat().st_size / total_csv_size) * 100
    print(f"  Compression: {compression_ratio:.1f}% savings over CSV")
    print(f"  Original CSV size: {total_csv_size / 1024**2:.1f} MB")

    manifest_path = Path(PREPROCESSED_DIR) / 'ingest_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest saved: {manifest_path}")

    return preprocessed_file


def explore_dataset():
    """
    Phase 1: Deep exploration and visualization of Edge-IIoTset dataset
    Uses preprocessed binary format for fast loading
    """
    print("\n" + "=" * 80)
    print("PHASE 1: DATASET EXPLORATION & VISUALIZATION")
    print("=" * 80)

    # Convert CSV to binary format (only once)
    preprocessed_file = convert_csv_to_binary()

    # Load from efficient binary format
    print(f"\nLoading preprocessed data...")
    df = pd.read_hdf(preprocessed_file, key='data')

    print(f"✓ Loaded {len(df):,} rows × {len(df.columns)} columns")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    # Basic dataset info
    print("\n1. DATASET STRUCTURE")
    print("-" * 80)
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nColumn types:")
    print(df.dtypes.value_counts())

    print(f"\nFirst few columns:")
    for i, col in enumerate(df.columns[:10], 1):
        print(f"  {i:2d}. {col:40s} ({df[col].dtype})")
    if len(df.columns) > 10:
        print(f"  ... and {len(df.columns) - 10} more columns")

    # The cached Parquet dataset preserves all columns, so identify label column dynamically
    # Label column is typically categorical with a limited number of unique values
    # For Edge-IIoTset, prioritize 'Attack_type' column
    if 'Attack_type' not in df.columns and 'source_file' in df.columns:
        # Derive Attack_type from source_file if missing
        def derive_attack_type(filename):
            if '_attack.csv' in filename:
                return filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                return 'Mixed'  # These contain multiple attack types
            else:
                return 'Normal'  # Sensor data files
        df['Attack_type'] = df['source_file'].apply(derive_attack_type)
        print(f"\n✓ Derived 'Attack_type' column from 'source_file'")

    if 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"\n✓ Label column identified: '{label_col}' (Attack_type)")
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        potential_labels = [col for col in df.columns if col not in numeric_cols]

        if potential_labels:
            # Use the first potential label column
            label_col = potential_labels[0]
            print(f"\n✓ Label column identified: '{label_col}'")
        else:
            # If all columns are numeric, assume the last column is the label
            # (common convention in ML datasets)
            label_col = df.columns[-1]
            print(f"\n✓ Label column (assumed): '{label_col}' (last column)")

    # Determine if it's multi-class or binary
    unique_labels = df[label_col].nunique()
    if unique_labels > 2 or USE_MULTICLASS:
        print(f"✓ Using MULTI-CLASS classification ({unique_labels} classes)")
    else:
        print(f"✓ Using BINARY classification ({unique_labels} classes)")

    # Class distribution analysis
    print("\n2. CLASS DISTRIBUTION")
    print("-" * 80)
    class_counts = df[label_col].value_counts()
    print(f"Number of classes: {len(class_counts)}")

    # Show all classes if multi-class, otherwise show binary
    print(f"\nClass distribution:")
    if len(class_counts) <= 20:
        # Show all classes
        for class_name, count in class_counts.items():
            pct = count / len(df) * 100
            bar = '█' * min(int(pct), 50)  # Cap bar at 50 chars
            print(f"  {str(class_name):30s}: {count:8,} ({pct:5.2f}%) {bar}")
    else:
        # Show top 20 classes
        for class_name, count in class_counts.head(20).items():
            pct = count / len(df) * 100
            bar = '█' * min(int(pct), 50)
            print(f"  {str(class_name):30s}: {count:8,} ({pct:5.2f}%) {bar}")
        print(f"  ... and {len(class_counts) - 20} more classes")

    # Calculate class imbalance ratio
    max_count = class_counts.max()
    min_count = class_counts.min()
    imbalance_ratio = max_count / min_count
    print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}:1")
    print(f"  → This is a {'highly ' if imbalance_ratio > 100 else ''}imbalanced dataset!")

    # Visualize class distribution
    if len(class_counts) <= 20:
        # Show all classes
        plt.figure(figsize=(16, 8))
        plt.subplot(1, 2, 1)
        class_counts.plot(kind='bar')
        plt.title('Class Distribution (Count)', fontsize=14, fontweight='bold')
        plt.xlabel('Attack Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

        plt.subplot(1, 2, 2)
        # Only show pie chart if <=10 classes (too messy otherwise)
        if len(class_counts) <= 10:
            class_counts.plot(kind='pie', autopct='%1.1f%%')
            plt.title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
            plt.ylabel('')
        else:
            class_counts.head(10).plot(kind='pie', autopct='%1.1f%%')
            plt.title('Top 10 Classes (Percentage)', fontsize=14, fontweight='bold')
            plt.ylabel('')
        plt.tight_layout()
    else:
        # Too many classes - show top 20 only
        plt.figure(figsize=(16, 6))
        class_counts.head(20).plot(kind='bar')
        plt.title('Top 20 Classes Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Attack Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

    plt.savefig(Path(VIS_DIR) / 'class_distribution.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved visualization: {Path(VIS_DIR) / 'class_distribution.png'}")
    plt.close()

    # Feature analysis
    print("\n3. FEATURE ANALYSIS")
    print("-" * 80)

    # Separate numeric and non-numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Remove label from feature lists
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    if label_col in non_numeric_cols:
        non_numeric_cols.remove(label_col)

    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Non-numeric features: {len(non_numeric_cols)}")

    if non_numeric_cols:
        print(f"\nNon-numeric columns (will be ordinal-encoded for time-series modeling):")
        for col in non_numeric_cols[:10]:
            unique_vals = df[col].nunique()
            print(f"  - {col:40s} ({unique_vals} unique values)")
        if len(non_numeric_cols) > 10:
            print(f"  ... and {len(non_numeric_cols) - 10} more")

    # Missing values
    print(f"\n4. DATA QUALITY")
    print("-" * 80)
    missing = df[numeric_cols].isnull().sum()
    if missing.sum() > 0:
        print(f"Missing values:")
        for col, count in missing[missing > 0].items():
            pct = count / len(df) * 100
            print(f"  {col:40s}: {count:,} ({pct:.2f}%)")
    else:
        print("✓ No missing values in numeric features")

    # Infinite values
    inf_counts = {}
    for col in numeric_cols[:20]:  # Check first 20 numeric columns
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            inf_counts[col] = inf_count

    if inf_counts:
        print(f"\nInfinite values detected:")
        for col, count in list(inf_counts.items())[:10]:
            print(f"  {col:40s}: {count:,}")
    else:
        print("✓ No infinite values in numeric features (checked first 20)")

    # Feature statistics
    print(f"\n5. FEATURE STATISTICS (first 10 numeric features)")
    print("-" * 80)
    feature_stats = df[numeric_cols[:10]].describe()
    print(feature_stats.to_string())

    # Correlation analysis (sample)
    print(f"\n6. CORRELATION ANALYSIS")
    print("-" * 80)
    print("Computing correlations for sample features...")

    # Select a subset of features for correlation
    sample_features = numeric_cols[:20] if len(numeric_cols) > 20 else numeric_cols
    corr_matrix = df[sample_features].corr()

    # Find highly correlated features
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.9:
                high_corr.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i, j]
                ))

    if high_corr:
        print(f"Found {len(high_corr)} highly correlated feature pairs (|r| > 0.9):")
        for feat1, feat2, corr_val in high_corr[:5]:
            print(f"  {feat1:30s} ↔ {feat2:30s}: {corr_val:.3f}")
        if len(high_corr) > 5:
            print(f"  ... and {len(high_corr) - 5} more pairs")
    else:
        print("No highly correlated feature pairs found (in sampled features)")

    # Visualize correlation matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title(f'Feature Correlation Matrix (first {len(sample_features)} features)',
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization: {Path(VIS_DIR) / 'correlation_matrix.png'}")
    plt.close()

    # Feature distributions by class
    print(f"\n7. FEATURE DISTRIBUTIONS BY CLASS")
    print("-" * 80)
    print("Visualizing feature distributions for sample features...")

    # Select interesting features to visualize
    viz_features = sample_features[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, col in enumerate(viz_features):
        if idx >= len(axes):
            break

        # Sample data for visualization
        sample_data = df[[col, label_col]].sample(min(10000, len(df)))

        for class_name in sample_data[label_col].unique()[:5]:  # Top 5 classes
            class_data = sample_data[sample_data[label_col] == class_name][col]
            axes[idx].hist(class_data, bins=50, alpha=0.5, label=str(class_name)[:20])

        axes[idx].set_xlabel(col if len(col) < 30 else col[:27] + '...')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend(fontsize=8)
        axes[idx].set_title(f'{col[:40]}', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'feature_distributions.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization: {Path(VIS_DIR) / 'feature_distributions.png'}")
    plt.close()

    # Time-series characteristics
    print(f"\n8. TIME-SERIES CHARACTERISTICS")
    print("-" * 80)
    print(f"For time-series modeling, we will:")
    print(f"  • Use sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Create sliding windows from the data")
    print(f"  • Each sequence will have shape: ({SEQUENCE_LENGTH}, {len(numeric_cols)})")
    print(f"  • Model architecture: GRU + CNN (as per DeepFed paper)")

    # Save exploration metadata
    exploration_meta = {
        'num_samples': len(df),
        'num_features': len(numeric_cols),
        'num_classes': len(class_counts),
        'class_names': [str(c) for c in class_counts.index.tolist()],
        'class_counts': {str(k): int(v) for k, v in class_counts.to_dict().items()},
        'imbalance_ratio': float(imbalance_ratio),
        'label_column': label_col,
        'classification_type': 'multi-class' if len(class_counts) > 2 else 'binary',
        'numeric_columns': numeric_cols,
        'non_numeric_columns': non_numeric_cols,
        'sequence_length': SEQUENCE_LENGTH,
        'window_stride': WINDOW_STRIDE,
        'using_full_dataset': SAMPLE_SIZE is None
    }

    with open(Path(CACHE_DIR) / 'exploration_metadata.json', 'w') as f:
        json.dump(exploration_meta, f, indent=2)

    print(f"\n✓ Saved exploration metadata: {Path(CACHE_DIR) / 'exploration_metadata.json'}")
    print(f"{'='*80}")

    return df, label_col


def prepare_time_series_data(df, label_col):
    """
    Phase 2: Prepare time-series sequences from the dataset
    Uses cached preprocessed sequences if available
    """
    print("\n" + "=" * 80)
    print("PHASE 2: TIME-SERIES DATA PREPARATION")
    print("=" * 80)

    # Check if preprocessed sequences already exist
    cached_files = {
        'X_train': Path(CACHE_DIR) / 'X_train.npy',
        'X_test': Path(CACHE_DIR) / 'X_test.npy',
        'y_train': Path(CACHE_DIR) / 'y_train.npy',
        'y_test': Path(CACHE_DIR) / 'y_test.npy',
        'label_encoder': Path(CACHE_DIR) / 'label_encoder.pkl',
        'scaler': Path(CACHE_DIR) / 'scaler.pkl',
        'feature_encoder': Path(CACHE_DIR) / 'feature_encoder.pkl',
        'metadata': Path(CACHE_DIR) / 'exploration_metadata.json'
    }

    if USE_CACHED_DATA and all(f.exists() for f in cached_files.values()):
        print("\n✓ CACHED PREPROCESSED SEQUENCES FOUND - LOADING FROM DISK")
        print("=" * 80)

        # Load from cache
        X_train = np.load(cached_files['X_train'], mmap_mode='r')  # Memory-map for efficiency
        X_test = np.load(cached_files['X_test'], mmap_mode='r')
        y_train = np.load(cached_files['y_train'])
        y_test = np.load(cached_files['y_test'])

        with open(cached_files['label_encoder'], 'rb') as f:
            le = pickle.load(f)
        with open(cached_files['scaler'], 'rb') as f:
            scaler = pickle.load(f)
        with open(cached_files['feature_encoder'], 'rb') as f:
            feature_encoder = pickle.load(f)
        with open(cached_files['metadata'], 'r') as f:
            metadata = json.load(f)

        num_classes = len(le.classes_)

        print(f"\n✓ Loaded preprocessed sequences:")
        print(f"  • Training sequences: {len(X_train):,}")
        print(f"  • Testing sequences: {len(X_test):,}")
        print(f"  • Sequence shape: {X_train.shape}")
        print(f"  • Number of classes: {num_classes}")
        if 'total_features_after_encoding' in metadata:
            print(f"  • Feature dimension: {metadata['total_features_after_encoding']}")
        if feature_encoder is not None:
            print(f"  • Encoded categorical columns: {len(feature_encoder.categories_)}")
        print(f"  • Total memory: ~{(X_train.nbytes + X_test.nbytes) / 1024**2:.1f} MB (memory-mapped)")
        print("=" * 80)

        return X_train, X_test, y_train, y_test, le, scaler, num_classes

    # Summary of what we're using
    print(f"\nDataset Summary:")
    print(f"  • Total samples: {len(df):,}")
    print(f"  • Label column: '{label_col}'")
    print(f"  • Unique classes: {df[label_col].nunique()}")
    print(f"  • Classification: {'Multi-class' if df[label_col].nunique() > 2 else 'Binary'}")
    print(f"  • Full dataset: {'Yes' if SAMPLE_SIZE is None else f'No (sampled {SAMPLE_SIZE:,} rows/file)'}")

    feature_cols = [col for col in df.columns if col != label_col]
    numeric_cols_initial = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df[feature_cols].select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    non_numeric_cols_initial = [col for col in feature_cols if col not in numeric_cols_initial]
    categorical_cols = [col for col in non_numeric_cols_initial if col not in datetime_cols]

    print("\n1. Preparing feature matrix...")
    print(f"  • Total feature columns (pre-encoding): {len(feature_cols)}")
    print(f"  • Numeric columns detected: {len(numeric_cols_initial)}")
    print(f"  • Categorical columns detected: {len(categorical_cols)}")
    if datetime_cols:
        print(f"  • Datetime columns detected: {len(datetime_cols)} (will convert to float seconds)")

    X = df[feature_cols].copy()
    y = df[label_col].copy()

    # Convert datetime columns to numeric (seconds since epoch)
    if datetime_cols:
        for col in datetime_cols:
            dt_series = pd.to_datetime(X[col], errors='coerce')
            dt_numeric = dt_series.view('int64').astype('float64')
            dt_numeric[dt_numeric == np.iinfo(np.int64).min] = np.nan
            X[col] = dt_numeric / 1e9

    feature_encoder = None
    if categorical_cols:
        print("  • Encoding categorical columns with OrdinalEncoder")
        X[categorical_cols] = X[categorical_cols].fillna('__MISSING__').astype(str)
        feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categorical_cols] = feature_encoder.fit_transform(X[categorical_cols])
        print(f"    Encoded {len(categorical_cols)} categorical columns")

    feature_names = X.columns.tolist()
    print(f"  • Feature columns after encoding: {len(feature_names)}")

    # Handle infinite values
    print("  • Replacing infinite values...")
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Handle missing values
    nan_before = X.isna().sum().sum()
    if nan_before > 0:
        print(f"  • Found {nan_before:,} NaN values")
        print("  • Filling NaN with column medians...")
        X.fillna(X.median(), inplace=True)
        X.fillna(0, inplace=True)  # Fallback to 0

    # Encode labels
    print("\n2. Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(le.classes_)

    print(f"  • Number of classes: {num_classes}")
    print(f"  • Class encoding:")
    for idx, class_name in enumerate(le.classes_):
        count = np.sum(y_encoded == idx)
        print(f"    {idx:3d}: {str(class_name):30s} ({count:,} samples)")

    # Scale features
    print("\n3. Scaling features...")
    print(f"  • Using RobustScaler (better for outliers)")
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    print(f"  • Scaled data shape: {X_scaled.shape}")
    print(f"  • Scaled data range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")

    # Create sequences
    print(f"\n4. Creating time-series sequences...")
    print(f"  • Sequence length: {SEQUENCE_LENGTH}")
    print(f"  • Feature dimension: {len(feature_names)}")
    print(f"  • Creating sliding windows...")

    # Create sequences using sliding window approach
    print(f"  • Creating sliding windows with stride {WINDOW_STRIDE}...")
    sequences = []
    sequence_labels = []

    # Sliding window approach to create more sequences
    for i in range(0, len(X_scaled) - SEQUENCE_LENGTH + 1, WINDOW_STRIDE):
        seq = X_scaled[i:i + SEQUENCE_LENGTH]
        # Use the label of the last sample in the sequence
        label = y_encoded[i + SEQUENCE_LENGTH - 1]

        sequences.append(seq)
        sequence_labels.append(label)

    X_sequences = np.array(sequences)
    y_sequences = np.array(sequence_labels)

    print(f"  ✓ Created {len(X_sequences):,} sequences")
    print(f"  • Sequence shape: {X_sequences.shape}")
    print(f"  • Labels shape: {y_sequences.shape}")

    # Split data
    print(f"\n5. Splitting data (test size: {VALIDATION_SPLIT*100:.0f}%)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_sequences, y_sequences,
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_STATE,
        stratify=y_sequences
    )

    print(f"  • Training sequences: {len(X_train):,}")
    print(f"  • Testing sequences: {len(X_test):,}")
    print(f"  • Training shape: {X_train.shape}")
    print(f"  • Testing shape: {X_test.shape}")

    # Save preprocessed data
    print(f"\n6. Caching preprocessed data...")
    np.save(Path(CACHE_DIR) / 'X_train.npy', X_train)
    np.save(Path(CACHE_DIR) / 'X_test.npy', X_test)
    np.save(Path(CACHE_DIR) / 'y_train.npy', y_train)
    np.save(Path(CACHE_DIR) / 'y_test.npy', y_test)

    with open(Path(CACHE_DIR) / 'label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    with open(Path(CACHE_DIR) / 'scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    with open(Path(CACHE_DIR) / 'feature_encoder.pkl', 'wb') as f:
        pickle.dump(feature_encoder, f)

    metadata_path = Path(CACHE_DIR) / 'exploration_metadata.json'
    if metadata_path.exists():
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
    else:
        metadata = {}

    metadata.update({
        'feature_columns': feature_cols,
        'numeric_columns': numeric_cols_initial,
        'non_numeric_columns': non_numeric_cols_initial,
        'categorical_columns_encoded': categorical_cols,
        'datetime_columns_converted': datetime_cols,
        'feature_columns_after_encoding': feature_names,
        'total_features_after_encoding': len(feature_names),
        'window_stride': WINDOW_STRIDE
    })

    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"  ✓ Saved to {CACHE_DIR}/")
    print(f"{'='*80}")

    return X_train, X_test, y_train, y_test, le, scaler, num_classes


def build_deepfed_model(input_shape, num_classes):
    """
    Phase 3: Build DeepFed time-series model (GRU + CNN + MLP)
    Based on the paper architecture
    """
    print("\n" + "=" * 80)
    print("PHASE 3: BUILDING DEEPFED TIME-SERIES MODEL")
    print("=" * 80)

    seq_length, num_features = input_shape

    print(f"\nModel Configuration:")
    print(f"  • Input shape: ({seq_length}, {num_features})")
    print(f"  • Output classes: {num_classes}")
    print(f"  • Architecture: GRU + Conv1D + MLP (Parallel branches)")

    # Input layer
    input_layer = layers.Input(shape=(seq_length, num_features), name='input')

    # ============================================================
    # BRANCH 1: GRU Module (Sequential Pattern Detection)
    # ============================================================
    print("\n[Branch 1: GRU Module]")

    # GRU works directly on (batch, seq_len, features) format
    x_gru = input_layer

    # First GRU layer
    x_gru = layers.GRU(128, return_sequences=True, name='gru_1')(x_gru)

    # Second GRU layer
    x_gru = layers.GRU(128, return_sequences=False, name='gru_2')(x_gru)

    print(f"  • GRU Layer 1: 128 units (return sequences)")
    print(f"  • GRU Layer 2: 128 units (return last)")
    print(f"  • Output shape: (batch, 128)")

    # ============================================================
    # BRANCH 2: CNN Module (Spatial Feature Extraction)
    # ============================================================
    print("\n[Branch 2: CNN Module]")

    # Permute for Conv1D: (batch, seq_len, features) -> (batch, features, seq_len)
    x_cnn = layers.Permute((2, 1), name='cnn_permute')(input_layer)

    # Conv Block 1
    x_cnn = layers.Conv1D(32, kernel_size=3, padding='same', name='conv_1')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_1')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_1')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_1')(x_cnn)

    # Conv Block 2
    x_cnn = layers.Conv1D(64, kernel_size=3, padding='same', name='conv_2')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_2')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_2')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_2')(x_cnn)

    # Conv Block 3
    x_cnn = layers.Conv1D(128, kernel_size=3, padding='same', name='conv_3')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_3')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_3')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_3')(x_cnn)

    # Flatten CNN output
    x_cnn = layers.Flatten(name='cnn_flatten')(x_cnn)

    print(f"  • Conv Block 1: 32 filters")
    print(f"  • Conv Block 2: 64 filters")
    print(f"  • Conv Block 3: 128 filters")
    print(f"  • Each with BatchNorm, ReLU, MaxPool")

    # ============================================================
    # CONCATENATE: Merge GRU and CNN features
    # ============================================================
    print("\n[Feature Fusion]")
    concatenated = layers.Concatenate(name='concatenate')([x_cnn, x_gru])
    print(f"  • Concatenating GRU and CNN features")

    # ============================================================
    # MLP Module (Classification Head)
    # ============================================================
    print("\n[Classification Head: MLP]")

    x = layers.Dense(128, activation='relu', name='fc_1')(concatenated)
    x = layers.Dense(64, activation='relu', name='fc_2')(x)
    x = layers.Dropout(0.5, name='dropout')(x)

    # Output layer
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    print(f"  • Dense Layer 1: 128 units")
    print(f"  • Dense Layer 2: 64 units")
    print(f"  • Dropout: 0.5")
    print(f"  • Output Layer: {num_classes} units (softmax)")

    # Create model
    model = models.Model(inputs=input_layer, outputs=output, name='DeepFed')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*80}")
    print("MODEL SUMMARY")
    print(f"{'='*80}")
    model.summary()

    total_params = model.count_params()
    print(f"\nTotal parameters: {total_params:,}")
    print(f"{'='*80}")

    return model


class LazyDataGenerator(keras.utils.PyDataset):
    """
    Memory-efficient data generator that loads data in batches from disk
    Similar to tf.data.Dataset for lazy loading
    """
    def __init__(self, data_file, indices, batch_size=BATCH_SIZE, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data_file = data_file
        self.indices = indices
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.num_samples = len(indices)
        self.num_batches = int(np.ceil(self.num_samples / batch_size))

        # Memory-map the files for efficient access
        self.X = np.load(self.data_file['X'], mmap_mode='r')
        self.y = np.load(self.data_file['y'], mmap_mode='r')

    def __len__(self):
        """Number of batches per epoch"""
        return self.num_batches

    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min(start_idx + self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]

        # Load batch from memory-mapped files
        batch_X = self.X[batch_indices].copy()  # Copy to ensure contiguous array
        batch_y = self.y[batch_indices].copy()

        return batch_X, batch_y

    def on_epoch_end(self):
        """Shuffle indices at end of epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)


def create_data_generator(X, y, batch_size=BATCH_SIZE, shuffle=True):
    """
    Create a data generator to avoid loading entire dataset in memory
    Legacy function for backward compatibility
    """
    num_samples = len(X)
    indices = np.arange(num_samples)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_x = X[batch_indices]
            batch_y = y[batch_indices]

            yield batch_x, batch_y


def train_model(model, X_train, y_train, X_test, y_test):
    """
    Phase 4: Train the DeepFed model
    """
    print("\n" + "=" * 80)
    print("PHASE 4: TRAINING DEEPFED MODEL")
    print("=" * 80)

    print(f"\nTraining Configuration:")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")
    print(f"  • Training samples: {len(X_train):,}")
    print(f"  • Validation samples: {len(X_test):,}")
    print(f"  • Steps per epoch: {len(X_train) // BATCH_SIZE}")

    # Callbacks
    model_callbacks = [
        callbacks.ModelCheckpoint(
            filepath=str(Path(MODEL_DIR) / 'deepfed_best.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        callbacks.CSVLogger(
            filename=str(Path(MODEL_DIR) / 'training_log.csv'),
            separator=',',
            append=False
        )
    ]

    print(f"\n{'='*80}")
    print("STARTING TRAINING")
    print(f"{'='*80}\n")

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=model_callbacks,
        verbose=1
    )

    # Plot training history
    print(f"\n{'='*80}")
    print("TRAINING COMPLETED")
    print(f"{'='*80}")

    # Visualize training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Training')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'training_history.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved training history: {Path(VIS_DIR) / 'training_history.png'}")
    plt.close()

    return history


def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Phase 5: Evaluate the trained model
    """
    print("\n" + "=" * 80)
    print("PHASE 5: MODEL EVALUATION")
    print("=" * 80)

    print("\nGenerating predictions...")
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{'='*80}")
    print("OVERALL METRICS")
    print(f"{'='*80}")
    print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nNOTE: Accuracy can be misleading in this highly imbalanced dataset.")
    print("      Focus on metrics like F1-score, precision, and recall,")
    print("      especially for minority classes, as shown in the Classification Report.")
    print("-" * 80)
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"{'='*80}")

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 80)
    # Convert label_encoder.classes_ (which are numpy.int8) to strings
    target_names_str = [str(cls) for cls in label_encoder.classes_]
    print(classification_report(
        y_test, y_pred,
        target_names=target_names_str,
        digits=4
    ))

    # Per-class metrics
    print("\nPER-CLASS ACCURACY")
    print("-" * 80)
    for i, class_name in enumerate(label_encoder.classes_):
        mask = y_test == i
        if mask.sum() > 0:
            class_acc = accuracy_score(y_test[mask], y_pred[mask])
            print(f"{str(class_name):30s}: {class_acc:.4f} ({class_acc*100:5.2f}%) [{mask.sum():>6,} samples]")

    # Confusion matrix
    print("\nCONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(y_test, y_pred)

    # Visualize confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {Path(VIS_DIR) / 'confusion_matrix.png'}")
    plt.close()

    # Save confusion matrix
    np.save(Path(MODEL_DIR) / 'confusion_matrix.npy', cm)

    return accuracy, f1_macro, f1_weighted


def save_model_artifacts(model, label_encoder, scaler, metadata):
    """
    Save all model artifacts
    """
    print("\n" + "=" * 80)
    print("SAVING MODEL ARTIFACTS")
    print("=" * 80)

    # Save full model
    model_path = Path(MODEL_DIR) / 'deepfed_final.keras'
    model.save(model_path)
    print(f"✓ Model saved: {model_path}")

    # Save label encoder
    le_path = Path(MODEL_DIR) / 'label_encoder.pkl'
    with open(le_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved: {le_path}")

    # Save scaler
    scaler_path = Path(MODEL_DIR) / 'scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved: {scaler_path}")

    # Save metadata
    meta_path = Path(MODEL_DIR) / 'model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved: {meta_path}")

    print(f"\n{'='*80}")
    print("ALL ARTIFACTS SAVED")
    print(f"{'='*80}")


# ============================================================================
# COMPARISON FUNCTIONS FOR IMBALANCED DATASET HANDLING
# ============================================================================

def train_baseline(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 1: Baseline - Train without any imbalance handling
    """
    print("\n" + "="*80)
    print("APPROACH 1: BASELINE (No Imbalance Handling)")
    print("="*80)

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Early stopping callback
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    return model, history


def train_with_class_weights(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 2: Class Weights - Penalize misclassification of minority classes
    """
    print("\n" + "="*80)
    print("APPROACH 2: CLASS WEIGHTS")
    print("="*80)

    # Compute class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print("\nComputed class weights:")
    for cls, weight in class_weight_dict.items():
        print(f"  Class {cls}: {weight:.4f}")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with class weights
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    return model, history


def train_with_oversampling(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 3: SMOTE Oversampling - Generate synthetic samples for minority classes
    """
    print("\n" + "="*80)
    print("APPROACH 3: SMOTE OVERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before SMOTE:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for SMOTE (flatten time series)
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply SMOTE
    print("\nApplying SMOTE...")
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after SMOTE:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        X_train_resampled, y_train_resampled,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    return model, history


def train_with_undersampling(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 4: Random Undersampling - Reduce majority class samples
    """
    print("\n" + "="*80)
    print("APPROACH 4: RANDOM UNDERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before undersampling:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for undersampling
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply random undersampling
    print("\nApplying random undersampling...")
    rus = RandomUnderSampler(random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after undersampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        X_train_resampled, y_train_resampled,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    return model, history


def train_with_combined(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 5: Combined SMOTE-Tomek - Oversample minority + clean decision boundary
    """
    print("\n" + "="*80)
    print("APPROACH 5: COMBINED (SMOTE + Tomek Links)")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before resampling:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for resampling
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply SMOTE-Tomek
    print("\nApplying SMOTE + Tomek Links...")
    smote_tomek = SMOTETomek(random_state=42)
    X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after resampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        X_train_resampled, y_train_resampled,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    return model, history


def evaluate_and_compare(models, histories, X_test, y_test, label_encoder, approach_names):
    """
    Evaluate all approaches and create comparison visualizations
    """
    print("\n" + "="*80)
    print("COMPARISON EVALUATION")
    print("="*80)

    results = []

    for i, (model, history, approach) in enumerate(zip(models, histories, approach_names)):
        if model is None:
            print(f"\nSkipping {approach} (not available)")
            continue

        print(f"\n{'-'*80}")
        print(f"Evaluating: {approach}")
        print(f"{'-'*80}")

        # Predictions
        y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        # Aggregate metrics
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)

        print(f"Accuracy:          {accuracy:.4f}")
        print(f"Macro F1:          {macro_f1:.4f}")
        print(f"Weighted F1:       {weighted_f1:.4f}")
        print(f"Macro Precision:   {macro_precision:.4f}")
        print(f"Macro Recall:      {macro_recall:.4f}")

        results.append({
            'approach': approach,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'per_class_f1': f1.tolist(),
            'per_class_precision': precision.tolist(),
            'per_class_recall': recall.tolist(),
            'per_class_support': support.tolist(),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })

    # Create comparison visualizations
    if len(results) > 0:
        create_comparison_plots(results, label_encoder)
        save_comparison_results(results)

    return results


def create_comparison_plots(results, label_encoder):
    """
    Create comprehensive comparison visualizations
    """
    print("\n" + "="*80)
    print("CREATING COMPARISON VISUALIZATIONS")
    print("="*80)

    approaches = [r['approach'] for r in results]

    # 1. Overall Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision']
    metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score', 'Macro Precision']

    for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
        ax = axes[idx // 2, idx % 2]
        values = [r[metric] for r in results]
        bars = ax.bar(approaches, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
        ax.set_ylabel(name, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_title(f'{name} Comparison', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_overall_metrics.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_overall_metrics.png'}")
    plt.close()

    # 2. Per-Class F1-Score Comparison
    num_classes = len(results[0]['per_class_f1'])
    x = np.arange(num_classes)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 8))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, result in enumerate(results):
        offset = width * (i - len(results)/2 + 0.5)
        ax.bar(x + offset, result['per_class_f1'], width,
               label=result['approach'], color=colors[i % len(colors)])

    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title('Per-Class F1-Score Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_encoder.classes_], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_per_class_f1.png'}")
    plt.close()

    # 3. Macro Recall Comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    values = [r['macro_recall'] for r in results]
    bars = ax.bar(approaches, values,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
    ax.set_ylabel('Macro Recall', fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title('Macro Recall Comparison (Minority Class Performance)',
                fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_macro_recall.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_macro_recall.png'}")
    plt.close()


def save_comparison_results(results):
    """
    Save comparison results to JSON and generate text summary
    """
    # Save JSON
    json_path = Path(MODEL_DIR) / 'comparison_results.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved: {json_path}")

    # Create text summary
    summary_path = Path(MODEL_DIR) / 'comparison_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("IMBALANCED DATASET HANDLING COMPARISON SUMMARY\n")
        f.write("="*80 + "\n\n")

        for result in results:
            f.write(f"\n{result['approach']}\n")
            f.write("-"*80 + "\n")
            f.write(f"  Accuracy:          {result['accuracy']:.4f}\n")
            f.write(f"  Macro F1-Score:    {result['macro_f1']:.4f}\n")
            f.write(f"  Weighted F1-Score: {result['weighted_f1']:.4f}\n")
            f.write(f"  Macro Precision:   {result['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall:      {result['macro_recall']:.4f}\n")

        # Best approach per metric
        f.write("\n" + "="*80 + "\n")
        f.write("BEST APPROACHES PER METRIC\n")
        f.write("="*80 + "\n")

        metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']
        metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score',
                       'Macro Precision', 'Macro Recall']

        for metric, name in zip(metrics, metric_names):
            best = max(results, key=lambda x: x[metric])
            f.write(f"\n{name:20s}: {best['approach']:30s} ({best[metric]:.4f})\n")

    print(f"✓ Saved: {summary_path}")


def main():
    """
    Main execution pipeline with efficient data loading and caching
    """
    print("\n" + "=" * 80)
    print("DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM")
    print("Dataset: Edge-IIoTset")
    print("Model: GRU + CNN (Time-Series Architecture)")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  • Classification: {'Multi-class (attack types)' if USE_MULTICLASS else 'Binary (normal vs attack)'}")
    print(f"  • Dataset: {'Full dataset' if SAMPLE_SIZE is None else f'Sampled ({SAMPLE_SIZE:,} rows/file)'}")
    print(f"  • Use cached data: {USE_CACHED_DATA}")
    print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")

    # Check what's already cached
    cached_sequences = all(Path(CACHE_DIR, f).exists() for f in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy'])
    cached_hdf5 = HDF5_DATASET.exists()

    print(f"\nCache Status:")
    print(f"  • HDF5 preprocessed: {'✓ Found' if cached_hdf5 else '✗ Not found'}")
    print(f"  • Sequence arrays: {'✓ Found' if cached_sequences else '✗ Not found'}")
    if cached_sequences and USE_CACHED_DATA:
        print(f"  → Will skip CSV parsing and sequence creation!")

    try:
        # Check if dataset exists
        csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
        if not csv_exists:
            print("\nDataset not found. Downloading...")
            if not download_dataset():
                print("\n✗ Download failed. Please download manually:")
                print(f"   https://www.kaggle.com/datasets/{DATASET_NAME}")
                return 1

        # Phase 1: Explore dataset
        df, label_col = explore_dataset()

        # Phase 2: Prepare time-series data
        X_train, X_test, y_train, y_test, le, scaler, num_classes = \
            prepare_time_series_data(df, label_col)

        # Phase 3-5: Train and evaluate all 5 comparison approaches
        print("\n" + "="*80)
        print("STARTING COMPARISON STUDY")
        print("Training 5 Different Approaches for Handling Imbalanced Data")
        print("="*80)

        approach_names = [
            "Baseline (No Handling)",
            "Class Weights",
            "SMOTE Oversampling",
            "Random Undersampling",
            "Combined (SMOTE-Tomek)"
        ]

        models = []
        histories = []

        # Approach 1: Baseline
        print("\n" + "█"*80)
        print("█ APPROACH 1/5: BASELINE")
        print("█"*80)
        model1, hist1 = train_baseline(X_train, y_train, X_test, y_test, num_classes)
        models.append(model1)
        histories.append(hist1)

        # Approach 2: Class Weights
        print("\n" + "█"*80)
        print("█ APPROACH 2/5: CLASS WEIGHTS")
        print("█"*80)
        model2, hist2 = train_with_class_weights(X_train, y_train, X_test, y_test, num_classes)
        models.append(model2)
        histories.append(hist2)

        # Approach 3: SMOTE Oversampling
        print("\n" + "█"*80)
        print("█ APPROACH 3/5: SMOTE OVERSAMPLING")
        print("█"*80)
        model3, hist3 = train_with_oversampling(X_train, y_train, X_test, y_test, num_classes)
        models.append(model3)
        histories.append(hist3)

        # Approach 4: Random Undersampling
        print("\n" + "█"*80)
        print("█ APPROACH 4/5: RANDOM UNDERSAMPLING")
        print("█"*80)
        model4, hist4 = train_with_undersampling(X_train, y_train, X_test, y_test, num_classes)
        models.append(model4)
        histories.append(hist4)

        # Approach 5: Combined SMOTE-Tomek
        print("\n" + "█"*80)
        print("█ APPROACH 5/5: COMBINED (SMOTE-TOMEK)")
        print("█"*80)
        model5, hist5 = train_with_combined(X_train, y_train, X_test, y_test, num_classes)
        models.append(model5)
        histories.append(hist5)

        # Evaluate and compare all approaches
        print("\n" + "="*80)
        print("ALL TRAINING COMPLETED - STARTING COMPARISON EVALUATION")
        print("="*80)

        results = evaluate_and_compare(models, histories, X_test, y_test, le, approach_names)

        # Save metadata
        metadata = {
            'model_name': 'DeepFed Comparison Study',
            'dataset': 'Edge-IIoTset',
            'architecture': 'GRU + CNN + MLP',
            'sequence_length': SEQUENCE_LENGTH,
            'num_features': X_train.shape[2],
            'num_classes': num_classes,
            'class_names': le.classes_.tolist(),
            'approaches': approach_names,
            'comparison_results': results,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS
        }

        # Save best model (highest macro F1)
        if results:
            best_idx = max(range(len(results)), key=lambda i: results[i]['macro_f1'])
            best_model = models[best_idx]
            best_approach = approach_names[best_idx]

            print(f"\n{'='*80}")
            print(f"BEST APPROACH: {best_approach}")
            print(f"  Macro F1-Score: {results[best_idx]['macro_f1']:.4f}")
            print(f"{'='*80}")

            save_model_artifacts(best_model, le, scaler, metadata)

        print("\n" + "=" * 80)
        print("✓ COMPARISON STUDY COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"\nResults Summary:")
        for i, (name, result) in enumerate(zip(approach_names, results)):
            print(f"\n{i+1}. {name}")
            print(f"   Accuracy:   {result['accuracy']:.4f}")
            print(f"   Macro F1:   {result['macro_f1']:.4f}")
            print(f"   Macro Recall: {result['macro_recall']:.4f}")
        print(f"\n  • Results saved in:     {MODEL_DIR}/")
        print(f"  • Visualizations in:    {VIS_DIR}/")
        print("=" * 80)

        return 0

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    exit_code = main()
    sys.exit(exit_code)


DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM
Dataset: Edge-IIoTset
Model: GRU + CNN (Time-Series Architecture)

Configuration:
  • Classification: Multi-class (attack types)
  • Dataset: Sampled (100,000 rows/file)
  • Use cached data: True
  • Sequence length: 128 time steps
  • Batch size: 128
  • Epochs: 50
  • Learning rate: 0.001

Cache Status:
  • HDF5 preprocessed: ✗ Not found
  • Sequence arrays: ✗ Not found

Dataset not found. Downloading...

DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE
✓ Running in Google Colab - using secrets
  • Username: pctablet505
  • API Key: ********************************

Extracting edgeiiotset-cyber-security-dataset-of-iot-iiot.zip...
✓ Dataset downloaded and extracted successfully!

PHASE 1: DATASET EXPLORATION & VISUALIZATION

CONVERTING CSV TO EFFICIENT BINARY FORMAT

✓ Found 26 CSV file(s):
  - Password_attack.csv (623.0 MB)
  - DDoS_ICMP_Flood_attack.csv (850.8 MB)
  - SQL_injection_attack.csv (25.7 MB)
  - Ransomware_attack.csv (7.2 MB)
 

Model: "DeepFed"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 67)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 67, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 67, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 67, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 67, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 33, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 33, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 33, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 33, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 16, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 16, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 16, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 16, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 8, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     75,648 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 1024)      │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1152)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    147,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_2 (Dense)        │ (None, 64)        │      8,256 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 375,728 (1.43 MB)

 Trainable params: 375,280 (1.43 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 67)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 67, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 67, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 67, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 67, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 33, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 33, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 33, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 33, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 16, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 16, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 16, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 16, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 8, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     75,648 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 1024)      │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1152)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    147,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_2 (Dense)        │ (None, 64)        │      8,256 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 375,728 (1.43 MB)

 Trainable params: 375,280 (1.43 MB)

 Non-trainable params: 448 (1.75 KB)


Total parameters: 375,728
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 2s/step - accuracy: 0.0611 - loss: 3.5989 - val_accuracy: 0.1843 - val_loss: 4.0194
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 483ms/step - accuracy: 0.2286 - loss: 2.6612 - val_accuracy: 0.2769 - val_loss: 5.0436
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 488ms/step - accuracy: 0.4433 - loss: 2.1920 - val_accuracy: 0.4000 - val_loss: 5.1628
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 477ms/step - accuracy: 0.4730 - loss: 2.0622 - val_accuracy: 0.4111 - val_loss: 4.7989
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 481ms/step - accuracy: 0.5689 - loss: 1.9578 - val_accuracy: 0.5099 - val_loss: 4.5908
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 485ms/step - accuracy: 0.5749 - loss: 1.7949 - val_accuracy: 0.5658 - val_loss: 4.5622
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 484ms/step - accuracy: 0.6430 - loss: 1.5972 - val_accuracy: 0.5938 - val_loss: 4.4246
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 480ms/step - accuracy: 0.6322 - loss: 1.4799 - val_accu

In [ ]:
"""
DeepFed: Intrusion Detection with Imbalanced Dataset Handling
==============================================================

Research-focused implementation comparing different approaches to handle
highly imbalanced network attack datasets (majority Normal, minority Attacks).

COMPARISON APPROACHES:
1. Baseline: Train on imbalanced data (no handling)
2. Class Weights: Penalize misclassification of minority classes
3. SMOTE Oversampling: Synthetic minority oversampling technique
4. Random Undersampling: Random undersampling of majority class
5. Combined: SMOTE + Tomek links cleaning

Dataset: Edge-IIoTset - Cyber Security Dataset of IoT & IIoT
Model: DeepFed (GRU + CNN parallel architecture)

Based on DeepFed paper: "DeepFed: Federated Deep Learning for Intrusion Detection
in Industrial Cyber-Physical Systems"

INSTALLATION:
-------------
For full functionality (all 5 approaches), install imbalanced-learn:
    pip install imbalanced-learn

Without imbalanced-learn, only approaches 1 and 2 will run.

USAGE:
------
python deepfed_train.py

The script will:
- Load and preprocess Edge-IIoTset dataset
- Train 5 models with different imbalance handling approaches
- Evaluate and compare all approaches
- Generate comparison visualizations
- Save results to models/deepfed/ and visualizations/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import subprocess
import sys
import pickle
from collections import Counter
import json
import gc  # For garbage collection

# Check for GPU availability and dynamic batch sizing
try:
    import tensorflow as tf
    TF_AVAILABLE = True
    print("TensorFlow available for GPU detection")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available, using CPU-only batch sizing")

# Use Keras 3 (not tensorflow.keras)
import keras
from keras import layers, models, callbacks, ops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OrdinalEncoder
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for handling imbalanced datasets
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.combine import SMOTETomek
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("WARNING: imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("         Only baseline and class weights approaches will be available.")
    IMBLEARN_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

# Check for required packages
try:
    import tables  # For HDF5 support
except ImportError:
    print("ERROR: pytables is required for HDF5 export. Please install with `pip install tables`.")
    sys.exit(1)

# Configuration
DATASET_NAME = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
DATA_DIR = "./data/edge_iiot"
MODEL_DIR = "./models/deepfed"
VIS_DIR = "./visualizations"
CACHE_DIR = "./cache"
PREPROCESSED_DIR = "./cache/preprocessed"  # For efficient binary format
HDF5_DATASET = Path(PREPROCESSED_DIR) / "dataset.h5"
BATCH_SIZE = 128  # Default fallback batch size
EPOCHS = 50
LEARNING_RATE = 0.001
RANDOM_STATE = 42
SEQUENCE_LENGTH = 128  # Time steps for time-series sequences (set None for dynamic windows)
WINDOW_STRIDE = 64  # Stride for sliding window sequence creation
VALIDATION_SPLIT = 0.2
SAMPLE_SIZE = 100000  # Use a small sample for debugging; set to None for full dataset


def create_optimized_tf_dataset(X, y, batch_size=None, shuffle=True, autotune=True):
    """
    Create optimized TensorFlow dataset with automatic performance tuning.

    Args:
        X: Input features array
        y: Target labels array
        batch_size: Batch size (if None, will be determined automatically)
        shuffle: Whether to shuffle the dataset
        autotune: Whether to use AUTOTUNE for prefetch and batching

    Returns:
        Optimized tf.data.Dataset
    """
    # Convert to TensorFlow tensors
    X_tf = tf.convert_to_tensor(X, dtype=tf.float32)
    y_tf = tf.convert_to_tensor(y, dtype=tf.int32)

    # Create dataset
    dataset = tf.data.Dataset.from_tensor_slices((X_tf, y_tf))

    # Shuffle if requested
    if shuffle:
        dataset = dataset.shuffle(buffer_size=min(len(X), 10000))

    # Batch with autotune if requested
    if autotune and batch_size is None:
        # Let TensorFlow determine optimal batch size
        dataset = dataset.batch(tf.data.AUTOTUNE)
        print("✓ Using TensorFlow AUTOTUNE for batch size")
    elif batch_size is not None:
        dataset = dataset.batch(batch_size)

    # Prefetch for performance
    if autotune:
        dataset = dataset.prefetch(tf.data.AUTOTUNE)

    return dataset


def determine_optimal_batch_size_tf(X_train_shape, num_classes, target_memory_utilization=0.8):
    """
    Determine optimal batch size using TensorFlow's performance profiling.

    Args:
        X_train_shape: Shape of training data (samples, timesteps, features)
        num_classes: Number of output classes
        target_memory_utilization: Target GPU memory utilization (0.0-1.0)

    Returns:
        Optimal batch size for current hardware
    """
    print("\n" + "="*60)
    print("DETERMINING OPTIMAL BATCH SIZE WITH TF.DATA")
    print("="*60)

    # Check GPU availability
    gpus = tf.config.list_physical_devices('GPU')
    if gpus:
        print(f"✓ GPU detected: {len(gpus)} device(s)")
        for i, gpu in enumerate(gpus):
            try:
                gpu_details = tf.config.experimental.get_device_details(gpu)
                print(f"  GPU {i}: {gpu_details.get('device_name', 'Unknown')}")
            except:
                print(f"  GPU {i}: Available")
    else:
        print("✗ No GPU detected, using CPU optimization")
        return min(64, max(16, X_train_shape[0] // 100))  # Conservative CPU batch size

    # Estimate memory requirements
    samples, timesteps, features = X_train_shape
    input_size_bytes = timesteps * features * 4  # float32
    model_params_estimate = (timesteps * features * num_classes) * 4  # Rough model size

    # Get GPU memory info
    try:
        gpu_memory = tf.config.experimental.get_memory_info('GPU:0')
        total_memory = gpu_memory['total'] if 'total' in gpu_memory else 8 * 1024**3  # 8GB default
        print(f"  GPU memory: {total_memory / (1024**3):.1f} GB total")
    except:
        total_memory = 8 * 1024**3  # Conservative 8GB default
        print(f"  Using conservative memory estimate: {total_memory / (1024**3):.1f} GB")

    # Calculate usable memory
    usable_memory = int(total_memory * target_memory_utilization)
    print(f"  Target utilization: {target_memory_utilization*100:.0f}% = {usable_memory / (1024**3):.1f} GB")

    # Estimate batch size based on memory
    # Memory per batch = input + model + gradients + overhead
    memory_per_sample = input_size_bytes + model_params_estimate + (model_params_estimate * 2)  # gradients
    max_batch_by_memory = usable_memory // memory_per_sample

    # Apply practical limits
    optimal_batch = min(max_batch_by_memory, 512)  # Cap at 512
    optimal_batch = max(optimal_batch, 16)  # Minimum 16

    print(f"  Memory per sample: {memory_per_sample / (1024**2):.1f} MB")
    print(f"  Theoretical max batch: {max_batch_by_memory}")
    print(f"  Optimal batch size: {optimal_batch}")

    # Test the batch size with a small model
    try:
        test_model = build_deepfed_model(input_shape=(timesteps, features), num_classes=num_classes)
        dummy_input = tf.random.normal((optimal_batch, timesteps, features))

        # Warm up
        _ = test_model(dummy_input, training=False)

        # Time a few batches to verify performance
        import time
        start_time = time.time()
        for _ in range(5):
            _ = test_model(dummy_input, training=False)
        end_time = time.time()

        inference_time = (end_time - start_time) / 5
        print(f"  Performance test: {inference_time:.4f}s per batch")

        # Adjust batch size based on performance
        if inference_time > 0.1:  # Too slow
            optimal_batch = max(16, optimal_batch // 2)
            print(f"  Adjusted for performance: {optimal_batch}")
        elif inference_time < 0.01:  # Can go larger
            optimal_batch = min(1024, optimal_batch * 2)
            print(f"  Increased for better utilization: {optimal_batch}")

    except Exception as e:
        print(f"⚠️  Performance test failed: {e}")
        optimal_batch = min(64, optimal_batch)  # Conservative fallback

    print(f"\n🎯 FINAL OPTIMAL BATCH SIZE: {optimal_batch}")
    print("="*60)

    return optimal_batch
USE_MULTICLASS = True  # Use multi-class attack type classification
USE_CACHED_DATA = True  # Use preprocessed binary data if available

# Set random seeds
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Create directories
for dir_path in [DATA_DIR, MODEL_DIR, VIS_DIR, CACHE_DIR, PREPROCESSED_DIR]:
    os.makedirs(dir_path, exist_ok=True)


def download_dataset():
    """Download Edge-IIoTset dataset from Kaggle"""
    print("\n" + "=" * 80)
    print("DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE")
    print("=" * 80)

    # Setup Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        print("✓ Running in Google Colab - using secrets")

        # Get credentials from Colab secrets
        kaggle_username = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')

        if not kaggle_username or not kaggle_key:
            raise ValueError("KAGGLE_USERNAME and KAGGLE_KEY must be set in Colab secrets")

        # Set environment variables for Kaggle API
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key

        print(f"  • Username: {kaggle_username}")
        print(f"  • API Key: {'*' * len(kaggle_key)}")

    except ImportError:
        print("✓ Not running in Colab - using default kaggle.json authentication")
    except Exception as e:
        print(f"✗ Error setting up Kaggle credentials: {e}")
        print("\nPlease add these secrets in Colab:")
        print("  1. Click the key icon (🔑) in the left sidebar")
        print("  2. Add secret: KAGGLE_USERNAME")
        print("  3. Add secret: KAGGLE_KEY")
        print("\nGet your credentials from: https://www.kaggle.com/settings/account")
        raise

    try:
        import kaggle
    except ImportError:
        print("Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
        import kaggle

    try:
        print(f"\nDownloading {DATASET_NAME}...")
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET_NAME, "-p", DATA_DIR
        ], check=True)

        # Extract zip files
        for zip_file in Path(DATA_DIR).glob("*.zip"):
            print(f"Extracting {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            zip_file.unlink()

        print("✓ Dataset downloaded and extracted successfully!")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        print(f"\nPlease download manually from:")
        print(f"https://www.kaggle.com/datasets/{DATASET_NAME}")
        return False


def convert_csv_to_binary():
    """
    Convert CSV files to a consolidated Parquet dataset while preserving all features.
    Adds source metadata and derived temporal features for downstream processing.
    """
    preprocessed_file = HDF5_DATASET

    if preprocessed_file.exists() and USE_CACHED_DATA:
        print("\n" + "=" * 80)
        print("✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING")
        print("=" * 80)
        print(f"Using cached file: {preprocessed_file}")
        print(f"Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")
        return preprocessed_file

    print("\n" + "=" * 80)
    print("CONVERTING CSV TO EFFICIENT BINARY FORMAT")
    print("=" * 80)

    # Find all CSV files
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV files found!")

    print(f"\n✓ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

    # Load and combine all CSVs
    if SAMPLE_SIZE:
        print(f"\nLoading sample data (max {SAMPLE_SIZE:,} rows per file)...")
    else:
        print(f"\nLoading FULL dataset (this may take a while)...")

    dfs = []
    manifest = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, nrows=SAMPLE_SIZE, low_memory=False)
            original_rows = len(df)

            # Normalize string columns to avoid mixed dtype issues
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            for col in object_cols:
                df[col] = df[col].astype(str).str.strip().fillna('__NA__')

            # Attach source metadata
            df['source_file'] = csv_file.name
            df['source_path'] = str(csv_file.relative_to(DATA_DIR))
            df['source_category'] = csv_file.parent.name

            # Derive attack type from filename
            filename = csv_file.name
            if '_attack.csv' in filename:
                attack_type = filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                attack_type = 'Mixed'  # These contain multiple attack types
            else:
                attack_type = 'Normal'  # Sensor data files
            df['Attack_type'] = attack_type

            # Build temporal features without dropping original column
            if 'frame.time' in df.columns:
                time_str = df['frame.time'].astype(str).str.strip()
                parsed_time = pd.to_datetime(time_str, format='%Y %H:%M:%S.%f', errors='coerce')
                if parsed_time.isna().all():
                    parsed_time = pd.to_datetime(time_str, errors='coerce')
                parsed_time = parsed_time.fillna(method='ffill').fillna(method='bfill')
                df['frame_time_datetime'] = parsed_time
                base_time = parsed_time.iloc[0]
                rel_seconds = (parsed_time - base_time).dt.total_seconds()
                df['frame_time_relative_sec'] = rel_seconds.astype('float64')

            dfs.append(df)
            duration = float(df.get('frame_time_relative_sec', pd.Series([0])).max()) if len(df) else 0.0
            manifest.append({
                'file': str(csv_file.relative_to(DATA_DIR)),
                'rows_loaded': int(original_rows),
                'duration_seconds': duration
            })
            print(f"  ✓ {csv_file.name}: {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading {csv_file.name}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*80}")
    print(f"Combined dataset: {len(df):,} rows × {len(df.columns)} columns")
    print(f"{'='*80}")

    print("\nPreparing data for HDF5 storage...")
    df_filtered = df.copy()
    print(f"Final dataset shape: {df_filtered.shape}")
    print(f"Memory usage: {df_filtered.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    preprocessed_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle HDF5 file locking issues (common in Colab)
    if preprocessed_file.exists():
        print(f"  → HDF5 file exists, removing to recreate...")
        try:
            preprocessed_file.unlink()  # Remove the existing file
            print("  ✓ Removed existing HDF5 file")
        except Exception as e:
            print(f"  ✗ Could not remove existing file: {e}")
            print("  → Please manually delete the file and try again")
            raise RuntimeError(f"Cannot remove existing HDF5 file {preprocessed_file}. Please delete it manually.")

    # Save to HDF5
    df_filtered.to_hdf(preprocessed_file, key='data', mode='w', index=False)
    print(f"✓ Saved: {preprocessed_file}")
    print(f"  Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")

    total_csv_size = sum(f.stat().st_size for f in csv_files)
    compression_ratio = (1 - preprocessed_file.stat().st_size / total_csv_size) * 100
    print(f"  Compression: {compression_ratio:.1f}% savings over CSV")
    print(f"  Original CSV size: {total_csv_size / 1024**2:.1f} MB")

    manifest_path = Path(PREPROCESSED_DIR) / 'ingest_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest saved: {manifest_path}")

    return preprocessed_file


def explore_dataset():
    """
    Phase 1: Deep exploration and visualization of Edge-IIoTset dataset
    Uses preprocessed binary format for fast loading
    """
    print("\n" + "=" * 80)
    print("PHASE 1: DATASET EXPLORATION & VISUALIZATION")
    print("=" * 80)

    # Convert CSV to binary format (only once)
    preprocessed_file = convert_csv_to_binary()

    # Load from efficient binary format
    print(f"\nLoading preprocessed data...")
    df = pd.read_hdf(preprocessed_file, key='data')

    print(f"✓ Loaded {len(df):,} rows × {len(df.columns)} columns")
    print(f"  Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    # Basic dataset info
    print("\n1. DATASET STRUCTURE")
    print("-" * 80)
    print(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
    print(f"\nColumn types:")
    print(df.dtypes.value_counts())

    print(f"\nFirst few columns:")
    for i, col in enumerate(df.columns[:10], 1):
        print(f"  {i:2d}. {col:40s} ({df[col].dtype})")
    if len(df.columns) > 10:
        print(f"  ... and {len(df.columns) - 10} more columns")

    # The cached Parquet dataset preserves all columns, so identify label column dynamically
    # Label column is typically categorical with a limited number of unique values
    # For Edge-IIoTset, prioritize 'Attack_type' column
    if 'Attack_type' not in df.columns and 'source_file' in df.columns:
        # Derive Attack_type from source_file if missing
        def derive_attack_type(filename):
            if '_attack.csv' in filename:
                return filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                return 'Mixed'  # These contain multiple attack types
            else:
                return 'Normal'  # Sensor data files
        df['Attack_type'] = df['source_file'].apply(derive_attack_type)
        print(f"\n✓ Derived 'Attack_type' column from 'source_file'")

    if 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"\n✓ Label column identified: '{label_col}' (Attack_type)")
    else:
        numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
        potential_labels = [col for col in df.columns if col not in numeric_cols]

        if potential_labels:
            # Use the first potential label column
            label_col = potential_labels[0]
            print(f"\n✓ Label column identified: '{label_col}'")
        else:
            # If all columns are numeric, assume the last column is the label
            # (common convention in ML datasets)
            label_col = df.columns[-1]
            print(f"\n✓ Label column (assumed): '{label_col}' (last column)")

    # Determine if it's multi-class or binary
    unique_labels = df[label_col].nunique()
    if unique_labels > 2 or USE_MULTICLASS:
        print(f"✓ Using MULTI-CLASS classification ({unique_labels} classes)")
    else:
        print(f"✓ Using BINARY classification ({unique_labels} classes)")

    # Class distribution analysis
    print("\n2. CLASS DISTRIBUTION")
    print("-" * 80)
    class_counts = df[label_col].value_counts()
    print(f"Number of classes: {len(class_counts)}")

    # Show all classes if multi-class, otherwise show binary
    print(f"\nClass distribution:")
    if len(class_counts) <= 20:
        # Show all classes
        for class_name, count in class_counts.items():
            pct = count / len(df) * 100
            bar = '█' * min(int(pct), 50)  # Cap bar at 50 chars
            print(f"  {str(class_name):30s}: {count:8,} ({pct:5.2f}%) {bar}")
    else:
        # Show top 20 classes
        for class_name, count in class_counts.head(20).items():
            pct = count / len(df) * 100
            bar = '█' * min(int(pct), 50)
            print(f"  {str(class_name):30s}: {count:8,} ({pct:5.2f}%) {bar}")
        print(f"  ... and {len(class_counts) - 20} more classes")

    # Calculate class imbalance ratio
    max_count = class_counts.max()
    min_count = class_counts.min()
    imbalance_ratio = max_count / min_count
    print(f"\nClass imbalance ratio: {imbalance_ratio:.2f}:1")
    print(f"  → This is a {'highly ' if imbalance_ratio > 100 else ''}imbalanced dataset!")

    # Visualize class distribution
    if len(class_counts) <= 20:
        # Show all classes
        plt.figure(figsize=(16, 8))
        plt.subplot(1, 2, 1)
        class_counts.plot(kind='bar')
        plt.title('Class Distribution (Count)', fontsize=14, fontweight='bold')
        plt.xlabel('Attack Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

        plt.subplot(1, 2, 2)
        # Only show pie chart if <=10 classes (too messy otherwise)
        if len(class_counts) <= 10:
            class_counts.plot(kind='pie', autopct='%1.1f%%')
            plt.title('Class Distribution (Percentage)', fontsize=14, fontweight='bold')
            plt.ylabel('')
        else:
            class_counts.head(10).plot(kind='pie', autopct='%1.1f%%')
            plt.title('Top 10 Classes (Percentage)', fontsize=14, fontweight='bold')
            plt.ylabel('')
        plt.tight_layout()
    else:
        # Too many classes - show top 20 only
        plt.figure(figsize=(16, 6))
        class_counts.head(20).plot(kind='bar')
        plt.title('Top 20 Classes Distribution', fontsize=14, fontweight='bold')
        plt.xlabel('Attack Type')
        plt.ylabel('Count')
        plt.xticks(rotation=45, ha='right')
        plt.tight_layout()

    plt.savefig(Path(VIS_DIR) / 'class_distribution.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved visualization: {Path(VIS_DIR) / 'class_distribution.png'}")
    plt.close()

    # Feature analysis
    print("\n3. FEATURE ANALYSIS")
    print("-" * 80)

    # Separate numeric and non-numeric features
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    non_numeric_cols = df.select_dtypes(exclude=[np.number]).columns.tolist()

    # Remove label from feature lists
    if label_col in numeric_cols:
        numeric_cols.remove(label_col)
    if label_col in non_numeric_cols:
        non_numeric_cols.remove(label_col)

    print(f"Numeric features: {len(numeric_cols)}")
    print(f"Non-numeric features: {len(non_numeric_cols)}")

    if non_numeric_cols:
        print(f"\nNon-numeric columns (will be ordinal-encoded for time-series modeling):")
        for col in non_numeric_cols[:10]:
            unique_vals = df[col].nunique()
            print(f"  - {col:40s} ({unique_vals} unique values)")
        if len(non_numeric_cols) > 10:
            print(f"  ... and {len(non_numeric_cols) - 10} more")

    # Missing values
    print(f"\n4. DATA QUALITY")
    print("-" * 80)
    missing = df[numeric_cols].isnull().sum()
    if missing.sum() > 0:
        print(f"Missing values:")
        for col, count in missing[missing > 0].items():
            pct = count / len(df) * 100
            print(f"  {col:40s}: {count:,} ({pct:.2f}%)")
    else:
        print("✓ No missing values in numeric features")

    # Infinite values
    inf_counts = {}
    for col in numeric_cols[:20]:  # Check first 20 numeric columns
        inf_count = np.isinf(df[col]).sum()
        if inf_count > 0:
            inf_counts[col] = inf_count

    if inf_counts:
        print(f"\nInfinite values detected:")
        for col, count in list(inf_counts.items())[:10]:
            print(f"  {col:40s}: {count:,}")
    else:
        print("✓ No infinite values in numeric features (checked first 20)")

    # Feature statistics
    print(f"\n5. FEATURE STATISTICS (first 10 numeric features)")
    print("-" * 80)
    feature_stats = df[numeric_cols[:10]].describe()
    print(feature_stats.to_string())

    # Correlation analysis (sample)
    print(f"\n6. CORRELATION ANALYSIS")
    print("-" * 80)
    print("Computing correlations for sample features...")

    # Select a subset of features for correlation
    sample_features = numeric_cols[:20] if len(numeric_cols) > 20 else numeric_cols
    corr_matrix = df[sample_features].corr()

    # Find highly correlated features
    high_corr = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i+1, len(corr_matrix.columns)):
            if abs(corr_matrix.iloc[i, j]) > 0.9:
                high_corr.append((
                    corr_matrix.columns[i],
                    corr_matrix.columns[j],
                    corr_matrix.iloc[i, j]
                ))

    if high_corr:
        print(f"Found {len(high_corr)} highly correlated feature pairs (|r| > 0.9):")
        for feat1, feat2, corr_val in high_corr[:5]:
            print(f"  {feat1:30s} ↔ {feat2:30s}: {corr_val:.3f}")
        if len(high_corr) > 5:
            print(f"  ... and {len(high_corr) - 5} more pairs")
    else:
        print("No highly correlated feature pairs found (in sampled features)")

    # Visualize correlation matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
                square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
    plt.title(f'Feature Correlation Matrix (first {len(sample_features)} features)',
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'correlation_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization: {Path(VIS_DIR) / 'correlation_matrix.png'}")
    plt.close()

    # Feature distributions by class
    print(f"\n7. FEATURE DISTRIBUTIONS BY CLASS")
    print("-" * 80)
    print("Visualizing feature distributions for sample features...")

    # Select interesting features to visualize
    viz_features = sample_features[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for idx, col in enumerate(viz_features):
        if idx >= len(axes):
            break

        # Sample data for visualization
        sample_data = df[[col, label_col]].sample(min(10000, len(df)))

        for class_name in sample_data[label_col].unique()[:5]:  # Top 5 classes
            class_data = sample_data[sample_data[label_col] == class_name][col]
            axes[idx].hist(class_data, bins=50, alpha=0.5, label=str(class_name)[:20])

        axes[idx].set_xlabel(col if len(col) < 30 else col[:27] + '...')
        axes[idx].set_ylabel('Frequency')
        axes[idx].legend(fontsize=8)
        axes[idx].set_title(f'{col[:40]}', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'feature_distributions.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved visualization: {Path(VIS_DIR) / 'feature_distributions.png'}")
    plt.close()

    # Time-series characteristics
    print(f"\n8. TIME-SERIES CHARACTERISTICS")
    print("-" * 80)
    print(f"For time-series modeling, we will:")
    print(f"  • Use sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Create sliding windows from the data")
    print(f"  • Each sequence will have shape: ({SEQUENCE_LENGTH}, {len(numeric_cols)})")
    print(f"  • Model architecture: GRU + CNN (as per DeepFed paper)")

    # Save exploration metadata
    exploration_meta = {
        'num_samples': len(df),
        'num_features': len(numeric_cols),
        'num_classes': len(class_counts),
        'class_names': [str(c) for c in class_counts.index.tolist()],
        'class_counts': {str(k): int(v) for k, v in class_counts.to_dict().items()},
        'imbalance_ratio': float(imbalance_ratio),
        'label_column': label_col,
        'classification_type': 'multi-class' if len(class_counts) > 2 else 'binary',
        'numeric_columns': numeric_cols,
        'non_numeric_columns': non_numeric_cols,
        'sequence_length': SEQUENCE_LENGTH,
        'window_stride': WINDOW_STRIDE,
        'using_full_dataset': SAMPLE_SIZE is None
    }

    with open(Path(CACHE_DIR) / 'exploration_metadata.json', 'w') as f:
        json.dump(exploration_meta, f, indent=2)

    print(f"\n✓ Saved exploration metadata: {Path(CACHE_DIR) / 'exploration_metadata.json'}")
    print(f"{'='*80}")

    return df, label_col


def prepare_time_series_data(df, label_col):
    """
    Phase 2: Prepare time-series sequences from the dataset
    Uses cached preprocessed sequences if available
    """
    print("\n" + "=" * 80)
    print("PHASE 2: TIME-SERIES DATA PREPARATION")
    print("=" * 80)

    # Check if preprocessed sequences already exist
    cached_files = {
        'X_train': Path(CACHE_DIR) / 'X_train.npy',
        'X_test': Path(CACHE_DIR) / 'X_test.npy',
        'y_train': Path(CACHE_DIR) / 'y_train.npy',
        'y_test': Path(CACHE_DIR) / 'y_test.npy',
        'label_encoder': Path(CACHE_DIR) / 'label_encoder.pkl',
        'scaler': Path(CACHE_DIR) / 'scaler.pkl',
        'feature_encoder': Path(CACHE_DIR) / 'feature_encoder.pkl',
        'metadata': Path(CACHE_DIR) / 'exploration_metadata.json'
    }

    if USE_CACHED_DATA and all(f.exists() for f in cached_files.values()):
        print("\n✓ CACHED PREPROCESSED SEQUENCES FOUND - LOADING FROM DISK")
        print("=" * 80)

        # Load from cache
        X_train = np.load(cached_files['X_train'], mmap_mode='r')  # Memory-map for efficiency
        X_test = np.load(cached_files['X_test'], mmap_mode='r')
        y_train = np.load(cached_files['y_train'])
        y_test = np.load(cached_files['y_test'])

        with open(cached_files['label_encoder'], 'rb') as f:
            le = pickle.load(f)
        with open(cached_files['scaler'], 'rb') as f:
            scaler = pickle.load(f)
        with open(cached_files['feature_encoder'], 'rb') as f:
            feature_encoder = pickle.load(f)
        with open(cached_files['metadata'], 'r') as f:
            metadata = json.load(f)

        num_classes = len(le.classes_)

        print(f"\n✓ Loaded preprocessed sequences:")
        print(f"  • Training sequences: {len(X_train):,}")
        print(f"  • Testing sequences: {len(X_test):,}")
        print(f"  • Sequence shape: {X_train.shape}")
        print(f"  • Number of classes: {num_classes}")
        if 'total_features_after_encoding' in metadata:
            print(f"  • Feature dimension: {metadata['total_features_after_encoding']}")
        if feature_encoder is not None:
            print(f"  • Encoded categorical columns: {len(feature_encoder.categories_)}")
        print(f"  • Total memory: ~{(X_train.nbytes + X_test.nbytes) / 1024**2:.1f} MB (memory-mapped)")
        print("=" * 80)

        return X_train, X_test, y_train, y_test, le, scaler, num_classes

    # Summary of what we're using
    print(f"\nDataset Summary:")
    print(f"  • Total samples: {len(df):,}")
    print(f"  • Label column: '{label_col}'")
    print(f"  • Unique classes: {df[label_col].nunique()}")
    print(f"  • Classification: {'Multi-class' if df[label_col].nunique() > 2 else 'Binary'}")
    print(f"  • Full dataset: {'Yes' if SAMPLE_SIZE is None else f'No (sampled {SAMPLE_SIZE:,} rows/file)'}")

    feature_cols = [col for col in df.columns if col != label_col]
    numeric_cols_initial = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df[feature_cols].select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    non_numeric_cols_initial = [col for col in feature_cols if col not in numeric_cols_initial]
    categorical_cols = [col for col in non_numeric_cols_initial if col not in datetime_cols]

    print("\n1. Preparing feature matrix...")
    print(f"  • Total feature columns (pre-encoding): {len(feature_cols)}")
    print(f"  • Numeric columns detected: {len(numeric_cols_initial)}")
    print(f"  • Categorical columns detected: {len(categorical_cols)}")
    if datetime_cols:
        print(f"  • Datetime columns detected: {len(datetime_cols)} (will convert to float seconds)")

    X = df[feature_cols].copy()
    y = df[label_col].copy()

    # Convert datetime columns to numeric (seconds since epoch)
    if datetime_cols:
        for col in datetime_cols:
            dt_series = pd.to_datetime(X[col], errors='coerce')
            dt_numeric = dt_series.view('int64').astype('float64')
            dt_numeric[dt_numeric == np.iinfo(np.int64).min] = np.nan
            X[col] = dt_numeric / 1e9

    feature_encoder = None
    if categorical_cols:
        print("  • Encoding categorical columns with OrdinalEncoder")
        X[categorical_cols] = X[categorical_cols].fillna('__MISSING__').astype(str)
        feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categorical_cols] = feature_encoder.fit_transform(X[categorical_cols])
        print(f"    Encoded {len(categorical_cols)} categorical columns")

    feature_names = X.columns.tolist()
    print(f"  • Feature columns after encoding: {len(feature_names)}")

    # Handle infinite values
    print("  • Replacing infinite values...")
    X.replace([np.inf, -np.inf], np.nan, inplace=True)

    # Handle missing values
    nan_before = X.isna().sum().sum()
    if nan_before > 0:
        print(f"  • Found {nan_before:,} NaN values")
        print("  • Filling NaN with column medians...")
        X.fillna(X.median(), inplace=True)
        X.fillna(0, inplace=True)  # Fallback to 0

    # Encode labels
    print("\n2. Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(le.classes_)

    print(f"  • Number of classes: {num_classes}")
    print(f"  • Class encoding:")
    for idx, class_name in enumerate(le.classes_):
        count = np.sum(y_encoded == idx)
        print(f"    {idx:3d}: {str(class_name):30s} ({count:,} samples)")

    # Scale features
    print("\n3. Scaling features...")
    print(f"  • Using RobustScaler (better for outliers)")
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    print(f"  • Scaled data shape: {X_scaled.shape}")
    print(f"  • Scaled data range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")

    # Create sequences
    print(f"\n4. Creating time-series sequences...")
    print(f"  • Sequence length: {SEQUENCE_LENGTH}")
    print(f"  • Feature dimension: {len(feature_names)}")
    print(f"  • Creating sliding windows...")

    # Create sequences using sliding window approach
    print(f"  • Creating sliding windows with stride {WINDOW_STRIDE}...")
    sequences = []
    sequence_labels = []

    # Sliding window approach to create more sequences
    for i in range(0, len(X_scaled) - SEQUENCE_LENGTH + 1, WINDOW_STRIDE):
        seq = X_scaled[i:i + SEQUENCE_LENGTH]
        # Use the label of the last sample in the sequence
        label = y_encoded[i + SEQUENCE_LENGTH - 1]

        sequences.append(seq)
        sequence_labels.append(label)

    X_sequences = np.array(sequences)
    y_sequences = np.array(sequence_labels)

    print(f"  ✓ Created {len(X_sequences):,} sequences")
    print(f"  • Sequence shape: {X_sequences.shape}")
    print(f"  • Labels shape: {y_sequences.shape}")

    # Split data
    print(f"\n5. Splitting data (test size: {VALIDATION_SPLIT*100:.0f}%)...")
    X_train, X_test, y_train, y_test = train_test_split(
        X_sequences, y_sequences,
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_STATE,
        stratify=y_sequences
    )

    print(f"  • Training sequences: {len(X_train):,}")
    print(f"  • Testing sequences: {len(X_test):,}")
    print(f"  • Training shape: {X_train.shape}")
    print(f"  • Testing shape: {X_test.shape}")

    # Save preprocessed data
    print(f"\n6. Caching preprocessed data...")
    np.save(Path(CACHE_DIR) / 'X_train.npy', X_train)
    np.save(Path(CACHE_DIR) / 'X_test.npy', X_test)
    np.save(Path(CACHE_DIR) / 'y_train.npy', y_train)
    np.save(Path(CACHE_DIR) / 'y_test.npy', y_test)

    with open(Path(CACHE_DIR) / 'label_encoder.pkl', 'wb') as f:
        pickle.dump(le, f)
    with open(Path(CACHE_DIR) / 'scaler.pkl', 'wb') as f:
        pickle.dump(scaler, f)
    with open(Path(CACHE_DIR) / 'feature_encoder.pkl', 'wb') as f:
        pickle.dump(feature_encoder, f)

    metadata_path = Path(CACHE_DIR) / 'exploration_metadata.json'
    if metadata_path.exists():
        with open(metadata_path, 'r') as f:
            metadata = json.load(f)
    else:
        metadata = {}

    metadata.update({
        'feature_columns': feature_cols,
        'numeric_columns': numeric_cols_initial,
        'non_numeric_columns': non_numeric_cols_initial,
        'categorical_columns_encoded': categorical_cols,
        'datetime_columns_converted': datetime_cols,
        'feature_columns_after_encoding': feature_names,
        'total_features_after_encoding': len(feature_names),
        'window_stride': WINDOW_STRIDE
    })

    with open(metadata_path, 'w') as f:
        json.dump(metadata, f, indent=2)

    print(f"  ✓ Saved to {CACHE_DIR}/")
    print(f"{'='*80}")

    return X_train, X_test, y_train, y_test, le, scaler, num_classes


def build_deepfed_model(input_shape, num_classes):
    """
    Phase 3: Build DeepFed time-series model (GRU + CNN + MLP)
    Based on the paper architecture
    """
    print("\n" + "=" * 80)
    print("PHASE 3: BUILDING DEEPFED TIME-SERIES MODEL")
    print("=" * 80)

    seq_length, num_features = input_shape

    print(f"\nModel Configuration:")
    print(f"  • Input shape: ({seq_length}, {num_features})")
    print(f"  • Output classes: {num_classes}")
    print(f"  • Architecture: GRU + Conv1D + MLP (Parallel branches)")

    # Input layer
    input_layer = layers.Input(shape=(seq_length, num_features), name='input')

    # ============================================================
    # BRANCH 1: GRU Module (Sequential Pattern Detection)
    # ============================================================
    print("\n[Branch 1: GRU Module]")

    # GRU works directly on (batch, seq_len, features) format
    x_gru = input_layer

    # First GRU layer
    x_gru = layers.GRU(128, return_sequences=True, name='gru_1')(x_gru)

    # Second GRU layer
    x_gru = layers.GRU(128, return_sequences=False, name='gru_2')(x_gru)

    print(f"  • GRU Layer 1: 128 units (return sequences)")
    print(f"  • GRU Layer 2: 128 units (return last)")
    print(f"  • Output shape: (batch, 128)")

    # ============================================================
    # BRANCH 2: CNN Module (Spatial Feature Extraction)
    # ============================================================
    print("\n[Branch 2: CNN Module]")

    # Permute for Conv1D: (batch, seq_len, features) -> (batch, features, seq_len)
    x_cnn = layers.Permute((2, 1), name='cnn_permute')(input_layer)

    # Conv Block 1
    x_cnn = layers.Conv1D(32, kernel_size=3, padding='same', name='conv_1')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_1')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_1')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_1')(x_cnn)

    # Conv Block 2
    x_cnn = layers.Conv1D(64, kernel_size=3, padding='same', name='conv_2')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_2')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_2')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_2')(x_cnn)

    # Conv Block 3
    x_cnn = layers.Conv1D(128, kernel_size=3, padding='same', name='conv_3')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_3')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_3')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_3')(x_cnn)

    # Flatten CNN output
    x_cnn = layers.Flatten(name='cnn_flatten')(x_cnn)

    print(f"  • Conv Block 1: 32 filters")
    print(f"  • Conv Block 2: 64 filters")
    print(f"  • Conv Block 3: 128 filters")
    print(f"  • Each with BatchNorm, ReLU, MaxPool")

    # ============================================================
    # CONCATENATE: Merge GRU and CNN features
    # ============================================================
    print("\n[Feature Fusion]")
    concatenated = layers.Concatenate(name='concatenate')([x_cnn, x_gru])
    print(f"  • Concatenating GRU and CNN features")

    # ============================================================
    # MLP Module (Classification Head)
    # ============================================================
    print("\n[Classification Head: MLP]")

    x = layers.Dense(128, activation='relu', name='fc_1')(concatenated)
    x = layers.Dense(64, activation='relu', name='fc_2')(x)
    x = layers.Dropout(0.5, name='dropout')(x)

    # Output layer
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    print(f"  • Dense Layer 1: 128 units")
    print(f"  • Dense Layer 2: 64 units")
    print(f"  • Dropout: 0.5")
    print(f"  • Output Layer: {num_classes} units (softmax)")

    # Create model
    model = models.Model(inputs=input_layer, outputs=output, name='DeepFed')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*80}")
    print("MODEL SUMMARY")
    print(f"{'='*80}")
    model.summary()

    total_params = model.count_params()
    print(f"\nTotal parameters: {total_params:,}")
    print(f"{'='*80}")

    return model


class LazyDataGenerator(keras.utils.PyDataset):
    """
    Memory-efficient data generator that loads data in batches from disk
    Similar to tf.data.Dataset for lazy loading
    """
    def __init__(self, data_file, indices, batch_size=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data_file = data_file
        self.indices = indices
        self.batch_size = batch_size if batch_size is not None else BATCH_SIZE
        self.shuffle = shuffle
        self.num_samples = len(indices)
        self.num_batches = int(np.ceil(self.num_samples / self.batch_size))

        # Memory-map the files for efficient access
        self.X = np.load(self.data_file['X'], mmap_mode='r')
        self.y = np.load(self.data_file['y'], mmap_mode='r')

    def __len__(self):
        """Number of batches per epoch"""
        return self.num_batches

    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min(start_idx + self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]

        # Load batch from memory-mapped files
        batch_X = self.X[batch_indices].copy()  # Copy to ensure contiguous array
        batch_y = self.y[batch_indices].copy()

        return batch_X, batch_y

    def on_epoch_end(self):
        """Shuffle indices at end of epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)


def create_data_generator(X, y, batch_size=None, shuffle=True):
    """
    Create a data generator to avoid loading entire dataset in memory
    Legacy function for backward compatibility
    """
    if batch_size is None:
        batch_size = BATCH_SIZE

    num_samples = len(X)
    indices = np.arange(num_samples)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_x = X[batch_indices]
            batch_y = y[batch_indices]

            yield batch_x, batch_y


def train_model(model, X_train, y_train, X_test, y_test):
    """
    Phase 4: Train the DeepFed model
    """
    print("\n" + "=" * 80)
    print("PHASE 4: TRAINING DEEPFED MODEL")
    print("=" * 80)

    print(f"\nTraining Configuration:")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")
    print(f"  • Training samples: {len(X_train):,}")
    print(f"  • Validation samples: {len(X_test):,}")
    print(f"  • Steps per epoch: {len(X_train) // BATCH_SIZE}")

    # Callbacks
    model_callbacks = [
        callbacks.ModelCheckpoint(
            filepath=str(Path(MODEL_DIR) / 'deepfed_best.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        callbacks.CSVLogger(
            filename=str(Path(MODEL_DIR) / 'training_log.csv'),
            separator=',',
            append=False
        )
    ]

    print(f"\n{'='*80}")
    print("STARTING TRAINING")
    print(f"{'='*80}\n")

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=model_callbacks,
        verbose=1
    )

    # Plot training history
    print(f"\n{'='*80}")
    print("TRAINING COMPLETED")
    print(f"{'='*80}")

    # Visualize training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Training')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'training_history.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved training history: {Path(VIS_DIR) / 'training_history.png'}")
    plt.close()

    return history


def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Phase 5: Evaluate the trained model
    """
    print("\n" + "=" * 80)
    print("PHASE 5: MODEL EVALUATION")
    print("=" * 80)

    print("\nGenerating predictions...")
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{'='*80}")
    print("OVERALL METRICS")
    print(f"{'='*80}")
    print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nNOTE: Accuracy can be misleading in this highly imbalanced dataset.")
    print("      Focus on metrics like F1-score, precision, and recall,")
    print("      especially for minority classes, as shown in the Classification Report.")
    print("-" * 80)
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"{'='*80}")

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 80)
    # Convert label_encoder.classes_ (which are numpy.int8) to strings
    target_names_str = [str(cls) for cls in label_encoder.classes_]
    print(classification_report(
        y_test, y_pred,
        target_names=target_names_str,
        digits=4
    ))

    # Per-class metrics
    print("\nPER-CLASS ACCURACY")
    print("-" * 80)
    for i, class_name in enumerate(label_encoder.classes_):
        mask = y_test == i
        if mask.sum() > 0:
            class_acc = accuracy_score(y_test[mask], y_pred[mask])
            print(f"{str(class_name):30s}: {class_acc:.4f} ({class_acc*100:5.2f}%) [{mask.sum():>6,} samples]")

    # Confusion matrix
    print("\nCONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(y_test, y_pred)

    # Visualize confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {Path(VIS_DIR) / 'confusion_matrix.png'}")
    plt.close()

    # Save confusion matrix
    np.save(Path(MODEL_DIR) / 'confusion_matrix.npy', cm)

    return accuracy, f1_macro, f1_weighted


def save_model_artifacts(model, label_encoder, scaler, metadata):
    """
    Save all model artifacts
    """
    print("\n" + "=" * 80)
    print("SAVING MODEL ARTIFACTS")
    print("=" * 80)

    # Save full model
    model_path = Path(MODEL_DIR) / 'deepfed_final.keras'
    model.save(model_path)
    print(f"✓ Model saved: {model_path}")

    # Save label encoder
    le_path = Path(MODEL_DIR) / 'label_encoder.pkl'
    with open(le_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved: {le_path}")

    # Save scaler
    scaler_path = Path(MODEL_DIR) / 'scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved: {scaler_path}")

    # Save metadata
    meta_path = Path(MODEL_DIR) / 'model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved: {meta_path}")

    print(f"\n{'='*80}")
    print("ALL ARTIFACTS SAVED")
    print(f"{'='*80}")


# ============================================================================
# COMPARISON FUNCTIONS FOR IMBALANCED DATASET HANDLING
# ============================================================================

def train_baseline(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 1: Baseline - Train without any imbalance handling
    """
    print("\n" + "="*80)
    print("APPROACH 1: BASELINE (No Imbalance Handling)")
    print("="*80)

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Create optimized datasets
    train_dataset = create_optimized_tf_dataset(X_train, y_train, batch_size=BATCH_SIZE, autotune=True)
    test_dataset = create_optimized_tf_dataset(X_test, y_test, batch_size=BATCH_SIZE, autotune=True)

    # Early stopping callback
    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with optimized dataset
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=test_dataset,
        callbacks=[early_stop],
        verbose=1
    )

    # Generate predictions for evaluation
    y_pred_prob = model.predict(test_dataset, verbose=0)

    return model, history, y_pred_prob


def train_with_class_weights(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 2: Class Weights - Penalize misclassification of minority classes
    """
    print("\n" + "="*80)
    print("APPROACH 2: CLASS WEIGHTS")
    print("="*80)

    # Compute class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train),
        y=y_train
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print("\nComputed class weights:")
    for cls, weight in class_weight_dict.items():
        print(f"  Class {cls}: {weight:.4f}")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Create optimized datasets
    train_dataset = create_optimized_tf_dataset(X_train, y_train, batch_size=BATCH_SIZE, autotune=True)
    test_dataset = create_optimized_tf_dataset(X_test, y_test, batch_size=BATCH_SIZE, autotune=True)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with class weights
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=test_dataset,
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    # Generate predictions for evaluation
    y_pred_prob = model.predict(test_dataset, verbose=0)

    return model, history, y_pred_prob


def train_with_oversampling(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 3: SMOTE Oversampling - Generate synthetic samples for minority classes
    """
    print("\n" + "="*80)
    print("APPROACH 3: SMOTE OVERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before SMOTE:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for SMOTE (flatten time series)
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply SMOTE
    print("\nApplying SMOTE...")
    smote = SMOTE(random_state=42, k_neighbors=5)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after SMOTE:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Create optimized datasets
    train_dataset = create_optimized_tf_dataset(X_train_resampled, y_train_resampled, batch_size=BATCH_SIZE, autotune=True)
    test_dataset = create_optimized_tf_dataset(X_test, y_test, batch_size=BATCH_SIZE, autotune=True)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=test_dataset,
        callbacks=[early_stop],
        verbose=1
    )

    # Generate predictions for evaluation
    y_pred_prob = model.predict(test_dataset, verbose=0)

    return model, history, y_pred_prob


def train_with_undersampling(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 4: Random Undersampling - Reduce majority class samples
    """
    print("\n" + "="*80)
    print("APPROACH 4: RANDOM UNDERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before undersampling:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for undersampling
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply random undersampling
    print("\nApplying random undersampling...")
    rus = RandomUnderSampler(random_state=42)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after undersampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Create optimized datasets
    train_dataset = create_optimized_tf_dataset(X_train_resampled, y_train_resampled, batch_size=BATCH_SIZE, autotune=True)
    test_dataset = create_optimized_tf_dataset(X_test, y_test, batch_size=BATCH_SIZE, autotune=True)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=test_dataset,
        callbacks=[early_stop],
        verbose=1
    )

    # Generate predictions for evaluation
    y_pred_prob = model.predict(test_dataset, verbose=0)

    return model, history, y_pred_prob


def train_with_combined(X_train, y_train, X_test, y_test, num_classes):
    """
    Approach 5: Combined SMOTE-Tomek - Oversample minority + clean decision boundary
    """
    print("\n" + "="*80)
    print("APPROACH 5: COMBINED (SMOTE + Tomek Links)")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal training set size: {X_train.shape[0]:,}")
    print("Class distribution before resampling:")
    unique, counts = np.unique(y_train, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train)*100:5.2f}%)")

    # Reshape for resampling
    n_samples, n_timesteps, n_features = X_train.shape
    X_train_flat = X_train.reshape(n_samples, n_timesteps * n_features)

    # Apply SMOTE-Tomek
    print("\nApplying SMOTE + Tomek Links...")
    smote_tomek = SMOTETomek(random_state=42)
    X_train_resampled, y_train_resampled = smote_tomek.fit_resample(X_train_flat, y_train)

    # Reshape back to 3D
    X_train_resampled = X_train_resampled.reshape(-1, n_timesteps, n_features)

    print(f"\nResampled training set size: {X_train_resampled.shape[0]:,}")
    print("Class distribution after resampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train.shape[1:], num_classes=num_classes)

    # Create optimized datasets
    train_dataset = create_optimized_tf_dataset(X_train_resampled, y_train_resampled, batch_size=BATCH_SIZE, autotune=True)
    test_dataset = create_optimized_tf_dataset(X_test, y_test, batch_size=BATCH_SIZE, autotune=True)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    # Train with resampled data
    history = model.fit(
        train_dataset,
        epochs=EPOCHS,
        validation_data=test_dataset,
        callbacks=[early_stop],
        verbose=1
    )

    # Generate predictions for evaluation
    y_pred_prob = model.predict(test_dataset, verbose=0)

    return model, history, y_pred_prob


def evaluate_and_compare(predictions_list, histories, X_test, y_test, label_encoder, approach_names):
    """
    Evaluate all approaches and create comparison visualizations
    """
    print("\n" + "="*80)
    print("COMPARISON EVALUATION")
    print("="*80)

    results = []

    for i, (y_pred_prob, history, approach) in enumerate(zip(predictions_list, histories, approach_names)):
        if y_pred_prob is None:
            print(f"\nSkipping {approach} (not available)")
            continue

        print(f"\n{'-'*80}")
        print(f"Evaluating: {approach}")
        print(f"{'-'*80}")

        # Predictions (already generated during training)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        # Aggregate metrics
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)

        print(f"Accuracy:          {accuracy:.4f}")
        print(f"Macro F1:          {macro_f1:.4f}")
        print(f"Weighted F1:       {weighted_f1:.4f}")
        print(f"Macro Precision:   {macro_precision:.4f}")
        print(f"Macro Recall:      {macro_recall:.4f}")

        results.append({
            'approach': approach,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'per_class_f1': f1.tolist(),
            'per_class_precision': precision.tolist(),
            'per_class_recall': recall.tolist(),
            'per_class_support': support.tolist(),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })

    # Create comparison visualizations
    if len(results) > 0:
        create_comparison_plots(results, label_encoder)
        save_comparison_results(results)

    return results


def cleanup_model_memory(model, approach_name, save_weights=True):
    """
    Clean up model memory after training to free GPU/CPU resources
    """
    if model is None:
        return

    try:
        # Save model weights if requested
        if save_weights:
            weights_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_weights.h5"
            weights_path.parent.mkdir(parents=True, exist_ok=True)
            model.save_weights(str(weights_path))
            print(f"✓ Model weights saved: {weights_path}")

        # Clear Keras/TensorFlow session and delete model
        if hasattr(model, 'clear_session'):
            model.clear_session()

        # Delete model object
        del model

        # Force garbage collection
        gc.collect()

        # Clear any cached Keras/TF graphs
        try:
            import keras.backend as K
            K.clear_session()
        except:
            pass

        print(f"✓ Memory cleaned up for: {approach_name}")

    except Exception as e:
        print(f"⚠️  Warning during cleanup for {approach_name}: {e}")


def create_comparison_plots(results, label_encoder):
    """
    Create comprehensive comparison visualizations
    """
    print("\n" + "="*80)
    print("CREATING COMPARISON VISUALIZATIONS")
    print("="*80)

    approaches = [r['approach'] for r in results]

    # 1. Overall Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision']
    metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score', 'Macro Precision']

    for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
        ax = axes[idx // 2, idx % 2]
        values = [r[metric] for r in results]
        bars = ax.bar(approaches, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
        ax.set_ylabel(name, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_title(f'{name} Comparison', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_overall_metrics.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_overall_metrics.png'}")
    plt.close()

    # 2. Per-Class F1-Score Comparison
    num_classes = len(results[0]['per_class_f1'])
    x = np.arange(num_classes)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 8))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, result in enumerate(results):
        offset = width * (i - len(results)/2 + 0.5)
        ax.bar(x + offset, result['per_class_f1'], width,
               label=result['approach'], color=colors[i % len(colors)])

    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title('Per-Class F1-Score Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_encoder.classes_], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_per_class_f1.png'}")
    plt.close()

    # 3. Macro Recall Comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    values = [r['macro_recall'] for r in results]
    bars = ax.bar(approaches, values,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
    ax.set_ylabel('Macro Recall', fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title('Macro Recall Comparison (Minority Class Performance)',
                fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_macro_recall.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_macro_recall.png'}")
    plt.close()


def save_comparison_results(results):
    """
    Save comparison results to JSON and generate text summary
    """
    # Save JSON
    json_path = Path(MODEL_DIR) / 'comparison_results.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved: {json_path}")

    # Create text summary
    summary_path = Path(MODEL_DIR) / 'comparison_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("IMBALANCED DATASET HANDLING COMPARISON SUMMARY\n")
        f.write("="*80 + "\n\n")

        for result in results:
            f.write(f"\n{result['approach']}\n")
            f.write("-"*80 + "\n")
            f.write(f"  Accuracy:          {result['accuracy']:.4f}\n")
            f.write(f"  Macro F1-Score:    {result['macro_f1']:.4f}\n")
            f.write(f"  Weighted F1-Score: {result['weighted_f1']:.4f}\n")
            f.write(f"  Macro Precision:   {result['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall:      {result['macro_recall']:.4f}\n")

        # Best approach per metric
        f.write("\n" + "="*80 + "\n")
        f.write("BEST APPROACHES PER METRIC\n")
        f.write("="*80 + "\n")

        metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']
        metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score',
                       'Macro Precision', 'Macro Recall']

        for metric, name in zip(metrics, metric_names):
            best = max(results, key=lambda x: x[metric])
            f.write(f"\n{name:20s}: {best['approach']:30s} ({best[metric]:.4f})\n")

    print(f"✓ Saved: {summary_path}")


def main():
    """
    Main execution pipeline with efficient data loading and caching
    """
    print("\n" + "=" * 80)
    print("DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM")
    print("Dataset: Edge-IIoTset")
    print("Model: GRU + CNN (Time-Series Architecture)")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  • Classification: {'Multi-class (attack types)' if USE_MULTICLASS else 'Binary (normal vs attack)'}")
    print(f"  • Dataset: {'Full dataset' if SAMPLE_SIZE is None else f'Sampled ({SAMPLE_SIZE:,} rows/file)'}")
    print(f"  • Use cached data: {USE_CACHED_DATA}")
    print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Batch size: Dynamic (GPU-optimized, determined after data loading)")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")

    # Check what's already cached
    cached_sequences = all(Path(CACHE_DIR, f).exists() for f in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy'])
    cached_hdf5 = HDF5_DATASET.exists()

    print(f"\nCache Status:")
    print(f"  • HDF5 preprocessed: {'✓ Found' if cached_hdf5 else '✗ Not found'}")
    print(f"  • Sequence arrays: {'✓ Found' if cached_sequences else '✗ Not found'}")
    if cached_sequences and USE_CACHED_DATA:
        print(f"  → Will skip CSV parsing and sequence creation!")

    try:
        # Check if dataset exists
        csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
        if not csv_exists:
            print("\nDataset not found. Downloading...")
            if not download_dataset():
                print("\n✗ Download failed. Please download manually:")
                print(f"   https://www.kaggle.com/datasets/{DATASET_NAME}")
                return 1

        # Phase 1: Explore dataset
        df, label_col = explore_dataset()

        # Phase 2: Prepare time-series data
        X_train, X_test, y_train, y_test, le, scaler, num_classes = \
            prepare_time_series_data(df, label_col)

        # Phase 2.5: Determine optimal batch size for GPU utilization
        BATCH_SIZE = determine_optimal_batch_size_tf(
            X_train_shape=X_train.shape,
            num_classes=num_classes,
            target_memory_utilization=0.8
        )
        print(f"\n✓ Optimal batch size determined: {BATCH_SIZE}")

        # Phase 3-5: Train and evaluate all 5 comparison approaches
        print("\n" + "="*80)
        print("STARTING COMPARISON STUDY")
        print("Training 5 Different Approaches for Handling Imbalanced Data")
        print("="*80)

        approach_names = [
            "Baseline (No Handling)",
            "Class Weights",
            "SMOTE Oversampling",
            "Random Undersampling",
            "Combined (SMOTE-Tomek)"
        ]

        models = []
        histories = []
        predictions_list = []

        # Approach 1: Baseline
        print("\n" + "█"*80)
        print("█ APPROACH 1/5: BASELINE")
        print("█"*80)
        model1, hist1, pred1 = train_baseline(X_train, y_train, X_test, y_test, num_classes)
        models.append(model1)
        histories.append(hist1)
        predictions_list.append(pred1)
        if model1 is not None:
            cleanup_model_memory(model1, "Baseline")

        # Approach 2: Class Weights
        print("\n" + "█"*80)
        print("█ APPROACH 2/5: CLASS WEIGHTS")
        print("█"*80)
        model2, hist2, pred2 = train_with_class_weights(X_train, y_train, X_test, y_test, num_classes)
        models.append(model2)
        histories.append(hist2)
        predictions_list.append(pred2)
        if model2 is not None:
            cleanup_model_memory(model2, "Class_Weights")

        # Approach 3: SMOTE Oversampling
        print("\n" + "█"*80)
        print("█ APPROACH 3/5: SMOTE OVERSAMPLING")
        print("█"*80)
        model3, hist3, pred3 = train_with_oversampling(X_train, y_train, X_test, y_test, num_classes)
        models.append(model3)
        histories.append(hist3)
        predictions_list.append(pred3)
        if model3 is not None:
            cleanup_model_memory(model3, "SMOTE_Oversampling")

        # Approach 4: Random Undersampling
        print("\n" + "█"*80)
        print("█ APPROACH 4/5: RANDOM UNDERSAMPLING")
        print("█"*80)
        model4, hist4, pred4 = train_with_undersampling(X_train, y_train, X_test, y_test, num_classes)
        models.append(model4)
        histories.append(hist4)
        predictions_list.append(pred4)
        if model4 is not None:
            cleanup_model_memory(model4, "Random_Undersampling")

        # Approach 5: Combined SMOTE-Tomek
        print("\n" + "█"*80)
        print("█ APPROACH 5/5: COMBINED (SMOTE-TOMEK)")
        print("█"*80)
        model5, hist5, pred5 = train_with_combined(X_train, y_train, X_test, y_test, num_classes)
        models.append(model5)
        histories.append(hist5)
        predictions_list.append(pred5)
        if model5 is not None:
            cleanup_model_memory(model5, "Combined_SMOTE_Tomek")

        # Evaluate and compare all approaches
        print("\n" + "="*80)
        print("ALL TRAINING COMPLETED - STARTING COMPARISON EVALUATION")
        print("="*80)

        results = evaluate_and_compare(predictions_list, histories, X_test, y_test, le, approach_names)

        # Save metadata
        metadata = {
            'model_name': 'DeepFed Comparison Study',
            'dataset': 'Edge-IIoTset',
            'architecture': 'GRU + CNN + MLP',
            'sequence_length': SEQUENCE_LENGTH,
            'num_features': X_train.shape[2],
            'num_classes': num_classes,
            'class_names': le.classes_.tolist(),
            'approaches': approach_names,
            'comparison_results': results,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS
        }

        # Save best model (highest macro F1)
        if results:
            best_idx = max(range(len(results)), key=lambda i: results[i]['macro_f1'])
            best_model = models[best_idx]
            best_approach = approach_names[best_idx]

            print(f"\n{'='*80}")
            print(f"BEST APPROACH: {best_approach}")
            print(f"  Macro F1-Score: {results[best_idx]['macro_f1']:.4f}")
            print(f"{'='*80}")

            save_model_artifacts(best_model, le, scaler, metadata)

        print("\n" + "=" * 80)
        print("✓ COMPARISON STUDY COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"\nResults Summary:")
        for i, (name, result) in enumerate(zip(approach_names, results)):
            print(f"\n{i+1}. {name}")
            print(f"   Accuracy:   {result['accuracy']:.4f}")
            print(f"   Macro F1:   {result['macro_f1']:.4f}")
            print(f"   Macro Recall: {result['macro_recall']:.4f}")
        print(f"\n  • Results saved in:     {MODEL_DIR}/")
        print(f"  • Visualizations in:    {VIS_DIR}/")
        print("=" * 80)

        return 0

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    exit_code = main()
    sys.exit(exit_code)

TensorFlow available for GPU detection

DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM
Dataset: Edge-IIoTset
Model: GRU + CNN (Time-Series Architecture)

Configuration:
  • Classification: Multi-class (attack types)
  • Dataset: Sampled (100,000 rows/file)
  • Use cached data: True
  • Sequence length: 128 time steps
  • Batch size: Dynamic (GPU-optimized, determined after data loading)
  • Epochs: 50
  • Learning rate: 0.001

Cache Status:
  • HDF5 preprocessed: ✓ Found
  • Sequence arrays: ✓ Found
  → Will skip CSV parsing and sequence creation!

PHASE 1: DATASET EXPLORATION & VISUALIZATION

✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING
Using cached file: cache/preprocessed/dataset.h5
Size: 1116.0 MB

Loading preprocessed data...
✓ Loaded 1,965,333 rows × 68 columns
  Memory usage: 3676.07 MB

1. DATASET STRUCTURE
--------------------------------------------------------------------------------
Memory usage: 3676.07 MB

Column types:
object            35
float64           31
int64

Model: "DeepFed"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 67)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 67, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 67, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 67, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 67, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 33, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 33, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 33, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 33, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 16, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 16, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 16, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 16, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 8, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     75,648 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 1024)      │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1152)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    147,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_2 (Dense)        │ (None, 64)        │      8,256 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 375,728 (1.43 MB)

 Trainable params: 375,280 (1.43 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 67)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 67, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 67, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 67, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 67, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 33, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 33, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 33, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 33, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 16, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 16, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 16, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 16, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 8, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     75,648 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 1024)      │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1152)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    147,584 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_2 (Dense)        │ (None, 64)        │      8,256 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 375,728 (1.43 MB)

 Trainable params: 375,280 (1.43 MB)

 Non-trainable params: 448 (1.75 KB)


Total parameters: 375,728
Epoch 1/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 5s 1s/step - accuracy: 0.0675 - loss: 3.0445 - val_accuracy: 0.3500 - val_loss: 3.8505
Epoch 2/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 434ms/step - accuracy: 0.3245 - loss: 2.3603 - val_accuracy: 0.5925 - val_loss: 4.2593
Epoch 3/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 430ms/step - accuracy: 0.4972 - loss: 2.1732 - val_accuracy: 0.6758 - val_loss: 4.3122
Epoch 4/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 433ms/step - accuracy: 0.5601 - loss: 1.9881 - val_accuracy: 0.7027 - val_loss: 4.2589
Epoch 5/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 428ms/step - accuracy: 0.5933 - loss: 1.8175 - val_accuracy: 0.7196 - val_loss: 4.1153
Epoch 6/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 431ms/step - accuracy: 0.6276 - loss: 1.6065 - val_accuracy: 0.7198 - val_loss: 3.8908
Epoch 7/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 438ms/step - accuracy: 0.6849 - loss: 1.4373 - val_accuracy: 0.7073 - val_loss: 3.7440
Epoch 8/50
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 442ms/step - accuracy: 0.6779 - loss: 1.4142 - val_accu

In [ ]:
"""
DeepFed: Intrusion Detection with Imbalanced Dataset Handling
==============================================================

Research-focused implementation comparing different approaches to handle
highly imbalanced network attack datasets (majority Normal, minority Attacks).

COMPARISON APPROACHES:
1. Baseline: Train on imbalanced data (no handling)
2. Class Weights: Penalize misclassification of minority classes
3. SMOTE Oversampling: Synthetic minority oversampling technique
4. Random Undersampling: Random undersampling of majority class
5. Combined: SMOTE + Random undersampling (hybrid approach)

Dataset: Edge-IIoTset - Cyber Security Dataset of IoT & IIoT
Model: DeepFed (GRU + CNN parallel architecture)

Based on DeepFed paper: "DeepFed: Federated Deep Learning for Intrusion Detection
in Industrial Cyber-Physical Systems"

INSTALLATION:
-------------
For full functionality (all 5 approaches), install imbalanced-learn:
    pip install imbalanced-learn

Without imbalanced-learn, only approaches 1 and 2 will run.

USAGE:
------
python deepfed_train.py

The script will:
- Load and preprocess Edge-IIoTset dataset
- Train 5 models with different imbalance handling approaches
- Evaluate and compare all approaches
- Generate comparison visualizations
- Save results to models/deepfed/ and visualizations/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import subprocess
import sys
import pickle
from collections import Counter
import json
import gc  # For garbage collection

# Check for GPU availability and dynamic batch sizing
try:
    import tensorflow as tf
    TF_AVAILABLE = True
    print("TensorFlow available for GPU detection")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available, using CPU-only batch sizing")

# Use Keras 3 (not tensorflow.keras)
import keras
from keras import layers, models, callbacks, ops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for handling imbalanced datasets
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("WARNING: imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("         Only baseline and class weights approaches will be available.")
    IMBLEARN_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

# Check for required packages
try:
    import tables  # For HDF5 support
except ImportError:
    print("ERROR: pytables is required for HDF5 export. Please install with `pip install tables`.")
    sys.exit(1)

# Configuration
DATASET_NAME = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
DATA_DIR = "./data/edge_iiot"
MODEL_DIR = "./models/deepfed"
VIS_DIR = "./visualizations"
CACHE_DIR = "./cache"
PREPROCESSED_DIR = "./cache/preprocessed"  # For efficient binary format
HDF5_DATASET = Path(PREPROCESSED_DIR) / "dataset.h5"
BATCH_SIZE = 256  # Fixed batch size for consistent training
EPOCHS = 30  # Reduced from 50 to prevent overfitting
LEARNING_RATE = 0.001
RANDOM_STATE = 42
SEQUENCE_LENGTH = 128  # Time steps for time-series sequences
WINDOW_STRIDE = 64  # Stride for sliding window sequence creation
VALIDATION_SPLIT = 0.2
SAMPLE_SIZE = 100000  # Sample size per file; set to None for full dataset
USE_MULTICLASS = True  # Use multi-class attack type classification
USE_CACHED_DATA = True  # Use preprocessed binary data if available

# Set random seeds
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Create directories
for dir_path in [DATA_DIR, MODEL_DIR, VIS_DIR, CACHE_DIR, PREPROCESSED_DIR]:
    os.makedirs(dir_path, exist_ok=True)


def download_dataset():
    """Download Edge-IIoTset dataset from Kaggle"""
    print("\n" + "=" * 80)
    print("DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE")
    print("=" * 80)

    # Check if dataset is already downloaded
    csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
    if csv_exists:
        print("✓ Dataset already exists - skipping download")
        print(f"Found CSV files in: {DATA_DIR}")
        return True

    print("Dataset not found locally - proceeding with download...")

    # Setup Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        print("✓ Running in Google Colab - using secrets")

        # Get credentials from Colab secrets
        kaggle_username = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')

        if not kaggle_username or not kaggle_key:
            raise ValueError("KAGGLE_USERNAME and KAGGLE_KEY must be set in Colab secrets")

        # Set environment variables for Kaggle API
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key

        print(f"  • Username: {kaggle_username}")
        print(f"  • API Key: {'*' * len(kaggle_key)}")

    except ImportError:
        print("✓ Not running in Colab - using default kaggle.json authentication")
    except Exception as e:
        print(f"✗ Error setting up Kaggle credentials: {e}")
        print("\nPlease add these secrets in Colab:")
        print("  1. Click the key icon (🔑) in the left sidebar")
        print("  2. Add secret: KAGGLE_USERNAME")
        print("  3. Add secret: KAGGLE_KEY")
        print("\nGet your credentials from: https://www.kaggle.com/settings/account")
        raise

    try:
        import kaggle
    except ImportError:
        print("Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
        import kaggle

    try:
        print(f"\nDownloading {DATASET_NAME}...")
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET_NAME, "-p", DATA_DIR
        ], check=True)

        # Extract zip files
        for zip_file in Path(DATA_DIR).glob("*.zip"):
            print(f"Extracting {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            zip_file.unlink()

        print("✓ Dataset downloaded and extracted successfully!")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        print(f"\nPlease download manually from:")
        print(f"https://www.kaggle.com/datasets/{DATASET_NAME}")
        return False


def convert_csv_to_binary():
    """
    Convert CSV files to a consolidated HDF5 dataset while preserving all features.
    Adds source metadata and derived temporal features for downstream processing.
    This function only creates the HDF5 file and returns its path.
    """
    preprocessed_file = HDF5_DATASET

    if preprocessed_file.exists() and USE_CACHED_DATA:
        print("\n" + "=" * 80)
        print("✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING")
        print("=" * 80)
        print(f"Using cached file: {preprocessed_file}")
        print(f"Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")
        return preprocessed_file

    print("\n" + "=" * 80)
    print("CONVERTING CSV TO EFFICIENT BINARY FORMAT")
    print("=" * 80)

    # Find all CSV files
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV files found!")

    print(f"\n✓ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

    # Load and combine all CSVs with sampling
    if SAMPLE_SIZE:
        print(f"\nLoading sample data (max {SAMPLE_SIZE:,} rows per file)...")
    else:
        print(f"\nLoading FULL dataset (this may take a while)...")

    dfs = []
    manifest = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, nrows=SAMPLE_SIZE, low_memory=False)
            original_rows = len(df)

            # Normalize string columns to avoid mixed dtype issues
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            for col in object_cols:
                df[col] = df[col].astype(str).str.strip().fillna('__NA__')

            # Attach source metadata
            df['source_file'] = csv_file.name
            df['source_path'] = str(csv_file.relative_to(DATA_DIR))
            df['source_category'] = csv_file.parent.name

            # Derive attack type from filename
            filename = csv_file.name
            if '_attack.csv' in filename:
                attack_type = filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                attack_type = 'Mixed'  # These contain multiple attack types
            else:
                attack_type = 'Normal'  # Sensor data files
            df['Attack_type'] = attack_type

            # Build temporal features without dropping original column
            if 'frame.time' in df.columns:
                time_str = df['frame.time'].astype(str).str.strip()
                parsed_time = pd.to_datetime(time_str, format='%Y %H:%M:%S.%f', errors='coerce')
                if parsed_time.isna().all():
                    parsed_time = pd.to_datetime(time_str, errors='coerce')
                parsed_time = parsed_time.ffill().bfill()
                df['frame_time_datetime'] = parsed_time
                base_time = parsed_time.iloc[0]
                rel_seconds = (parsed_time - base_time).dt.total_seconds()
                df['frame_time_relative_sec'] = rel_seconds.astype('float64')

            dfs.append(df)
            duration = float(df.get('frame_time_relative_sec', pd.Series([0])).max()) if len(df) else 0.0
            manifest.append({
                'file': str(csv_file.relative_to(DATA_DIR)),
                'rows_loaded': int(original_rows),
                'duration_seconds': duration
            })
            print(f"  ✓ {csv_file.name}: {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading {csv_file.name}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*80}")
    print(f"Combined dataset: {len(df):,} rows × {len(df.columns)} columns")
    print(f"{'='*80}")

    print("\nPreparing data for HDF5 storage...")
    df_filtered = df.copy()
    print(f"Final dataset shape: {df_filtered.shape}")
    print(f"Memory usage: {df_filtered.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    preprocessed_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle HDF5 file locking issues (common in Colab)
    if preprocessed_file.exists():
        print(f"  → HDF5 file exists, removing to recreate...")
        try:
            preprocessed_file.unlink()  # Remove the existing file
            print("  ✓ Removed existing HDF5 file")
        except Exception as e:
            print(f"  ✗ Could not remove existing file: {e}")
            print("  → Please manually delete the file and try again")
            raise RuntimeError(f"Cannot remove existing HDF5 file {preprocessed_file}. Please delete it manually.")

    # Save to HDF5
    df_filtered.to_hdf(preprocessed_file, key='data', mode='w', index=False)
    print(f"✓ Saved: {preprocessed_file}")
    print(f"  Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")

    total_csv_size = sum(f.stat().st_size for f in csv_files)
    compression_ratio = (1 - preprocessed_file.stat().st_size / total_csv_size) * 100
    print(f"  Compression: {compression_ratio:.1f}% savings over CSV")
    print(f"  Original CSV size: {total_csv_size / 1024**2:.1f} MB")

    manifest_path = Path(PREPROCESSED_DIR) / 'ingest_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest saved: {manifest_path}")

    return preprocessed_file


def prepare_time_series_data():
    """
    Load preprocessed data, split, and create test sequences.
    Returns flat train data and sequenced test data.
    """
    print("\n" + "=" * 80)
    print("PHASE 2: TIME-SERIES DATA PREPARATION")
    print("=" * 80)

    # Load the HDF5 file
    preprocessed_file = convert_csv_to_binary()
    print(f"\nLoading preprocessed data from {preprocessed_file}...")
    df = pd.read_hdf(preprocessed_file, key='data')

    print(f"✓ Loaded {len(df):,} rows × {len(df.columns)} columns")

    # Identify label column - PREFER Attack_type (multi-class) over Attack_label (binary)
    if USE_MULTICLASS and 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using MULTI-CLASS labels: '{label_col}'")
    elif 'Attack_label' in df.columns:
        label_col = 'Attack_label'
        print(f"✓ Using BINARY labels: '{label_col}'")
    elif 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using labels: '{label_col}'")
    else:
        raise ValueError("No label column found in dataset")

    print(f"  Label column: '{label_col}'")

    # Prepare features - exclude label and metadata columns
    metadata_cols = ['source_file', 'source_path', 'source_category', 'frame.time', 'frame_time_datetime',
                     'Attack_label', 'Attack_type']  # Exclude both potential label columns
    feature_cols = [col for col in df.columns if col != label_col and col not in metadata_cols]

    # Separate numeric and categorical
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df[feature_cols].select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    categorical_cols = [col for col in feature_cols if col not in numeric_cols and col not in datetime_cols]

    print(f"\n✓ Feature analysis:")
    print(f"  • Total features: {len(feature_cols)}")
    print(f"  • Numeric: {len(numeric_cols)}")
    print(f"  • Categorical: {len(categorical_cols)}")
    print(f"  • Datetime: {len(datetime_cols)}")

    X = df[feature_cols].copy()
    y = df[label_col].copy()

    # Convert datetime to numeric
    if datetime_cols:
        for col in datetime_cols:
            dt_series = pd.to_datetime(X[col], errors='coerce')
            dt_numeric = dt_series.view('int64').astype('float64')
            dt_numeric[dt_numeric == np.iinfo(np.int64).min] = np.nan
            X[col] = dt_numeric / 1e9

    # Encode categorical
    feature_encoder = None
    if categorical_cols:
        print("  • Encoding categorical columns...")
        X[categorical_cols] = X[categorical_cols].fillna('__MISSING__').astype(str)
        feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categorical_cols] = feature_encoder.fit_transform(X[categorical_cols])

    # Handle missing/infinite values
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_count = X.isna().sum().sum()
    if nan_count > 0:
        print(f"  • Filling {nan_count:,} NaN values...")
        X.fillna(X.median(), inplace=True)
        X.fillna(0, inplace=True)

    # Encode labels
    print("\n✓ Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(le.classes_)

    print(f"  • Number of classes: {num_classes}")
    for idx, class_name in enumerate(le.classes_):
        count = np.sum(y_encoded == idx)
        print(f"    {idx:3d}: {str(class_name):30s} ({count:,} samples)")

    # Scale features
    print("\n✓ Scaling features with RobustScaler...")
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X)
    print(f"  • Scaled data shape: {X_scaled.shape}")
    print(f"  • Range: [{X_scaled.min():.3f}, {X_scaled.max():.3f}]")

    # Train/test split on flat data
    print(f"\n✓ Train/test split ({VALIDATION_SPLIT*100:.0f}% test)...")
    X_train_flat, X_test_flat, y_train_flat, y_test_flat = train_test_split(
        X_scaled, y_encoded,
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    )

    print(f"  • Training samples: {X_train_flat.shape[0]:,}")
    print(f"  • Test samples: {X_test_flat.shape[0]:,}")

    # Create test sequences
    print(f"\n✓ Creating test sequences...")
    X_test, y_test = create_sequences(X_test_flat, y_test_flat, time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE)
    print(f"  • Test sequences: {X_test.shape}")

    print(f"{'='*80}")

    return X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes


def create_sequences(X, y, time_steps=128, stride=64):
    """
    Create overlapping sequences for time-series analysis.

    Args:
        X: Input features array
        y: Target labels array
        time_steps: Number of time steps per sequence
        stride: Stride (step size) for sliding window

    Returns:
        Tuple of (sequences, labels) for model training
    """
    sequences = []
    labels = []

    # --- OPTIMIZED SEQUENCE CREATION ---
    for start in range(0, len(X) - time_steps + 1, stride):
        end = start + time_steps
        seq = X[start:end]
        label = y[end - 1]  # Use the label of the last time step

        sequences.append(seq)
        labels.append(label)

    print(f"✓ Created {len(sequences):,} sequences (shape: {sequences[0].shape})")

    return np.array(sequences), np.array(labels)


def create_sliding_window_dataset(X, y, window_size=128, stride=64, batch_size=32):
    """
    Create a TensorFlow dataset with sliding window sequences.

    Args:
        X: Input features array
        y: Target labels array
        window_size: Size of the sliding window (number of time steps)
        stride: Stride (step size) for sliding window
        batch_size: Batch size for the TensorFlow dataset

    Returns:
        tf.data.Dataset object ready for training
    """
    # --- SLIDING WINDOW DATASET CREATION ---
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    # Create sliding windows
    dataset = dataset.window(
        size=window_size,
        shift=stride,
        drop_remainder=True
    )

    # Flatten the windows into samples
    dataset = dataset.flat_map(lambda window: window.batch(window_size))

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    print(f"✓ Created sliding window dataset with {window_size} time steps per window")

    return dataset


def build_deepfed_model(input_shape, num_classes):
    """
    Phase 3: Build DeepFed time-series model (GRU + CNN + MLP)
    Based on the paper architecture
    """
    print("\n" + "=" * 80)
    print("PHASE 3: BUILDING DEEPFED TIME-SERIES MODEL")
    print("=" * 80)

    seq_length, num_features = input_shape

    print(f"\nModel Configuration:")
    print(f"  • Input shape: ({seq_length}, {num_features})")
    print(f"  • Output classes: {num_classes}")
    print(f"  • Architecture: GRU + Conv1D + MLP (Parallel branches)")

    # Input layer
    input_layer = layers.Input(shape=(seq_length, num_features), name='input')

    # ============================================================
    # BRANCH 1: GRU Module (Sequential Pattern Detection)
    # ============================================================
    print("\n[Branch 1: GRU Module]")

    # GRU works directly on (batch, seq_len, features) format
    x_gru = input_layer

    # First GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, name='gru_1')(x_gru)

    # Second GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.2, name='gru_2')(x_gru)

    print(f"  • GRU Layer 1: 128 units (return sequences, dropout=0.2)")
    print(f"  • GRU Layer 2: 128 units (return last, dropout=0.2)")
    print(f"  • Output shape: (batch, 128)")

    # ============================================================
    # BRANCH 2: CNN Module (Spatial Feature Extraction)
    # ============================================================
    print("\n[Branch 2: CNN Module]")

    # Permute for Conv1D: (batch, seq_len, features) -> (batch, features, seq_len)
    x_cnn = layers.Permute((2, 1), name='cnn_permute')(input_layer)

    # Conv Block 1
    x_cnn = layers.Conv1D(32, kernel_size=3, padding='same', name='conv_1')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_1')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_1')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_1')(x_cnn)

    # Conv Block 2
    x_cnn = layers.Conv1D(64, kernel_size=3, padding='same', name='conv_2')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_2')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_2')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_2')(x_cnn)

    # Conv Block 3
    x_cnn = layers.Conv1D(128, kernel_size=3, padding='same', name='conv_3')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_3')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_3')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_3')(x_cnn)

    # Flatten CNN output
    x_cnn = layers.Flatten(name='cnn_flatten')(x_cnn)

    print(f"  • Conv Block 1: 32 filters")
    print(f"  • Conv Block 2: 64 filters")
    print(f"  • Conv Block 3: 128 filters")
    print(f"  • Each with BatchNorm, ReLU, MaxPool")

    # ============================================================
    # CONCATENATE: Merge GRU and CNN features
    # ============================================================
    print("\n[Feature Fusion]")
    concatenated = layers.Concatenate(name='concatenate')([x_cnn, x_gru])
    print(f"  • Concatenating GRU and CNN features")

    # ============================================================
    # MLP Module (Classification Head)
    # ============================================================
    print("\n[Classification Head: MLP]")

    x = layers.Dense(128, activation='relu', name='fc_1')(concatenated)
    x = layers.Dropout(0.3, name='dropout_1')(x)  # Add dropout after first dense
    x = layers.Dense(64, activation='relu', name='fc_2')(x)
    x = layers.Dropout(0.5, name='dropout_2')(x)  # Keep dropout after second dense

    # Output layer
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    print(f"  • Dense Layer 1: 128 units + Dropout(0.3)")
    print(f"  • Dense Layer 2: 64 units + Dropout(0.5)")
    print(f"  • Output Layer: {num_classes} units (softmax)")

    # Create model
    model = models.Model(inputs=input_layer, outputs=output, name='DeepFed')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*80}")
    print("MODEL SUMMARY")
    print(f"{'='*80}")
    model.summary()

    total_params = model.count_params()
    print(f"\nTotal parameters: {total_params:,}")
    print(f"{'='*80}")

    return model


class LazyDataGenerator(keras.utils.PyDataset):
    """
    Memory-efficient data generator that loads data in batches from disk
    Similar to tf.data.Dataset for lazy loading
    """
    def __init__(self, data_file, indices, batch_size=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data_file = data_file
        self.indices = indices
        self.batch_size = batch_size if batch_size is not None else BATCH_SIZE
        self.shuffle = shuffle
        self.num_samples = len(indices)
        self.num_batches = int(np.ceil(self.num_samples / self.batch_size))

        # Memory-map the files for efficient access
        self.X = np.load(self.data_file['X'], mmap_mode='r')
        self.y = np.load(self.data_file['y'], mmap_mode='r')

    def __len__(self):
        """Number of batches per epoch"""
        return self.num_batches

    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min(start_idx + self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]

        # Load batch from memory-mapped files
        batch_X = self.X[batch_indices].copy()  # Copy to ensure contiguous array
        batch_y = self.y[batch_indices].copy()

        return batch_X, batch_y

    def on_epoch_end(self):
        """Shuffle indices at end of epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)


def create_data_generator(X, y, batch_size=None, shuffle=True):
    """
    Create a data generator to avoid loading entire dataset in memory
    Legacy function for backward compatibility
    """
    if batch_size is None:
        batch_size = BATCH_SIZE

    num_samples = len(X)
    indices = np.arange(num_samples)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_x = X[batch_indices]
            batch_y = y[batch_indices]

            yield batch_x, batch_y


def train_model(model, X_train, y_train, X_test, y_test):
    """
    Phase 4: Train the DeepFed model
    """
    print("\n" + "=" * 80)
    print("PHASE 4: TRAINING DEEPFED MODEL")
    print("=" * 80)

    print(f"\nTraining Configuration:")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")
    print(f"  • Training samples: {len(X_train):,}")
    print(f"  • Validation samples: {len(X_test):,}")
    print(f"  • Steps per epoch: {len(X_train) // BATCH_SIZE}")

    # Callbacks
    model_callbacks = [
        callbacks.ModelCheckpoint(
            filepath=str(Path(MODEL_DIR) / 'deepfed_best.keras'),
            monitor='val_accuracy',
            save_best_only=True,
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=10,
            restore_best_weights=True,
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=5,
            min_lr=1e-7,
            verbose=1
        ),
        callbacks.CSVLogger(
            filename=str(Path(MODEL_DIR) / 'training_log.csv'),
            separator=',',
            append=False
        )
    ]

    print(f"\n{'='*80}")
    print("STARTING TRAINING")
    print(f"{'='*80}\n")

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=model_callbacks,
        verbose=1
    )

    # Plot training history
    print(f"\n{'='*80}")
    print("TRAINING COMPLETED")
    print(f"{'='*80}")

    # Visualize training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Training')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'training_history.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved training history: {Path(VIS_DIR) / 'training_history.png'}")
    plt.close()

    return history


def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Phase 5: Evaluate the trained model
    """
    print("\n" + "=" * 80)
    print("PHASE 5: MODEL EVALUATION")
    print("=" * 80)

    print("\nGenerating predictions...")
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{'='*80}")
    print("OVERALL METRICS")
    print(f"{'='*80}")
    print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nNOTE: Accuracy can be misleading in this highly imbalanced dataset.")
    print("      Focus on metrics like F1-score, precision, and recall,")
    print("      especially for minority classes, as shown in the Classification Report.")
    print("-" * 80)
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"{'='*80}")

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 80)
    # Convert label_encoder.classes_ (which are numpy.int8) to strings
    target_names_str = [str(cls) for cls in label_encoder.classes_]
    print(classification_report(
        y_test, y_pred,
        target_names=target_names_str,
        digits=4
    ))

    # Per-class metrics
    print("\nPER-CLASS ACCURACY")
    print("-" * 80)
    for i, class_name in enumerate(label_encoder.classes_):
        mask = y_test == i
        if mask.sum() > 0:
            class_acc = accuracy_score(y_test[mask], y_pred[mask])
            print(f"{str(class_name):30s}: {class_acc:.4f} ({class_acc*100:5.2f}%) [{mask.sum():>6,} samples]")

    # Confusion matrix
    print("\nCONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(y_test, y_pred)

    # Visualize confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {Path(VIS_DIR) / 'confusion_matrix.png'}")
    plt.close()

    # Save confusion matrix
    np.save(Path(MODEL_DIR) / 'confusion_matrix.npy', cm)

    return accuracy, f1_macro, f1_weighted


def save_model_artifacts(model, label_encoder, scaler, metadata):
    """
    Save all model artifacts
    """
    print("\n" + "=" * 80)
    print("SAVING MODEL ARTIFACTS")
    print("=" * 80)

    # Save full model
    model_path = Path(MODEL_DIR) / 'deepfed_final.keras'
    model.save(model_path)
    print(f"✓ Model saved: {model_path}")

    # Save label encoder
    le_path = Path(MODEL_DIR) / 'label_encoder.pkl'
    with open(le_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved: {le_path}")

    # Save scaler
    scaler_path = Path(MODEL_DIR) / 'scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved: {scaler_path}")

    # Save metadata
    meta_path = Path(MODEL_DIR) / 'model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved: {meta_path}")

    print(f"\n{'='*80}")
    print("ALL ARTIFACTS SAVED")
    print(f"{'='*80}")


# ============================================================================
# COMPARISON FUNCTIONS FOR IMBALANCED DATASET HANDLING
# ============================================================================

def train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 1: Baseline - Train without any imbalance handling
    TIME-SERIES PROPER: Create sequences from original flat data
    """
    print("\n" + "="*80)
    print("APPROACH 1: BASELINE (No Imbalance Handling)")
    print("="*80)

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from original flat data (no resampling)
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 2: Class Weights - Penalize misclassification of minority classes
    TIME-SERIES PROPER: Create sequences from original flat data, apply class weights
    """
    print("\n" + "="*80)
    print("APPROACH 2: CLASS WEIGHTS")
    print("="*80)

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from original flat data (no resampling)
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    # Compute class weights from sequence-level labels
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_sequences),
        y=y_train_sequences
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print("\nComputed class weights (based on sequence labels):")
    for cls, weight in class_weight_dict.items():
        print(f"  Class {cls}: {weight:.4f}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 3: SMOTE Oversampling - Generate synthetic samples for minority classes
    TIME-SERIES PROPER: Apply SMOTE on flat data, then create sequences
    """
    print("\n" + "="*80)
    print("APPROACH 3: SMOTE OVERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before SMOTE:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE on flat data
    print("\nApplying SMOTE on flat data...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, counts.min() - 1) if counts.min() > 1 else 1)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter SMOTE: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after SMOTE:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 4: Random Undersampling - Reduce majority class samples
    TIME-SERIES PROPER: Apply undersampling on flat data, then create sequences
    """
    print("\n" + "="*80)
    print("APPROACH 4: RANDOM UNDERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before undersampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply random undersampling on flat data
    print("\nApplying random undersampling on flat data...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter undersampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after undersampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 5: Combined SMOTE+UnderSampling - Oversample minority + undersample majority
    TIME-SERIES PROPER: Apply hybrid resampling on flat data, then create sequences

    NOTE: This approach combines SMOTE with RandomUnderSampler instead of Tomek links.
    Tomek links are extremely slow (10-100x slower) with minimal benefit (<1% improvement).
    This hybrid approach is much faster and often gives better results.
    """
    print("\n" + "="*80)
    print("APPROACH 5: COMBINED (SMOTE + Random UnderSampling)")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before resampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE first to oversample minority classes
    print("\n[Step 1/2] Applying SMOTE to oversample minority classes...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"After SMOTE: {X_smote.shape[0]:,} samples")
    unique, counts = np.unique(y_smote, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_smote)*100:5.2f}%)")

    # Apply undersampling to reduce majority class (faster than Tomek, better balance)
    print("\n[Step 2/2] Applying random undersampling for final balance...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy='auto')
    X_train_resampled, y_train_resampled = rus.fit_resample(X_smote, y_smote)

    print(f"\nAfter SMOTE+UnderSampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after combined resampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=10, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def evaluate_and_compare(predictions_list, histories, X_test, y_test, label_encoder, approach_names):
    """
    Evaluate all approaches and create comparison visualizations
    """
    print("\n" + "="*80)
    print("COMPARISON EVALUATION")
    print("="*80)

    results = []

    for i, (y_pred_prob, history, approach) in enumerate(zip(predictions_list, histories, approach_names)):
        if y_pred_prob is None:
            print(f"\nSkipping {approach} (not available)")
            continue

        print(f"\n{'-'*80}")
        print(f"Evaluating: {approach}")
        print(f"{'-'*80}")

        # Predictions (already generated during training)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        # Aggregate metrics
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)

        print(f"Accuracy:          {accuracy:.4f}")
        print(f"Macro F1:          {macro_f1:.4f}")
        print(f"Weighted F1:       {weighted_f1:.4f}")
        print(f"Macro Precision:   {macro_precision:.4f}")
        print(f"Macro Recall:      {macro_recall:.4f}")

        results.append({
            'approach': approach,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'per_class_f1': f1.tolist(),
            'per_class_precision': precision.tolist(),
            'per_class_recall': recall.tolist(),
            'per_class_support': support.tolist(),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })

    # Create comparison visualizations
    if len(results) > 0:
        create_comparison_plots(results, label_encoder)
        save_comparison_results(results)

    return results


def cleanup_model_memory(model, approach_name, save_weights=True):
    """
    Clean up model memory after training to free GPU/CPU resources
    """
    if model is None:
        return

    try:
        # Save model weights if requested
        if save_weights:
            weights_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_weights.h5"
            weights_path.parent.mkdir(parents=True, exist_ok=True)
            model.save_weights(str(weights_path))
            print(f"✓ Model weights saved: {weights_path}")

        # Clear Keras/TensorFlow session and delete model
        if hasattr(model, 'clear_session'):
            model.clear_session()

        # Delete model object
        del model

        # Force garbage collection
        gc.collect()

        # Clear any cached Keras/TF graphs
        try:
            import keras.backend as K
            K.clear_session()
        except:
            pass

        print(f"✓ Memory cleaned up for: {approach_name}")

    except Exception as e:
        print(f"⚠️  Warning during cleanup for {approach_name}: {e}")


def create_comparison_plots(results, label_encoder):
    """
    Create comprehensive comparison visualizations
    """
    print("\n" + "="*80)
    print("CREATING COMPARISON VISUALIZATIONS")
    print("="*80)

    approaches = [r['approach'] for r in results]

    # 1. Overall Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision']
    metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score', 'Macro Precision']

    for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
        ax = axes[idx // 2, idx % 2]
        values = [r[metric] for r in results]
        bars = ax.bar(approaches, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
        ax.set_ylabel(name, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_title(f'{name} Comparison', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_overall_metrics.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_overall_metrics.png'}")
    plt.close()

    # 2. Per-Class F1-Score Comparison
    num_classes = len(results[0]['per_class_f1'])
    x = np.arange(num_classes)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 8))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, result in enumerate(results):
        offset = width * (i - len(results)/2 + 0.5)
        ax.bar(x + offset, result['per_class_f1'], width,
               label=result['approach'], color=colors[i % len(colors)])

    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title('Per-Class F1-Score Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_encoder.classes_], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_per_class_f1.png'}")
    plt.close()

    # 3. Macro Recall Comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    values = [r['macro_recall'] for r in results]
    bars = ax.bar(approaches, values,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
    ax.set_ylabel('Macro Recall', fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title('Macro Recall Comparison (Minority Class Performance)',
                fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_macro_recall.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_macro_recall.png'}")
    plt.close()


def save_comparison_results(results):
    """
    Save comparison results to JSON and generate text summary
    """
    # Save JSON
    json_path = Path(MODEL_DIR) / 'comparison_results.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved: {json_path}")

    # Create text summary
    summary_path = Path(MODEL_DIR) / 'comparison_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("IMBALANCED DATASET HANDLING COMPARISON SUMMARY\n")
        f.write("="*80 + "\n\n")

        for result in results:
            f.write(f"\n{result['approach']}\n")
            f.write("-"*80 + "\n")
            f.write(f"  Accuracy:          {result['accuracy']:.4f}\n")
            f.write(f"  Macro F1-Score:    {result['macro_f1']:.4f}\n")
            f.write(f"  Weighted F1-Score: {result['weighted_f1']:.4f}\n")
            f.write(f"  Macro Precision:   {result['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall:      {result['macro_recall']:.4f}\n")

        # Best approach per metric
        f.write("\n" + "="*80 + "\n")
        f.write("BEST APPROACHES PER METRIC\n")
        f.write("="*80 + "\n")

        metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']
        metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score',
                       'Macro Precision', 'Macro Recall']

        for metric, name in zip(metrics, metric_names):
            best = max(results, key=lambda x: x[metric])
            f.write(f"\n{name:20s}: {best['approach']:30s} ({best[metric]:.4f})\n")

    print(f"✓ Saved: {summary_path}")


def main():
    """
    Main execution pipeline with efficient data loading and caching
    """
    print("\n" + "=" * 80)
    print("DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM")
    print("Dataset: Edge-IIoTset")
    print("Model: GRU + CNN (Time-Series Architecture)")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  • Classification: {'Multi-class (attack types)' if USE_MULTICLASS else 'Binary (normal vs attack)'}")
    print(f"  • Dataset: {'Full dataset' if SAMPLE_SIZE is None else f'Sampled ({SAMPLE_SIZE:,} rows/file)'}")
    print(f"  • Use cached data: {USE_CACHED_DATA}")
    print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Batch size: Dynamic (GPU-optimized, determined after data loading)")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")

    # Check what's already cached
    cached_sequences = all(Path(CACHE_DIR, f).exists() for f in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy'])
    cached_hdf5 = HDF5_DATASET.exists()

    print(f"\nCache Status:")
    print(f"  • HDF5 preprocessed: {'✓ Found' if cached_hdf5 else '✗ Not found'}")
    print(f"  • Sequence arrays: {'✓ Found' if cached_sequences else '✗ Not found'}")
    if cached_sequences and USE_CACHED_DATA:
        print(f"  → Will skip CSV parsing and sequence creation!")

    try:
        # Check if dataset exists
        csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
        if not csv_exists:
            print("\nDataset not found. Downloading...")
            if not download_dataset():
                print("\n✗ Download failed. Please download manually:")
                print(f"   https://www.kaggle.com/datasets/{DATASET_NAME}")
                return 1

        # Phase 1 & 2: Load and preprocess dataset
        X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes = prepare_time_series_data()

        print(f"\n✅ Data loaded successfully:")
        print(f"  • Training samples (flat): {X_train_flat.shape[0]:,}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Features: {X_train_flat.shape[1]}")
        print(f"  • Classes: {num_classes}")
        print(f"  • Class names: {le.classes_}")
        print(f"  • Batch size: {BATCH_SIZE}")

        # Phase 3: Train and evaluate all 5 comparison approaches
        print("\n" + "="*80)
        print("PHASE 3-5: COMPARISON STUDY")
        print("Training 5 Different Approaches for Handling Imbalanced Data")
        print("="*80)

        approach_names = [
            "Baseline (No Handling)",
            "Class Weights",
            "SMOTE Oversampling",
            "Random Undersampling",
            "Combined (SMOTE+UnderSample)"
        ]

        # All approaches will work with the same flat training data
        # Each approach will apply its resampling strategy, then create sequences
        print(f"\n📊 Training data prepared:")
        print(f"  • Flat training samples: {X_train_flat.shape[0]:,}")
        print(f"  • Features per sample: {X_train_flat.shape[1]}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Sequence shape: ({X_test.shape[1]}, {X_test.shape[2]})")

        models = []
        histories = []
        predictions_list = []

        # Approach 1: Baseline
        print("\n" + "█"*80)
        print("█ APPROACH 1/5: BASELINE")
        print("█"*80)
        model1, hist1, pred1 = train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes)
        models.append(model1)
        histories.append(hist1)
        predictions_list.append(pred1)
        if model1 is not None:
            cleanup_model_memory(model1, "Baseline")

        # Approach 2: Class Weights
        print("\n" + "█"*80)
        print("█ APPROACH 2/5: CLASS WEIGHTS")
        print("█"*80)
        model2, hist2, pred2 = train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes)
        models.append(model2)
        histories.append(hist2)
        predictions_list.append(pred2)
        if model2 is not None:
            cleanup_model_memory(model2, "Class_Weights")

        # Approach 3: SMOTE Oversampling
        print("\n" + "█"*80)
        print("█ APPROACH 3/5: SMOTE OVERSAMPLING")
        print("█"*80)
        model3, hist3, pred3 = train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append(model3)
        histories.append(hist3)
        predictions_list.append(pred3)
        if model3 is not None:
            cleanup_model_memory(model3, "SMOTE_Oversampling")

        # Approach 4: Random Undersampling
        print("\n" + "█"*80)
        print("█ APPROACH 4/5: RANDOM UNDERSAMPLING")
        print("█"*80)
        model4, hist4, pred4 = train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append(model4)
        histories.append(hist4)
        predictions_list.append(pred4)
        if model4 is not None:
            cleanup_model_memory(model4, "Random_Undersampling")

        # Approach 5: Combined SMOTE + UnderSampling
        print("\n" + "█"*80)
        print("█ APPROACH 5/5: COMBINED (SMOTE + UNDERSAMPLING)")
        print("█"*80)
        model5, hist5, pred5 = train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append(model5)
        histories.append(hist5)
        predictions_list.append(pred5)
        if model5 is not None:
            cleanup_model_memory(model5, "Combined_SMOTE_Tomek")

        # Evaluate and compare all approaches
        print("\n" + "="*80)
        print("ALL TRAINING COMPLETED - STARTING COMPARISON EVALUATION")
        print("="*80)

        results = evaluate_and_compare(predictions_list, histories, X_test, y_test, le, approach_names)

        # Save metadata
        metadata = {
            'model_name': 'DeepFed Comparison Study',
            'dataset': 'Edge-IIoTset',
            'architecture': 'GRU + CNN + MLP',
            'sequence_length': SEQUENCE_LENGTH,
            'num_features': X_test.shape[2],  # Use test sequences shape
            'num_classes': num_classes,
            'class_names': le.classes_.tolist(),
            'approaches': approach_names,
            'comparison_results': results,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS
        }

        # Save best model (highest macro F1)
        if results:
            best_idx = max(range(len(results)), key=lambda i: results[i]['macro_f1'])
            best_model = models[best_idx]
            best_approach = approach_names[best_idx]

            print(f"\n{'='*80}")
            print(f"BEST APPROACH: {best_approach}")
            print(f"  Macro F1-Score: {results[best_idx]['macro_f1']:.4f}")
            print(f"{'='*80}")

            save_model_artifacts(best_model, le, scaler, metadata)

        print("\n" + "=" * 80)
        print("✓ COMPARISON STUDY COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"\nResults Summary:")
        for i, (name, result) in enumerate(zip(approach_names, results)):
            print(f"\n{i+1}. {name}")
            print(f"   Accuracy:   {result['accuracy']:.4f}")
            print(f"   Macro F1:   {result['macro_f1']:.4f}")
            print(f"   Macro Recall: {result['macro_recall']:.4f}")
        print(f"\n  • Results saved in:     {MODEL_DIR}/")
        print(f"  • Visualizations in:    {VIS_DIR}/")
        print("=" * 80)

        return 0

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    exit_code = main()
    sys.exit(exit_code)

TensorFlow available for GPU detection

DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM
Dataset: Edge-IIoTset
Model: GRU + CNN (Time-Series Architecture)

Configuration:
  • Classification: Multi-class (attack types)
  • Dataset: Sampled (100,000 rows/file)
  • Use cached data: True
  • Sequence length: 128 time steps
  • Batch size: Dynamic (GPU-optimized, determined after data loading)
  • Epochs: 30
  • Learning rate: 0.001

Cache Status:
  • HDF5 preprocessed: ✗ Not found
  • Sequence arrays: ✗ Not found

PHASE 2: TIME-SERIES DATA PREPARATION

CONVERTING CSV TO EFFICIENT BINARY FORMAT

✓ Found 26 CSV file(s):
  - Password_attack.csv (623.0 MB)
  - DDoS_ICMP_Flood_attack.csv (850.8 MB)
  - SQL_injection_attack.csv (25.7 MB)
  - Ransomware_attack.csv (7.2 MB)
  - DDoS_TCP_SYN_Flood_attack.csv (897.6 MB)
  - Vulnerability_scanner_attack.csv (167.6 MB)
  - DDoS_UDP_Flood_attack.csv (858.5 MB)
  - DDoS_HTTP_Flood_attack.csv (91.3 MB)
  - OS_Fingerprinting_attack.csv (0.3 MB)
  - Backdoo

Model: "DeepFed"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 61)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 61, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 61, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 61, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 61, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 30, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 30, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 30, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 30, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 15, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 15, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 15, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 15, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 7, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     73,344 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 896)       │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    131,200 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 357,040 (1.36 MB)

 Trainable params: 356,592 (1.36 MB)

 Non-trainable params: 448 (1.75 KB)


Total parameters: 357,040
Epoch 1/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 64s 554ms/step - accuracy: 0.4622 - loss: 2.0983 - val_accuracy: 0.6199 - val_loss: 1.1432
Epoch 2/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 53s 547ms/step - accuracy: 0.6102 - loss: 1.1819 - val_accuracy: 0.7762 - val_loss: 0.7795
Epoch 3/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 52s 537ms/step - accuracy: 0.7191 - loss: 0.8795 - val_accuracy: 0.8321 - val_loss: 0.5533
Epoch 4/30
96/96 ━━━━━━━━━━━━━━━━━━━━ 51s 533ms/step - accuracy: 0.7769 - loss: 0.7187 - val_accuracy: 0.8407 - val_loss: 0.4780
Epoch 5/30
41/96 ━━━━━━━━━━━━━━━━━━━━ 27s 501ms/step - accuracy: 0.8008 - loss: 0.6241

KeyboardInterrupt: 

In [ ]:
"""
DeepFed: Intrusion Detection with Imbalanced Dataset Handling
==============================================================

Research-focused implementation comparing different approaches to handle
highly imbalanced network attack datasets (majority Normal, minority Attacks).

COMPARISON APPROACHES:
1. Baseline: Train on imbalanced data (no handling)
2. Class Weights: Penalize misclassification of minority classes
3. SMOTE Oversampling: Synthetic minority oversampling technique
4. Random Undersampling: Random undersampling of majority class
5. Combined: SMOTE + Random undersampling (hybrid approach)

Dataset: Edge-IIoTset - Cyber Security Dataset of IoT & IIoT
Model: DeepFed (GRU + CNN parallel architecture)

Based on DeepFed paper: "DeepFed: Federated Deep Learning for Intrusion Detection
in Industrial Cyber-Physical Systems"

INSTALLATION:
-------------
For full functionality (all 5 approaches), install imbalanced-learn:
    pip install imbalanced-learn

Without imbalanced-learn, only approaches 1 and 2 will run.

USAGE:
------
python deepfed_train.py

The script will:
- Load and preprocess Edge-IIoTset dataset
- Train 5 models with different imbalance handling approaches
- Evaluate and compare all approaches
- Generate comparison visualizations
- Save results to models/deepfed/ and visualizations/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import subprocess
import sys
import pickle
from collections import Counter
import json
import gc  # For garbage collection

# Check for GPU availability and dynamic batch sizing
try:
    import tensorflow as tf
    TF_AVAILABLE = True
    print("TensorFlow available for GPU detection")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available, using CPU-only batch sizing")

# Use Keras 3 (not tensorflow.keras)
import keras
from keras import layers, models, callbacks, ops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for handling imbalanced datasets
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("WARNING: imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("         Only baseline and class weights approaches will be available.")
    IMBLEARN_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

# Check for required packages
try:
    import tables  # For HDF5 support
except ImportError:
    print("ERROR: pytables is required for HDF5 export. Please install with `pip install tables`.")
    sys.exit(1)

# Configuration
DATASET_NAME = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
DATA_DIR = "./data/edge_iiot"
MODEL_DIR = "/content/drive/MyDrive/Projects/deepfed"  # Updated path for Google Drive
VIS_DIR = "./visualizations"
CACHE_DIR = "./cache"
PREPROCESSED_DIR = "./cache/preprocessed"  # For efficient binary format
HDF5_DATASET = Path(PREPROCESSED_DIR) / "dataset.h5"
BATCH_SIZE = 1024  # Fixed batch size for consistent training
EPOCHS = 100  # Sufficient epochs for convergence with early stopping
LEARNING_RATE = 0.001
RANDOM_STATE = 42
SEQUENCE_LENGTH = 128  # Time steps for time-series sequences
WINDOW_STRIDE = 64  # Stride for sliding window sequence creation
VALIDATION_SPLIT = 0.2
SAMPLE_SIZE = 100000  # Sample size per file; set to None for full dataset
USE_MULTICLASS = True  # Use multi-class attack type classification
USE_CACHED_DATA = True  # Use preprocessed binary data if available

# Set random seeds
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Create directories (skip if paths are not accessible)
for dir_path in [DATA_DIR, MODEL_DIR, VIS_DIR, CACHE_DIR, PREPROCESSED_DIR]:
    try:
        os.makedirs(dir_path, exist_ok=True)
    except (OSError, PermissionError) as e:
        print(f"Warning: Could not create directory {dir_path}: {e}")
        print("This is normal if running outside the target environment (e.g., Google Colab)")


def download_dataset():
    """Download Edge-IIoTset dataset from Kaggle"""
    print("\n" + "=" * 80)
    print("DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE")
    print("=" * 80)

    # Check if dataset is already downloaded
    csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
    if csv_exists:
        print("✓ Dataset already exists - skipping download")
        print(f"Found CSV files in: {DATA_DIR}")
        return True

    print("Dataset not found locally - proceeding with download...")

    # Setup Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        print("✓ Running in Google Colab - using secrets")

        # Get credentials from Colab secrets
        kaggle_username = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')

        if not kaggle_username or not kaggle_key:
            raise ValueError("KAGGLE_USERNAME and KAGGLE_KEY must be set in Colab secrets")

        # Set environment variables for Kaggle API
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key

        print(f"  • Username: {kaggle_username}")
        print(f"  • API Key: {'*' * len(kaggle_key)}")

    except ImportError:
        print("✓ Not running in Colab - using default kaggle.json authentication")
    except Exception as e:
        print(f"✗ Error setting up Kaggle credentials: {e}")
        print("\nPlease add these secrets in Colab:")
        print("  1. Click the key icon (🔑) in the left sidebar")
        print("  2. Add secret: KAGGLE_USERNAME")
        print("  3. Add secret: KAGGLE_KEY")
        print("\nGet your credentials from: https://www.kaggle.com/settings/account")
        raise

    try:
        import kaggle
    except ImportError:
        print("Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
        import kaggle

    try:
        print(f"\nDownloading {DATASET_NAME}...")
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET_NAME, "-p", DATA_DIR
        ], check=True)

        # Extract zip files
        for zip_file in Path(DATA_DIR).glob("*.zip"):
            print(f"Extracting {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            zip_file.unlink()

        print("✓ Dataset downloaded and extracted successfully!")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        print(f"\nPlease download manually from:")
        print(f"https://www.kaggle.com/datasets/{DATASET_NAME}")
        return False


def convert_csv_to_binary():
    """
    Convert CSV files to a consolidated HDF5 dataset while preserving all features.
    Adds source metadata and derived temporal features for downstream processing.
    This function only creates the HDF5 file and returns its path.
    """
    preprocessed_file = HDF5_DATASET

    if preprocessed_file.exists() and USE_CACHED_DATA:
        print("\n" + "=" * 80)
        print("✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING")
        print("=" * 80)
        print(f"Using cached file: {preprocessed_file}")
        print(f"Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")
        return preprocessed_file

    print("\n" + "=" * 80)
    print("CONVERTING CSV TO EFFICIENT BINARY FORMAT")
    print("=" * 80)

    # Find all CSV files
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV files found!")

    print(f"\n✓ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

    # Load and combine all CSVs with sampling
    if SAMPLE_SIZE:
        print(f"\nLoading sample data (max {SAMPLE_SIZE:,} rows per file)...")
    else:
        print(f"\nLoading FULL dataset (this may take a while)...")

    dfs = []
    manifest = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, nrows=SAMPLE_SIZE, low_memory=False)
            original_rows = len(df)

            # Normalize string columns to avoid mixed dtype issues
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            for col in object_cols:
                df[col] = df[col].astype(str).str.strip().fillna('__NA__')

            # Attach source metadata
            df['source_file'] = csv_file.name
            df['source_path'] = str(csv_file.relative_to(DATA_DIR))
            df['source_category'] = csv_file.parent.name

            # Derive attack type from filename
            filename = csv_file.name
            if '_attack.csv' in filename:
                attack_type = filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                attack_type = 'Mixed'  # These contain multiple attack types
            else:
                attack_type = 'Normal'  # Sensor data files
            df['Attack_type'] = attack_type

            # Build temporal features without dropping original column
            if 'frame.time' in df.columns:
                time_str = df['frame.time'].astype(str).str.strip()
                parsed_time = pd.to_datetime(time_str, format='%Y %H:%M:%S.%f', errors='coerce')
                if parsed_time.isna().all():
                    parsed_time = pd.to_datetime(time_str, errors='coerce')
                parsed_time = parsed_time.ffill().bfill()
                df['frame_time_datetime'] = parsed_time
                base_time = parsed_time.iloc[0]
                rel_seconds = (parsed_time - base_time).dt.total_seconds()
                df['frame_time_relative_sec'] = rel_seconds.astype('float64')

            dfs.append(df)
            duration = float(df.get('frame_time_relative_sec', pd.Series([0])).max()) if len(df) else 0.0
            manifest.append({
                'file': str(csv_file.relative_to(DATA_DIR)),
                'rows_loaded': int(original_rows),
                'duration_seconds': duration
            })
            print(f"  ✓ {csv_file.name}: {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading {csv_file.name}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*80}")
    print(f"Combined dataset: {len(df):,} rows × {len(df.columns)} columns")
    print(f"{'='*80}")

    print("\nPreparing data for HDF5 storage...")
    df_filtered = df.copy()
    print(f"Final dataset shape: {df_filtered.shape}")
    print(f"Memory usage: {df_filtered.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    preprocessed_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle HDF5 file locking issues (common in Colab)
    if preprocessed_file.exists():
        print(f"  → HDF5 file exists, removing to recreate...")
        try:
            preprocessed_file.unlink()  # Remove the existing file
            print("  ✓ Removed existing HDF5 file")
        except Exception as e:
            print(f"  ✗ Could not remove existing file: {e}")
            print("  → Please manually delete the file and try again")
            raise RuntimeError(f"Cannot remove existing HDF5 file {preprocessed_file}. Please delete it manually.")

    # Save to HDF5
    df_filtered.to_hdf(preprocessed_file, key='data', mode='w', index=False)
    print(f"✓ Saved: {preprocessed_file}")
    print(f"  Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")

    total_csv_size = sum(f.stat().st_size for f in csv_files)
    compression_ratio = (1 - preprocessed_file.stat().st_size / total_csv_size) * 100
    print(f"  Compression: {compression_ratio:.1f}% savings over CSV")
    print(f"  Original CSV size: {total_csv_size / 1024**2:.1f} MB")

    manifest_path = Path(PREPROCESSED_DIR) / 'ingest_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest saved: {manifest_path}")

    return preprocessed_file


def prepare_time_series_data():
    """
    Load preprocessed data, split, and create test sequences.
    Returns flat train data and sequenced test data.
    """
    print("\n" + "=" * 80)
    print("PHASE 2: TIME-SERIES DATA PREPARATION")
    print("=" * 80)

    # Load the HDF5 file
    preprocessed_file = convert_csv_to_binary()
    print(f"\nLoading preprocessed data from {preprocessed_file}...")
    df = pd.read_hdf(preprocessed_file, key='data')

    print(f"✓ Loaded {len(df):,} rows × {len(df.columns)} columns")

    # Identify label column - PREFER Attack_type (multi-class) over Attack_label (binary)
    if USE_MULTICLASS and 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using MULTI-CLASS labels: '{label_col}'")
    elif 'Attack_label' in df.columns:
        label_col = 'Attack_label'
        print(f"✓ Using BINARY labels: '{label_col}'")
    elif 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using labels: '{label_col}'")
    else:
        raise ValueError("No label column found in dataset")

    print(f"  Label column: '{label_col}'")

    # Prepare features - exclude label and metadata columns
    metadata_cols = ['source_file', 'source_path', 'source_category', 'frame.time', 'frame_time_datetime',
                     'Attack_label', 'Attack_type']  # Exclude both potential label columns
    feature_cols = [col for col in df.columns if col != label_col and col not in metadata_cols]

    # Separate numeric and categorical
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df[feature_cols].select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    categorical_cols = [col for col in feature_cols if col not in numeric_cols and col not in datetime_cols]

    print(f"\n✓ Feature analysis:")
    print(f"  • Total features: {len(feature_cols)}")
    print(f"  • Numeric: {len(numeric_cols)}")
    print(f"  • Categorical: {len(categorical_cols)}")
    print(f"  • Datetime: {len(datetime_cols)}")

    X = df[feature_cols].copy()
    y = df[label_col].copy()

    # Convert datetime to numeric
    if datetime_cols:
        for col in datetime_cols:
            dt_series = pd.to_datetime(X[col], errors='coerce')
            dt_numeric = dt_series.view('int64').astype('float64')
            dt_numeric[dt_numeric == np.iinfo(np.int64).min] = np.nan
            X[col] = dt_numeric / 1e9

    # Encode categorical
    feature_encoder = None
    if categorical_cols:
        print("  • Encoding categorical columns...")
        X[categorical_cols] = X[categorical_cols].fillna('__MISSING__').astype(str)
        feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categorical_cols] = feature_encoder.fit_transform(X[categorical_cols])

    # Handle missing/infinite values
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_count = X.isna().sum().sum()
    if nan_count > 0:
        print(f"  • Filling {nan_count:,} NaN values...")
        X.fillna(X.median(), inplace=True)
        X.fillna(0, inplace=True)

    # Encode labels
    print("\n✓ Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(le.classes_)

    print(f"  • Number of classes: {num_classes}")
    for idx, class_name in enumerate(le.classes_):
        count = np.sum(y_encoded == idx)
        print(f"    {idx:3d}: {str(class_name):30s} ({count:,} samples)")

    # ===================================================================
    # CRITICAL: Split data BEFORE scaling to prevent data leakage
    # Scaler must only learn statistics from training data
    # ===================================================================
    print(f"\n✓ Train/test split ({VALIDATION_SPLIT*100:.0f}% test)...")
    X_train_flat, X_test_flat, y_train_flat, y_test_flat = train_test_split(
        X.values, y_encoded,  # Split UNSCALED data
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    )

    print(f"  • Training samples: {X_train_flat.shape[0]:,}")
    print(f"  • Test samples: {X_test_flat.shape[0]:,}")

    # Scale features: fit on training data ONLY, transform both
    print("\n✓ Scaling features with RobustScaler...")
    scaler = RobustScaler()
    X_train_flat = scaler.fit_transform(X_train_flat)  # Fit and transform training data
    X_test_flat = scaler.transform(X_test_flat)        # Transform test data using training statistics

    print(f"  • Training data range: [{X_train_flat.min():.3f}, {X_train_flat.max():.3f}]")
    print(f"  • Test data range: [{X_test_flat.min():.3f}, {X_test_flat.max():.3f}]")
    print(f"  • Scaler fitted on training data only (no data leakage)")

    # Create test sequences
    print(f"\n✓ Creating test sequences...")
    X_test, y_test = create_sequences(X_test_flat, y_test_flat, time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE)
    print(f"  • Test sequences: {X_test.shape}")

    print(f"{'='*80}")

    return X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes


def create_sequences(X, y, time_steps=128, stride=64):
    """
    Create sliding window sequences from time-series data.

    IMPORTANT: This creates sequences from data that has already been shuffled/resampled.
    For proper time-series modeling, data should maintain temporal order, but since
    we're comparing class imbalance techniques (SMOTE, undersampling, etc.), we create
    sequences from the preprocessed data regardless of temporal continuity.

    Args:
        X: Input features array (2D: samples x features)
        y: Target labels array (1D: samples)
        time_steps: Number of time steps per sequence (window size)
        stride: Stride (step size) for sliding window

    Returns:
        Tuple of (sequences, labels) where:
        - sequences: 3D array (num_sequences, time_steps, features)
        - labels: 1D array (num_sequences,) - label from last timestep of each sequence
    """
    sequences = []
    labels = []

    # Create overlapping sequences using sliding window
    for start in range(0, len(X) - time_steps + 1, stride):
        end = start + time_steps
        seq = X[start:end]
        label = y[end - 1]  # Use the label of the last time step in the sequence

        sequences.append(seq)
        labels.append(label)

    if len(sequences) > 0:
        print(f"✓ Created {len(sequences):,} sequences (shape: {sequences[0].shape})")
    else:
        print(f"⚠️  Warning: No sequences created. Data length ({len(X)}) < time_steps ({time_steps})")

    return np.array(sequences), np.array(labels)


def create_sliding_window_dataset(X, y, window_size=128, stride=64, batch_size=32):
    """
    Create a TensorFlow dataset with sliding window sequences.

    Args:
        X: Input features array
        y: Target labels array
        window_size: Size of the sliding window (number of time steps)
        stride: Stride (step size) for sliding window
        batch_size: Batch size for the TensorFlow dataset

    Returns:
        tf.data.Dataset object ready for training
    """
    # --- SLIDING WINDOW DATASET CREATION ---
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    # Create sliding windows
    dataset = dataset.window(
        size=window_size,
        shift=stride,
        drop_remainder=True
    )

    # Flatten the windows into samples
    dataset = dataset.flat_map(lambda window: window.batch(window_size))

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    print(f"✓ Created sliding window dataset with {window_size} time steps per window")

    return dataset


def build_deepfed_model(input_shape, num_classes):
    """
    Phase 3: Build DeepFed time-series model (GRU + CNN + MLP)
    Based on the paper architecture
    """
    print("\n" + "=" * 80)
    print("PHASE 3: BUILDING DEEPFED TIME-SERIES MODEL")
    print("=" * 80)

    seq_length, num_features = input_shape

    print(f"\nModel Configuration:")
    print(f"  • Input shape: ({seq_length}, {num_features})")
    print(f"  • Output classes: {num_classes}")
    print(f"  • Architecture: GRU + Conv1D + MLP (Parallel branches)")

    # Input layer
    input_layer = layers.Input(shape=(seq_length, num_features), name='input')

    # ============================================================
    # BRANCH 1: GRU Module (Sequential Pattern Detection)
    # ============================================================
    print("\n[Branch 1: GRU Module]")

    # GRU works directly on (batch, seq_len, features) format
    x_gru = input_layer

    # First GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, name='gru_1')(x_gru)

    # Second GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.2, name='gru_2')(x_gru)

    print(f"  • GRU Layer 1: 128 units (return sequences, dropout=0.2)")
    print(f"  • GRU Layer 2: 128 units (return last, dropout=0.2)")
    print(f"  • Output shape: (batch, 128)")

    # ============================================================
    # BRANCH 2: CNN Module (Spatial Feature Extraction)
    # ============================================================
    print("\n[Branch 2: CNN Module]")

    # Permute for Conv1D: (batch, seq_len, features) -> (batch, features, seq_len)
    x_cnn = layers.Permute((2, 1), name='cnn_permute')(input_layer)

    # Conv Block 1
    x_cnn = layers.Conv1D(32, kernel_size=3, padding='same', name='conv_1')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_1')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_1')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_1')(x_cnn)

    # Conv Block 2
    x_cnn = layers.Conv1D(64, kernel_size=3, padding='same', name='conv_2')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_2')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_2')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_2')(x_cnn)

    # Conv Block 3
    x_cnn = layers.Conv1D(128, kernel_size=3, padding='same', name='conv_3')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_3')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_3')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_3')(x_cnn)

    # Flatten CNN output
    x_cnn = layers.Flatten(name='cnn_flatten')(x_cnn)

    print(f"  • Conv Block 1: 32 filters")
    print(f"  • Conv Block 2: 64 filters")
    print(f"  • Conv Block 3: 128 filters")
    print(f"  • Each with BatchNorm, ReLU, MaxPool")

    # ============================================================
    # CONCATENATE: Merge GRU and CNN features
    # ============================================================
    print("\n[Feature Fusion]")
    concatenated = layers.Concatenate(name='concatenate')([x_cnn, x_gru])
    print(f"  • Concatenating GRU and CNN features")

    # ============================================================
    # MLP Module (Classification Head)
    # ============================================================
    print("\n[Classification Head: MLP]")

    x = layers.Dense(128, activation='relu', name='fc_1')(concatenated)
    x = layers.Dropout(0.3, name='dropout_1')(x)  # Add dropout after first dense
    x = layers.Dense(64, activation='relu', name='fc_2')(x)
    x = layers.Dropout(0.5, name='dropout_2')(x)  # Keep dropout after second dense

    # Output layer
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    print(f"  • Dense Layer 1: 128 units + Dropout(0.3)")
    print(f"  • Dense Layer 2: 64 units + Dropout(0.5)")
    print(f"  • Output Layer: {num_classes} units (softmax)")

    # Create model
    model = models.Model(inputs=input_layer, outputs=output, name='DeepFed')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*80}")
    print("MODEL SUMMARY")
    print(f"{'='*80}")
    model.summary()

    total_params = model.count_params()
    print(f"\nTotal parameters: {total_params:,}")
    print(f"{'='*80}")

    return model


class LazyDataGenerator(keras.utils.PyDataset):
    """
    Memory-efficient data generator that loads data in batches from disk
    Similar to tf.data.Dataset for lazy loading
    """
    def __init__(self, data_file, indices, batch_size=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data_file = data_file
        self.indices = indices
        self.batch_size = batch_size if batch_size is not None else BATCH_SIZE
        self.shuffle = shuffle
        self.num_samples = len(indices)
        self.num_batches = int(np.ceil(self.num_samples / self.batch_size))

        # Memory-map the files for efficient access
        self.X = np.load(self.data_file['X'], mmap_mode='r')
        self.y = np.load(self.data_file['y'], mmap_mode='r')

    def __len__(self):
        """Number of batches per epoch"""
        return self.num_batches

    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min(start_idx + self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]

        # Load batch from memory-mapped files
        batch_X = self.X[batch_indices].copy()  # Copy to ensure contiguous array
        batch_y = self.y[batch_indices].copy()

        return batch_X, batch_y

    def on_epoch_end(self):
        """Shuffle indices at end of epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)


def create_data_generator(X, y, batch_size=None, shuffle=True):
    """
    Create a data generator to avoid loading entire dataset in memory
    Legacy function for backward compatibility
    """
    if batch_size is None:
        batch_size = BATCH_SIZE

    num_samples = len(X)
    indices = np.arange(num_samples)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_x = X[batch_indices]
            batch_y = y[batch_indices]

            yield batch_x, batch_y


def train_model(model, X_train, y_train, X_test, y_test):
    """
    Phase 4: Train the DeepFed model
    """
    print("\n" + "=" * 80)
    print("PHASE 4: TRAINING DEEPFED MODEL")
    print("=" * 80)

    print(f"\nTraining Configuration:")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")
    print(f"  • Training samples: {len(X_train):,}")
    print(f"  • Validation samples: {len(X_test):,}")
    print(f"  • Steps per epoch: {len(X_train) // BATCH_SIZE}")

    # Callbacks for training
    model_callbacks = [
        callbacks.ModelCheckpoint(
            filepath=str(Path(MODEL_DIR) / 'deepfed_best.keras'),
            monitor='val_accuracy',  # Monitor validation accuracy for best model
            save_best_only=True,
            save_weights_only=False,  # Save full model (architecture + weights + optimizer state)
            mode='max',  # Maximize validation accuracy
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,  # Increased patience for better convergence
            restore_best_weights=True,
            mode='min',  # Minimize validation loss
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=7,  # Increased to allow more exploration before LR reduction
            min_lr=1e-7,
            mode='min',
            verbose=1
        ),
        callbacks.CSVLogger(
            filename=str(Path(MODEL_DIR) / 'training_log.csv'),
            separator=',',
            append=False
        )
    ]

    print(f"\n{'='*80}")
    print("STARTING TRAINING")
    print(f"{'='*80}\n")

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=model_callbacks,
        verbose=1
    )

    # Plot training history
    print(f"\n{'='*80}")
    print("TRAINING COMPLETED")
    print(f"{'='*80}")

    # Visualize training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Training')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'training_history.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved training history: {Path(VIS_DIR) / 'training_history.png'}")
    plt.close()

    return history


def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Phase 5: Evaluate the trained model
    """
    print("\n" + "=" * 80)
    print("PHASE 5: MODEL EVALUATION")
    print("=" * 80)

    print("\nGenerating predictions...")
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{'='*80}")
    print("OVERALL METRICS")
    print(f"{'='*80}")
    print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nNOTE: Accuracy can be misleading in this highly imbalanced dataset.")
    print("      Focus on metrics like F1-score, precision, and recall,")
    print("      especially for minority classes, as shown in the Classification Report.")
    print("-" * 80)
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"{'='*80}")

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 80)
    # Convert label_encoder.classes_ (which are numpy.int8) to strings
    target_names_str = [str(cls) for cls in label_encoder.classes_]
    print(classification_report(
        y_test, y_pred,
        target_names=target_names_str,
        digits=4
    ))

    # Per-class metrics
    print("\nPER-CLASS ACCURACY")
    print("-" * 80)
    for i, class_name in enumerate(label_encoder.classes_):
        mask = y_test == i
        if mask.sum() > 0:
            class_acc = accuracy_score(y_test[mask], y_pred[mask])
            print(f"{str(class_name):30s}: {class_acc:.4f} ({class_acc*100:5.2f}%) [{mask.sum():>6,} samples]")

    # Confusion matrix
    print("\nCONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(y_test, y_pred)

    # Visualize confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {Path(VIS_DIR) / 'confusion_matrix.png'}")
    plt.close()

    # Save confusion matrix
    np.save(Path(MODEL_DIR) / 'confusion_matrix.npy', cm)

    return accuracy, f1_macro, f1_weighted


def save_model_artifacts(model, label_encoder, scaler, metadata):
    """
    Save all model artifacts
    """
    print("\n" + "=" * 80)
    print("SAVING MODEL ARTIFACTS")
    print("=" * 80)

    # Ensure model directory exists
    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

    # Save full model
    model_path = Path(MODEL_DIR) / 'deepfed_final.keras'
    model.save(model_path)
    print(f"✓ Model saved: {model_path}")

    # Save label encoder
    le_path = Path(MODEL_DIR) / 'label_encoder.pkl'
    with open(le_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved: {le_path}")

    # Save scaler
    scaler_path = Path(MODEL_DIR) / 'scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved: {scaler_path}")

    # Save metadata
    meta_path = Path(MODEL_DIR) / 'model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved: {meta_path}")

    print(f"\n{'='*80}")
    print("ALL ARTIFACTS SAVED")
    print(f"{'='*80}")


# ============================================================================
# COMPARISON FUNCTIONS FOR IMBALANCED DATASET HANDLING
# ============================================================================

def train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 1: Baseline - No imbalance handling techniques applied.

    This approach trains the DeepFed model on the original imbalanced data
    without any resampling or class weighting. Serves as the baseline for
    comparison with other imbalance handling techniques.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes

    Returns:
        Tuple of (model, training_history, predictions)
    """
    print("\n" + "="*80)
    print("APPROACH 1: BASELINE (No Imbalance Handling)")
    print("="*80)

    print(f"\nFlat training data shape: {X_train_flat.shape}")
    print("Class distribution in flat data:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Create sliding window sequences from flat training data
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 2: Class Weights - Penalize misclassification of minority classes.

    This approach uses scikit-learn's compute_class_weight with 'balanced' strategy
    to assign higher weights to minority classes during training. The loss function
    penalizes misclassification of rare classes more heavily, encouraging the model
    to pay more attention to them.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes

    Returns:
        Tuple of (model, training_history, predictions)
    """
    print("\n" + "="*80)
    print("APPROACH 2: CLASS WEIGHTS")
    print("="*80)

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from original flat data (no resampling)
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    # Compute class weights from sequence-level labels
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_sequences),
        y=y_train_sequences
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print("\nComputed class weights (based on sequence labels):")
    for cls, weight in class_weight_dict.items():
        print(f"  Class {cls}: {weight:.4f}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 3: SMOTE Oversampling - Generate synthetic samples for minority classes.

    SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic samples
    by interpolating between existing minority class samples. This balances the
    dataset by increasing representation of rare classes. SMOTE is applied to
    flat data first, then sequences are created from the resampled data.

    Note: Synthetic samples may not preserve true temporal patterns, but this
    approach is valuable for comparing imbalance handling techniques.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if SMOTE unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 3: SMOTE OVERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before SMOTE:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE on flat data
    print("\nApplying SMOTE on flat data...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, counts.min() - 1) if counts.min() > 1 else 1)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter SMOTE: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after SMOTE:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 4: Random Undersampling - Reduce majority class samples to balance dataset.

    RandomUnderSampler randomly removes samples from majority classes until all
    classes have equal representation. This balances the dataset but reduces total
    training data, which may hurt performance if minority classes have very few samples.

    Undersampling is applied to flat data first, then sequences are created from
    the resampled data.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if undersampling unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 4: RANDOM UNDERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before undersampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply random undersampling on flat data
    print("\nApplying random undersampling on flat data...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter undersampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after undersampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 5: Combined SMOTE + Random Undersampling - Hybrid resampling strategy.

    This approach combines the benefits of both oversampling and undersampling:
    1. First, SMOTE generates synthetic samples for minority classes
    2. Then, RandomUnderSampler reduces majority class samples

    This hybrid approach often outperforms using either technique alone, as it:
    - Increases minority class representation (via SMOTE)
    - Reduces majority class dominance (via undersampling)
    - Results in a more balanced dataset without excessive synthetic data

    Note: We use RandomUnderSampler instead of Tomek links because Tomek links
    are 10-100x slower with minimal accuracy improvement (<1%).

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if resampling unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 5: COMBINED (SMOTE + Random UnderSampling)")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before resampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE first to oversample minority classes
    print("\n[Step 1/2] Applying SMOTE to oversample minority classes...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"After SMOTE: {X_smote.shape[0]:,} samples")
    unique, counts = np.unique(y_smote, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_smote)*100:5.2f}%)")

    # Apply undersampling to reduce majority class (faster than Tomek, better balance)
    print("\n[Step 2/2] Applying random undersampling for final balance...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy='auto')
    X_train_resampled, y_train_resampled = rus.fit_resample(X_smote, y_smote)

    print(f"\nAfter SMOTE+UnderSampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after combined resampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def evaluate_and_compare(predictions_list, histories, X_test, y_test, label_encoder, approach_names):
    """
    Evaluate all approaches and create comparison visualizations
    """
    print("\n" + "="*80)
    print("COMPARISON EVALUATION")
    print("="*80)

    results = []

    for i, (y_pred_prob, history, approach) in enumerate(zip(predictions_list, histories, approach_names)):
        if y_pred_prob is None:
            print(f"\nSkipping {approach} (not available)")
            continue

        print(f"\n{'-'*80}")
        print(f"Evaluating: {approach}")
        print(f"{'-'*80}")

        # Predictions (already generated during training)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        # Aggregate metrics
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)

        print(f"Accuracy:          {accuracy:.4f}")
        print(f"Macro F1:          {macro_f1:.4f}")
        print(f"Weighted F1:       {weighted_f1:.4f}")
        print(f"Macro Precision:   {macro_precision:.4f}")
        print(f"Macro Recall:      {macro_recall:.4f}")

        results.append({
            'approach': approach,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'per_class_f1': f1.tolist(),
            'per_class_precision': precision.tolist(),
            'per_class_recall': recall.tolist(),
            'per_class_support': support.tolist(),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })

    # Create comparison visualizations
    if len(results) > 0:
        create_comparison_plots(results, label_encoder)
        save_comparison_results(results)

    return results


def cleanup_model_memory(model, approach_name, save_model=True):
    """
    Clean up GPU/CPU memory after training to free resources

    For proper GPU memory management in Keras:
    1. Save the model immediately after training
    2. Delete the model object to free GPU memory
    3. Run garbage collection

    Args:
        model: Trained Keras model
        approach_name: Name of the training approach (for logging)
        save_model: Whether to save the full model (default True)
    """
    if model is None:
        return

    try:
        # Save the full model immediately to preserve it
        if save_model:
            model_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_model.keras"
            model_path.parent.mkdir(parents=True, exist_ok=True)
            model.save(str(model_path))
            print(f"✓ Model saved: {model_path}")

        # Clear Keras backend session
        try:
            import keras.backend as K
            K.clear_session()
        except ImportError:
            try:
                import tensorflow as tf
                tf.keras.backend.clear_session()
            except:
                print(f"⚠️  Could not clear Keras session for {approach_name}")

        # Delete the model object to free GPU memory
        del model

        # Force garbage collection
        gc.collect()

        print(f"✓ GPU/CPU memory freed for: {approach_name}")

    except Exception as e:
        print(f"⚠️  Warning during cleanup for {approach_name}: {e}")


def load_saved_model(approach_name):
    """
    Load a previously saved model for evaluation

    Args:
        approach_name: Name of the training approach

    Returns:
        Loaded Keras model or None if not found
    """
    try:
        model_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_model.keras"
        if model_path.exists():
            model = keras.models.load_model(str(model_path))
            print(f"✓ Model loaded: {model_path}")
            return model
        else:
            print(f"⚠️  Model not found: {model_path}")
            return None
    except Exception as e:
        print(f"⚠️  Error loading model {approach_name}: {e}")
        return None


def create_comparison_plots(results, label_encoder):
    """
    Create comprehensive comparison visualizations
    """
    print("\n" + "="*80)
    print("CREATING COMPARISON VISUALIZATIONS")
    print("="*80)

    approaches = [r['approach'] for r in results]

    # 1. Overall Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision']
    metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score', 'Macro Precision']

    for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
        ax = axes[idx // 2, idx % 2]
        values = [r[metric] for r in results]
        bars = ax.bar(approaches, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
        ax.set_ylabel(name, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_title(f'{name} Comparison', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_overall_metrics.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_overall_metrics.png'}")
    plt.close()

    # 2. Per-Class F1-Score Comparison
    num_classes = len(results[0]['per_class_f1'])
    x = np.arange(num_classes)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 8))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, result in enumerate(results):
        offset = width * (i - len(results)/2 + 0.5)
        ax.bar(x + offset, result['per_class_f1'], width,
               label=result['approach'], color=colors[i % len(colors)])

    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title('Per-Class F1-Score Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_encoder.classes_], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_per_class_f1.png'}")
    plt.close()

    # 3. Macro Recall Comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    values = [r['macro_recall'] for r in results]
    bars = ax.bar(approaches, values,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
    ax.set_ylabel('Macro Recall', fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title('Macro Recall Comparison (Minority Class Performance)',
                fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_macro_recall.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_macro_recall.png'}")
    plt.close()


def save_comparison_results(results):
    """
    Save comparison results to JSON and generate text summary
    """
    # Save JSON
    json_path = Path(MODEL_DIR) / 'comparison_results.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved: {json_path}")

    # Create text summary
    summary_path = Path(MODEL_DIR) / 'comparison_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("IMBALANCED DATASET HANDLING COMPARISON SUMMARY\n")
        f.write("="*80 + "\n\n")

        for result in results:
            f.write(f"\n{result['approach']}\n")
            f.write("-"*80 + "\n")
            f.write(f"  Accuracy:          {result['accuracy']:.4f}\n")
            f.write(f"  Macro F1-Score:    {result['macro_f1']:.4f}\n")
            f.write(f"  Weighted F1-Score: {result['weighted_f1']:.4f}\n")
            f.write(f"  Macro Precision:   {result['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall:      {result['macro_recall']:.4f}\n")

        # Best approach per metric
        f.write("\n" + "="*80 + "\n")
        f.write("BEST APPROACHES PER METRIC\n")
        f.write("="*80 + "\n")

        metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']
        metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score',
                       'Macro Precision', 'Macro Recall']

        for metric, name in zip(metrics, metric_names):
            best = max(results, key=lambda x: x[metric])
            f.write(f"\n{name:20s}: {best['approach']:30s} ({best[metric]:.4f})\n")

    print(f"✓ Saved: {summary_path}")


def main():
    """
    Main execution pipeline with efficient data loading and caching
    """
    print("\n" + "=" * 80)
    print("DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM")
    print("Dataset: Edge-IIoTset")
    print("Model: GRU + CNN (Time-Series Architecture)")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  • Classification: {'Multi-class (attack types)' if USE_MULTICLASS else 'Binary (normal vs attack)'}")
    print(f"  • Dataset: {'Full dataset' if SAMPLE_SIZE is None else f'Sampled ({SAMPLE_SIZE:,} rows/file)'}")
    print(f"  • Use cached data: {USE_CACHED_DATA}")
    print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Window stride: {WINDOW_STRIDE} (for sliding window)")
    print(f"  • Batch size: {BATCH_SIZE} (fixed)")
    print(f"  • Epochs: {EPOCHS} (with early stopping)")
    print(f"  • Learning rate: {LEARNING_RATE}")

    # Check what's already cached
    cached_sequences = all(Path(CACHE_DIR, f).exists() for f in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy'])
    cached_hdf5 = HDF5_DATASET.exists()

    print(f"\nCache Status:")
    print(f"  • HDF5 preprocessed: {'✓ Found' if cached_hdf5 else '✗ Not found'}")
    print(f"  • Sequence arrays: {'✓ Found' if cached_sequences else '✗ Not found'}")
    if cached_sequences and USE_CACHED_DATA:
        print(f"  → Will skip CSV parsing and sequence creation!")

    try:
        # Check if dataset exists
        csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
        if not csv_exists:
            print("\nDataset not found. Downloading...")
            if not download_dataset():
                print("\n✗ Download failed. Please download manually:")
                print(f"   https://www.kaggle.com/datasets/{DATASET_NAME}")
                return 1

        # Phase 1 & 2: Load and preprocess dataset
        X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes = prepare_time_series_data()

        print(f"\n✅ Data loaded successfully:")
        print(f"  • Training samples (flat): {X_train_flat.shape[0]:,}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Features: {X_train_flat.shape[1]}")
        print(f"  • Classes: {num_classes}")
        print(f"  • Class names: {le.classes_}")
        print(f"  • Batch size: {BATCH_SIZE}")

        # Phase 3: Train and evaluate all 5 comparison approaches
        print("\n" + "="*80)
        print("PHASE 3-5: COMPARISON STUDY")
        print("Training 5 Different Approaches for Handling Imbalanced Data")
        print("="*80)

        approach_names = [
            "Baseline (No Handling)",
            "Class Weights",
            "SMOTE Oversampling",
            "Random Undersampling",
            "Combined (SMOTE+UnderSample)"
        ]

        # All approaches will work with the same flat training data
        # Each approach will apply its resampling strategy, then create sequences
        print(f"\n📊 Training data prepared:")
        print(f"  • Flat training samples: {X_train_flat.shape[0]:,}")
        print(f"  • Features per sample: {X_train_flat.shape[1]}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Sequence shape: ({X_test.shape[1]}, {X_test.shape[2]})")

        models = []
        histories = []
        predictions_list = []

        # Approach 1: Baseline
        print("\n" + "█"*80)
        print("█ APPROACH 1/5: BASELINE")
        print("█"*80)
        model1, hist1, pred1 = train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes)
        models.append("Baseline")  # Store approach name instead of model object
        histories.append(hist1)
        predictions_list.append(pred1)
        if model1 is not None:
            cleanup_model_memory(model1, "Baseline")  # Saves and deletes model

        # Approach 2: Class Weights
        print("\n" + "█"*80)
        print("█ APPROACH 2/5: CLASS WEIGHTS")
        print("█"*80)
        model2, hist2, pred2 = train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes)
        models.append("Class_Weights")  # Store approach name
        histories.append(hist2)
        predictions_list.append(pred2)
        if model2 is not None:
            cleanup_model_memory(model2, "Class_Weights")  # Saves and deletes model

        # Approach 3: SMOTE Oversampling
        print("\n" + "█"*80)
        print("█ APPROACH 3/5: SMOTE OVERSAMPLING")
        print("█"*80)
        model3, hist3, pred3 = train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append("SMOTE_Oversampling")  # Store approach name
        histories.append(hist3)
        predictions_list.append(pred3)
        if model3 is not None:
            cleanup_model_memory(model3, "SMOTE_Oversampling")  # Saves and deletes model

        # Approach 4: Random Undersampling
        print("\n" + "█"*80)
        print("█ APPROACH 4/5: RANDOM UNDERSAMPLING")
        print("█"*80)
        model4, hist4, pred4 = train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append("Random_Undersampling")  # Store approach name
        histories.append(hist4)
        predictions_list.append(pred4)
        if model4 is not None:
            cleanup_model_memory(model4, "Random_Undersampling")  # Saves and deletes model

        # Approach 5: Combined SMOTE + UnderSampling
        print("\n" + "█"*80)
        print("█ APPROACH 5/5: COMBINED (SMOTE + UNDERSAMPLING)")
        print("█"*80)
        model5, hist5, pred5 = train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
        models.append("Combined_SMOTE_Tomek")  # Store approach name
        histories.append(hist5)
        predictions_list.append(pred5)
        if model5 is not None:
            cleanup_model_memory(model5, "Combined_SMOTE_Tomek")  # Saves and deletes model

        # Evaluate and compare all approaches
        print("\n" + "="*80)
        print("ALL TRAINING COMPLETED - STARTING COMPARISON EVALUATION")
        print("="*80)

        results = evaluate_and_compare(predictions_list, histories, X_test, y_test, le, approach_names)

        # Save metadata
        metadata = {
            'model_name': 'DeepFed Comparison Study',
            'dataset': 'Edge-IIoTset',
            'architecture': 'GRU + CNN + MLP',
            'sequence_length': SEQUENCE_LENGTH,
            'num_features': X_test.shape[2],  # Use test sequences shape
            'num_classes': num_classes,
            'class_names': le.classes_.tolist(),
            'approaches': approach_names,
            'comparison_results': results,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS
        }

        # Save best model (highest macro F1)
        if results:
            best_idx = max(range(len(results)), key=lambda i: results[i]['macro_f1'])
            best_approach_name = models[best_idx]  # Get approach name
            best_model = load_saved_model(best_approach_name)  # Load the saved model
            best_approach = approach_names[best_idx]

            print(f"\n{'='*80}")
            print(f"BEST APPROACH: {best_approach}")
            print(f"  Macro F1-Score: {results[best_idx]['macro_f1']:.4f}")
            print(f"{'='*80}")

            if best_model is not None:
                save_model_artifacts(best_model, le, scaler, metadata)
                # Clean up the reloaded model too
                cleanup_model_memory(best_model, f"Best_{best_approach_name}", save_model=False)
            else:
                print("⚠️  Could not load best model for final saving")

        print("\n" + "=" * 80)
        print("✓ COMPARISON STUDY COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"\nResults Summary:")
        for i, (name, result) in enumerate(zip(approach_names, results)):
            print(f"\n{i+1}. {name}")
            print(f"   Accuracy:   {result['accuracy']:.4f}")
            print(f"   Macro F1:   {result['macro_f1']:.4f}")
            print(f"   Macro Recall: {result['macro_recall']:.4f}")
        print(f"\n  • Results saved in:     {MODEL_DIR}/")
        print(f"  • Visualizations in:    {VIS_DIR}/")
        print("=" * 80)

        return 0

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    exit_code = main()
    sys.exit(exit_code)

TensorFlow available for GPU detection

DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM
Dataset: Edge-IIoTset
Model: GRU + CNN (Time-Series Architecture)

Configuration:
  • Classification: Multi-class (attack types)
  • Dataset: Sampled (100,000 rows/file)
  • Use cached data: True
  • Sequence length: 128 time steps
  • Window stride: 64 (for sliding window)
  • Batch size: 1024 (fixed)
  • Epochs: 100 (with early stopping)
  • Learning rate: 0.001

Cache Status:
  • HDF5 preprocessed: ✓ Found
  • Sequence arrays: ✗ Not found

PHASE 2: TIME-SERIES DATA PREPARATION

✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING
Using cached file: cache/preprocessed/dataset.h5
Size: 1116.0 MB

Loading preprocessed data from cache/preprocessed/dataset.h5...
✓ Loaded 1,965,333 rows × 68 columns
✓ Using MULTI-CLASS labels: 'Attack_type'
  Label column: 'Attack_type'

✓ Feature analysis:
  • Total features: 61
  • Numeric: 31
  • Categorical: 30
  • Datetime: 0
  • Encoding categorical columns...
  • Fi

Model: "DeepFed"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 61)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 61, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 61, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 61, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 61, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 30, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 30, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 30, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 30, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 15, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 15, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 15, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 15, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 7, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     73,344 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 896)       │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    131,200 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 357,040 (1.36 MB)

 Trainable params: 356,592 (1.36 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 128, 61)   │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_permute         │ (None, 61, 128)   │          0 │ input[0][0]       │
│ (Permute)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_1 (Conv1D)     │ (None, 61, 32)    │     12,320 │ cnn_permute[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_1                │ (None, 61, 32)    │        128 │ conv_1[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_1 (Activation) │ (None, 61, 32)    │          0 │ bn_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_1              │ (None, 30, 32)    │          0 │ relu_1[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_2 (Conv1D)     │ (None, 30, 64)    │      6,208 │ pool_1[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_2                │ (None, 30, 64)    │        256 │ conv_2[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_2 (Activation) │ (None, 30, 64)    │          0 │ bn_2[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_2              │ (None, 15, 64)    │          0 │ relu_2[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv_3 (Conv1D)     │ (None, 15, 128)   │     24,704 │ pool_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bn_3                │ (None, 15, 128)   │        512 │ conv_3[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ relu_3 (Activation) │ (None, 15, 128)   │          0 │ bn_3[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ pool_3              │ (None, 7, 128)    │          0 │ relu_3[0][0]      │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 128, 128)  │     73,344 │ input[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ cnn_flatten         │ (None, 896)       │          0 │ pool_3[0][0]      │
│ (Flatten)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_2 (GRU)         │ (None, 128)       │     99,072 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 1024)      │          0 │ cnn_flatten[0][0… │
│ (Concatenate)       │                   │            │ gru_2[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ fc_1 (Dense)        │ (None, 128)       │    131,200 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ fc_1[0][0]        │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 357,040 (1.36 MB)

 Trainable params: 356,592 (1.36 MB)

 Non-trainable params: 448 (1.75 KB)


Total parameters: 357,040
Epoch 1/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 103s 490ms/step - accuracy: 0.6333 - loss: 1.1972 - val_accuracy: 0.6599 - val_loss: 1.0344
Epoch 2/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 95s 486ms/step - accuracy: 0.9525 - loss: 0.1699 - val_accuracy: 0.8221 - val_loss: 0.5874
Epoch 3/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 96s 491ms/step - accuracy: 0.9683 - loss: 0.1121 - val_accuracy: 0.8562 - val_loss: 0.4375
Epoch 4/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 96s 489ms/step - accuracy: 0.9747 - loss: 0.0894 - val_accuracy: 0.8707 - val_loss: 0.3904
Epoch 5/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 96s 489ms/step - accuracy: 0.9777 - loss: 0.0760 - val_accuracy: 0.8798 - val_loss: 0.3593
Epoch 6/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 95s 484ms/step - accuracy: 0.9806 - loss: 0.0666 - val_accuracy: 0.8935 - val_loss: 0.3199
Epoch 7/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 93s 476ms/step - accuracy: 0.9823 - loss: 0.0607 - val_accuracy: 0.9137 - val_loss: 0.2890
Epoch 8/100
196/196 ━━━━━━━━━━━━━━━━━━━━ 93s 473

In [1]:
"""
DeepFed: Intrusion Detection with Imbalanced Dataset Handling
==============================================================

Research-focused implementation comparing different approaches to handle
highly imbalanced network attack datasets (majority Normal, minority Attacks).

COMPARISON APPROACHES:
1. Baseline: Train on imbalanced data (no handling)
2. Class Weights: Penalize misclassification of minority classes
3. SMOTE Oversampling: Synthetic minority oversampling technique
4. Random Undersampling: Random undersampling of majority class
5. Combined: SMOTE + Random undersampling (hybrid approach)

Dataset: Edge-IIoTset - Cyber Security Dataset of IoT & IIoT
Model: DeepFed (GRU + CNN parallel architecture)

Based on DeepFed paper: "DeepFed: Federated Deep Learning for Intrusion Detection
in Industrial Cyber-Physical Systems"

INSTALLATION:
-------------
For full functionality (all 5 approaches), install imbalanced-learn:
    pip install imbalanced-learn

Without imbalanced-learn, only approaches 1 and 2 will run.

USAGE:
------
python deepfed_train.py

The script will:
- Load and preprocess Edge-IIoTset dataset
- Train 5 models with different imbalance handling approaches
- Evaluate and compare all approaches
- Generate comparison visualizations
- Save results to models/deepfed/ and visualizations/
"""

import os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Use non-interactive backend
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import zipfile
import subprocess
import sys
import pickle
from collections import Counter
import json
import gc  # For garbage collection

# Check for GPU availability and dynamic batch sizing
try:
    import tensorflow as tf
    TF_AVAILABLE = True
    print("TensorFlow available for GPU detection")
except ImportError:
    TF_AVAILABLE = False
    print("TensorFlow not available, using CPU-only batch sizing")

# Use Keras 3 (not tensorflow.keras)
import keras
from keras import layers, models, callbacks, ops
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, RobustScaler, OrdinalEncoder, MinMaxScaler
from sklearn.metrics import (classification_report, confusion_matrix, accuracy_score,
                            f1_score, precision_recall_fscore_support)
from sklearn.utils.class_weight import compute_class_weight

# Imbalanced-learn for handling imbalanced datasets
try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    IMBLEARN_AVAILABLE = True
except ImportError:
    print("WARNING: imbalanced-learn not installed. Install with: pip install imbalanced-learn")
    print("         Only baseline and class weights approaches will be available.")
    IMBLEARN_AVAILABLE = False

import warnings
warnings.filterwarnings('ignore')

# Check for required packages
try:
    import tables  # For HDF5 support
except ImportError:
    print("ERROR: pytables is required for HDF5 export. Please install with `pip install tables`.")
    sys.exit(1)

# Configuration
DATASET_NAME = "mohamedamineferrag/edgeiiotset-cyber-security-dataset-of-iot-iiot"
DATA_DIR = "./data/edge_iiot"
MODEL_DIR = "/content/drive/MyDrive/Projects/deepfed"  # Updated path for Google Drive
VIS_DIR = "./visualizations"
CACHE_DIR = "./cache"
PREPROCESSED_DIR = "./cache/preprocessed"  # For efficient binary format
HDF5_DATASET = Path(PREPROCESSED_DIR) / "dataset.h5"
BATCH_SIZE = 512  # Fixed batch size for consistent training
EPOCHS = 100  # Sufficient epochs for convergence with early stopping
LEARNING_RATE = 0.001
RANDOM_STATE = 42
SEQUENCE_LENGTH = 128  # Time steps for time-series sequences
WINDOW_STRIDE = 64  # Stride for sliding window sequence creation
VALIDATION_SPLIT = 0.2
SAMPLE_SIZE = 100000  # Sample size per file; set to None for full dataset
USE_MULTICLASS = True  # Use multi-class attack type classification
USE_CACHED_DATA = True  # Use preprocessed binary data if available
USE_PRETRAINED = True  # If True, load per-approach saved models from MODEL_DIR and skip training

# Set random seeds
np.random.seed(RANDOM_STATE)
keras.utils.set_random_seed(RANDOM_STATE)

# Create directories (skip if paths are not accessible)
for dir_path in [DATA_DIR, MODEL_DIR, VIS_DIR, CACHE_DIR, PREPROCESSED_DIR]:
    try:
        os.makedirs(dir_path, exist_ok=True)
    except (OSError, PermissionError) as e:
        print(f"Warning: Could not create directory {dir_path}: {e}")
        print("This is normal if running outside the target environment (e.g., Google Colab)")


def download_dataset():
    """Download Edge-IIoTset dataset from Kaggle"""
    print("\n" + "=" * 80)
    print("DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE")
    print("=" * 80)

    # Check if dataset is already downloaded
    csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
    if csv_exists:
        print("✓ Dataset already exists - skipping download")
        print(f"Found CSV files in: {DATA_DIR}")
        return True

    print("Dataset not found locally - proceeding with download...")

    # Setup Kaggle credentials from Colab secrets
    try:
        from google.colab import userdata
        print("✓ Running in Google Colab - using secrets")

        # Get credentials from Colab secrets
        kaggle_username = userdata.get('KAGGLE_USERNAME')
        kaggle_key = userdata.get('KAGGLE_KEY')

        if not kaggle_username or not kaggle_key:
            raise ValueError("KAGGLE_USERNAME and KAGGLE_KEY must be set in Colab secrets")

        # Set environment variables for Kaggle API
        os.environ['KAGGLE_USERNAME'] = kaggle_username
        os.environ['KAGGLE_KEY'] = kaggle_key

        print(f"  • Username: {kaggle_username}")
        print(f"  • API Key: {'*' * len(kaggle_key)}")

    except ImportError:
        print("✓ Not running in Colab - using default kaggle.json authentication")
    except Exception as e:
        print(f"✗ Error setting up Kaggle credentials: {e}")
        print("\nPlease add these secrets in Colab:")
        print("  1. Click the key icon (🔑) in the left sidebar")
        print("  2. Add secret: KAGGLE_USERNAME")
        print("  3. Add secret: KAGGLE_KEY")
        print("\nGet your credentials from: https://www.kaggle.com/settings/account")
        raise

    try:
        import kaggle
    except ImportError:
        print("Installing kaggle package...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "kaggle"])
        import kaggle

    try:
        print(f"\nDownloading {DATASET_NAME}...")
        subprocess.run([
            "kaggle", "datasets", "download", "-d", DATASET_NAME, "-p", DATA_DIR
        ], check=True)

        # Extract zip files
        for zip_file in Path(DATA_DIR).glob("*.zip"):
            print(f"Extracting {zip_file.name}...")
            with zipfile.ZipFile(zip_file, 'r') as zip_ref:
                zip_ref.extractall(DATA_DIR)
            zip_file.unlink()

        print("✓ Dataset downloaded and extracted successfully!")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        print(f"\nPlease download manually from:")
        print(f"https://www.kaggle.com/datasets/{DATASET_NAME}")
        return False


def convert_csv_to_binary():
    """
    Convert CSV files to a consolidated HDF5 dataset while preserving all features.
    Adds source metadata and derived temporal features for downstream processing.
    This function only creates the HDF5 file and returns its path.
    """
    preprocessed_file = HDF5_DATASET

    if preprocessed_file.exists() and USE_CACHED_DATA:
        print("\n" + "=" * 80)
        print("✓ PREPROCESSED DATA FOUND - SKIPPING CSV PARSING")
        print("=" * 80)
        print(f"Using cached file: {preprocessed_file}")
        print(f"Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")
        return preprocessed_file

    print("\n" + "=" * 80)
    print("CONVERTING CSV TO EFFICIENT BINARY FORMAT")
    print("=" * 80)

    # Find all CSV files
    csv_files = list(Path(DATA_DIR).rglob("*.csv"))
    if not csv_files:
        raise FileNotFoundError("No CSV files found!")

    print(f"\n✓ Found {len(csv_files)} CSV file(s):")
    for f in csv_files:
        print(f"  - {f.name} ({f.stat().st_size / 1024 / 1024:.1f} MB)")

    # Load and combine all CSVs with sampling
    if SAMPLE_SIZE:
        print(f"\nLoading sample data (max {SAMPLE_SIZE:,} rows per file)...")
    else:
        print(f"\nLoading FULL dataset (this may take a while)...")

    dfs = []
    manifest = []
    for csv_file in csv_files:
        try:
            df = pd.read_csv(csv_file, nrows=SAMPLE_SIZE, low_memory=False)
            original_rows = len(df)

            # Normalize string columns to avoid mixed dtype issues
            object_cols = df.select_dtypes(include=['object']).columns.tolist()
            for col in object_cols:
                df[col] = df[col].astype(str).str.strip().fillna('__NA__')

            # Attach source metadata
            df['source_file'] = csv_file.name
            df['source_path'] = str(csv_file.relative_to(DATA_DIR))
            df['source_category'] = csv_file.parent.name

            # Derive attack type from filename
            filename = csv_file.name
            if '_attack.csv' in filename:
                attack_type = filename.replace('_attack.csv', '').replace('_', ' ')
            elif filename in ['ML-EdgeIIoT-dataset.csv', 'DNN-EdgeIIoT-dataset.csv']:
                attack_type = 'Mixed'  # These contain multiple attack types
            else:
                attack_type = 'Normal'  # Sensor data files
            df['Attack_type'] = attack_type

            # Build temporal features without dropping original column
            if 'frame.time' in df.columns:
                time_str = df['frame.time'].astype(str).str.strip()
                parsed_time = pd.to_datetime(time_str, format='%Y %H:%M:%S.%f', errors='coerce')
                if parsed_time.isna().all():
                    parsed_time = pd.to_datetime(time_str, errors='coerce')
                parsed_time = parsed_time.ffill().bfill()
                df['frame_time_datetime'] = parsed_time
                base_time = parsed_time.iloc[0]
                rel_seconds = (parsed_time - base_time).dt.total_seconds()
                df['frame_time_relative_sec'] = rel_seconds.astype('float64')

            dfs.append(df)
            duration = float(df.get('frame_time_relative_sec', pd.Series([0])).max()) if len(df) else 0.0
            manifest.append({
                'file': str(csv_file.relative_to(DATA_DIR)),
                'rows_loaded': int(original_rows),
                'duration_seconds': duration
            })
            print(f"  ✓ {csv_file.name}: {len(df):,} rows, {len(df.columns)} columns")
        except Exception as e:
            print(f"  ✗ Error loading {csv_file.name}: {e}")

    df = pd.concat(dfs, ignore_index=True)
    print(f"\n{'='*80}")
    print(f"Combined dataset: {len(df):,} rows × {len(df.columns)} columns")
    print(f"{'='*80}")

    print("\nPreparing data for HDF5 storage...")
    df_filtered = df.copy()
    print(f"Final dataset shape: {df_filtered.shape}")
    print(f"Memory usage: {df_filtered.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

    preprocessed_file.parent.mkdir(parents=True, exist_ok=True)

    # Handle HDF5 file locking issues (common in Colab)
    if preprocessed_file.exists():
        print(f"  → HDF5 file exists, removing to recreate...")
        try:
            preprocessed_file.unlink()  # Remove the existing file
            print("  ✓ Removed existing HDF5 file")
        except Exception as e:
            print(f"  ✗ Could not remove existing file: {e}")
            print("  → Please manually delete the file and try again")
            raise RuntimeError(f"Cannot remove existing HDF5 file {preprocessed_file}. Please delete it manually.")

    # Save to HDF5
    df_filtered.to_hdf(preprocessed_file, key='data', mode='w', index=False)
    print(f"✓ Saved: {preprocessed_file}")
    print(f"  Size: {preprocessed_file.stat().st_size / 1024**2:.1f} MB")

    total_csv_size = sum(f.stat().st_size for f in csv_files)
    compression_ratio = (1 - preprocessed_file.stat().st_size / total_csv_size) * 100
    print(f"  Compression: {compression_ratio:.1f}% savings over CSV")
    print(f"  Original CSV size: {total_csv_size / 1024**2:.1f} MB")

    manifest_path = Path(PREPROCESSED_DIR) / 'ingest_manifest.json'
    with open(manifest_path, 'w') as f:
        json.dump(manifest, f, indent=2)
    print(f"  Manifest saved: {manifest_path}")

    return preprocessed_file


def prepare_time_series_data():
    """
    Load preprocessed data, split, and create test sequences.
    Returns flat train data and sequenced test data.
    """
    print("\n" + "=" * 80)
    print("PHASE 2: TIME-SERIES DATA PREPARATION")
    print("=" * 80)

    # Load the HDF5 file
    preprocessed_file = convert_csv_to_binary()
    print(f"\nLoading preprocessed data from {preprocessed_file}...")
    df = pd.read_hdf(preprocessed_file, key='data')

    print(f"✓ Loaded {len(df):,} rows × {len(df.columns)} columns")

    # Identify label column - PREFER Attack_type (multi-class) over Attack_label (binary)
    if USE_MULTICLASS and 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using MULTI-CLASS labels: '{label_col}'")
    elif 'Attack_label' in df.columns:
        label_col = 'Attack_label'
        print(f"✓ Using BINARY labels: '{label_col}'")
    elif 'Attack_type' in df.columns:
        label_col = 'Attack_type'
        print(f"✓ Using labels: '{label_col}'")
    else:
        raise ValueError("No label column found in dataset")

    print(f"  Label column: '{label_col}'")

    # Prepare features - exclude label and metadata columns
    metadata_cols = ['source_file', 'source_path', 'source_category', 'frame.time', 'frame_time_datetime',
                     'Attack_label', 'Attack_type']  # Exclude both potential label columns
    feature_cols = [col for col in df.columns if col != label_col and col not in metadata_cols]

    # Separate numeric and categorical
    numeric_cols = df[feature_cols].select_dtypes(include=[np.number]).columns.tolist()
    datetime_cols = df[feature_cols].select_dtypes(include=['datetime64[ns]', 'datetime64[ns, UTC]']).columns.tolist()
    categorical_cols = [col for col in feature_cols if col not in numeric_cols and col not in datetime_cols]

    print(f"\n✓ Feature analysis:")
    print(f"  • Total features: {len(feature_cols)}")
    print(f"  • Numeric: {len(numeric_cols)}")
    print(f"  • Categorical: {len(categorical_cols)}")
    print(f"  • Datetime: {len(datetime_cols)}")

    X = df[feature_cols].copy()
    y = df[label_col].copy()

    # Convert datetime to numeric
    if datetime_cols:
        for col in datetime_cols:
            dt_series = pd.to_datetime(X[col], errors='coerce')
            dt_numeric = dt_series.view('int64').astype('float64')
            dt_numeric[dt_numeric == np.iinfo(np.int64).min] = np.nan
            X[col] = dt_numeric / 1e9

    # Encode categorical
    feature_encoder = None
    if categorical_cols:
        print("  • Encoding categorical columns...")
        X[categorical_cols] = X[categorical_cols].fillna('__MISSING__').astype(str)
        feature_encoder = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        X[categorical_cols] = feature_encoder.fit_transform(X[categorical_cols])

    # Handle missing/infinite values
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    nan_count = X.isna().sum().sum()
    if nan_count > 0:
        print(f"  • Filling {nan_count:,} NaN values...")
        X.fillna(X.median(), inplace=True)
        X.fillna(0, inplace=True)

    # Encode labels
    print("\n✓ Encoding labels...")
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    num_classes = len(le.classes_)

    print(f"  • Number of classes: {num_classes}")
    for idx, class_name in enumerate(le.classes_):
        count = np.sum(y_encoded == idx)
        print(f"    {idx:3d}: {str(class_name):30s} ({count:,} samples)")

    # ===================================================================
    # CRITICAL: Split data BEFORE scaling to prevent data leakage
    # Scaler must only learn statistics from training data
    # ===================================================================
    print(f"\n✓ Train/test split ({VALIDATION_SPLIT*100:.0f}% test)...")
    X_train_flat, X_test_flat, y_train_flat, y_test_flat = train_test_split(
        X.values, y_encoded,  # Split UNSCALED data
        test_size=VALIDATION_SPLIT,
        random_state=RANDOM_STATE,
        stratify=y_encoded
    )

    print(f"  • Training samples: {X_train_flat.shape[0]:,}")
    print(f"  • Test samples: {X_test_flat.shape[0]:,}")

    # Scale features: fit on training data ONLY, transform both
    print("\n✓ Scaling features with RobustScaler...")
    scaler = RobustScaler()
    X_train_flat = scaler.fit_transform(X_train_flat)  # Fit and transform training data
    X_test_flat = scaler.transform(X_test_flat)        # Transform test data using training statistics

    print(f"  • Training data range: [{X_train_flat.min():.3f}, {X_train_flat.max():.3f}]")
    print(f"  • Test data range: [{X_test_flat.min():.3f}, {X_test_flat.max():.3f}]")
    print(f"  • Scaler fitted on training data only (no data leakage)")

    # Create test sequences
    print(f"\n✓ Creating test sequences...")
    X_test, y_test = create_sequences(X_test_flat, y_test_flat, time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE)
    print(f"  • Test sequences: {X_test.shape}")

    print(f"{'='*80}")

    return X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes


def create_sequences(X, y, time_steps=128, stride=64):
    """
    Create sliding window sequences from time-series data.

    IMPORTANT: This creates sequences from data that has already been shuffled/resampled.
    For proper time-series modeling, data should maintain temporal order, but since
    we're comparing class imbalance techniques (SMOTE, undersampling, etc.), we create
    sequences from the preprocessed data regardless of temporal continuity.

    Args:
        X: Input features array (2D: samples x features)
        y: Target labels array (1D: samples)
        time_steps: Number of time steps per sequence (window size)
        stride: Stride (step size) for sliding window

    Returns:
        Tuple of (sequences, labels) where:
        - sequences: 3D array (num_sequences, time_steps, features)
        - labels: 1D array (num_sequences,) - label from last timestep of each sequence
    """
    sequences = []
    labels = []

    # Create overlapping sequences using sliding window
    for start in range(0, len(X) - time_steps + 1, stride):
        end = start + time_steps
        seq = X[start:end]
        label = y[end - 1]  # Use the label of the last time step in the sequence

        sequences.append(seq)
        labels.append(label)

    if len(sequences) > 0:
        print(f"✓ Created {len(sequences):,} sequences (shape: {sequences[0].shape})")
    else:
        print(f"⚠️  Warning: No sequences created. Data length ({len(X)}) < time_steps ({time_steps})")

    return np.array(sequences), np.array(labels)


def create_sliding_window_dataset(X, y, window_size=128, stride=64, batch_size=32):
    """
    Create a TensorFlow dataset with sliding window sequences.

    Args:
        X: Input features array
        y: Target labels array
        window_size: Size of the sliding window (number of time steps)
        stride: Stride (step size) for sliding window
        batch_size: Batch size for the TensorFlow dataset

    Returns:
        tf.data.Dataset object ready for training
    """
    # --- SLIDING WINDOW DATASET CREATION ---
    dataset = tf.data.Dataset.from_tensor_slices((X, y))

    # Create sliding windows
    dataset = dataset.window(
        size=window_size,
        shift=stride,
        drop_remainder=True
    )

    # Flatten the windows into samples
    dataset = dataset.flat_map(lambda window: window.batch(window_size))

    # Shuffle, batch, and prefetch
    dataset = dataset.shuffle(buffer_size=10000)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(tf.data.AUTOTUNE)

    print(f"✓ Created sliding window dataset with {window_size} time steps per window")

    return dataset


def build_deepfed_model(input_shape, num_classes):
    """
    Phase 3: Build DeepFed time-series model (GRU + CNN + MLP)
    Based on the paper architecture
    """
    print("\n" + "=" * 80)
    print("PHASE 3: BUILDING DEEPFED TIME-SERIES MODEL")
    print("=" * 80)

    seq_length, num_features = input_shape

    print(f"\nModel Configuration:")
    print(f"  • Input shape: ({seq_length}, {num_features})")
    print(f"  • Output classes: {num_classes}")
    print(f"  • Architecture: GRU + Conv1D + MLP (Parallel branches)")

    # Input layer
    input_layer = layers.Input(shape=(seq_length, num_features), name='input')

    # ============================================================
    # BRANCH 1: GRU Module (Sequential Pattern Detection)
    # ============================================================
    print("\n[Branch 1: GRU Module]")

    # GRU works directly on (batch, seq_len, features) format
    x_gru = input_layer

    # First GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=True, dropout=0.2, recurrent_dropout=0.2, name='gru_1')(x_gru)

    # Second GRU layer with dropout
    x_gru = layers.GRU(128, return_sequences=False, dropout=0.2, recurrent_dropout=0.2, name='gru_2')(x_gru)

    print(f"  • GRU Layer 1: 128 units (return sequences, dropout=0.2)")
    print(f"  • GRU Layer 2: 128 units (return last, dropout=0.2)")
    print(f"  • Output shape: (batch, 128)")

    # ============================================================
    # BRANCH 2: CNN Module (Spatial Feature Extraction)
    # ============================================================
    print("\n[Branch 2: CNN Module]")

    # Permute for Conv1D: (batch, seq_len, features) -> (batch, features, seq_len)
    x_cnn = layers.Permute((2, 1), name='cnn_permute')(input_layer)

    # Conv Block 1
    x_cnn = layers.Conv1D(32, kernel_size=3, padding='same', name='conv_1')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_1')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_1')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_1')(x_cnn)

    # Conv Block 2
    x_cnn = layers.Conv1D(64, kernel_size=3, padding='same', name='conv_2')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_2')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_2')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_2')(x_cnn)

    # Conv Block 3
    x_cnn = layers.Conv1D(128, kernel_size=3, padding='same', name='conv_3')(x_cnn)
    x_cnn = layers.BatchNormalization(name='bn_3')(x_cnn)
    x_cnn = layers.Activation('relu', name='relu_3')(x_cnn)
    x_cnn = layers.MaxPooling1D(pool_size=2, name='pool_3')(x_cnn)

    # Flatten CNN output
    x_cnn = layers.Flatten(name='cnn_flatten')(x_cnn)

    print(f"  • Conv Block 1: 32 filters")
    print(f"  • Conv Block 2: 64 filters")
    print(f"  • Conv Block 3: 128 filters")
    print(f"  • Each with BatchNorm, ReLU, MaxPool")

    # ============================================================
    # CONCATENATE: Merge GRU and CNN features
    # ============================================================
    print("\n[Feature Fusion]")
    concatenated = layers.Concatenate(name='concatenate')([x_cnn, x_gru])
    print(f"  • Concatenating GRU and CNN features")

    # ============================================================
    # MLP Module (Classification Head)
    # ============================================================
    print("\n[Classification Head: MLP]")

    x = layers.Dense(128, activation='relu', name='fc_1')(concatenated)
    x = layers.Dropout(0.3, name='dropout_1')(x)  # Add dropout after first dense
    x = layers.Dense(64, activation='relu', name='fc_2')(x)
    x = layers.Dropout(0.5, name='dropout_2')(x)  # Keep dropout after second dense

    # Output layer
    output = layers.Dense(num_classes, activation='softmax', name='output')(x)

    print(f"  • Dense Layer 1: 128 units + Dropout(0.3)")
    print(f"  • Dense Layer 2: 64 units + Dropout(0.5)")
    print(f"  • Output Layer: {num_classes} units (softmax)")

    # Create model
    model = models.Model(inputs=input_layer, outputs=output, name='DeepFed')

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    print(f"\n{'='*80}")
    print("MODEL SUMMARY")
    print(f"{'='*80}")
    model.summary()

    total_params = model.count_params()
    print(f"\nTotal parameters: {total_params:,}")
    print(f"{'='*80}")

    return model


class LazyDataGenerator(keras.utils.PyDataset):
    """
    Memory-efficient data generator that loads data in batches from disk
    Similar to tf.data.Dataset for lazy loading
    """
    def __init__(self, data_file, indices, batch_size=None, shuffle=True, **kwargs):
        super().__init__(**kwargs)
        self.data_file = data_file
        self.indices = indices
        self.batch_size = batch_size if batch_size is not None else BATCH_SIZE
        self.shuffle = shuffle
        self.num_samples = len(indices)
        self.num_batches = int(np.ceil(self.num_samples / self.batch_size))

        # Memory-map the files for efficient access
        self.X = np.load(self.data_file['X'], mmap_mode='r')
        self.y = np.load(self.data_file['y'], mmap_mode='r')

    def __len__(self):
        """Number of batches per epoch"""
        return self.num_batches

    def __getitem__(self, idx):
        """Generate one batch of data"""
        # Get batch indices
        start_idx = idx * self.batch_size
        end_idx = min(start_idx + self.batch_size, self.num_samples)
        batch_indices = self.indices[start_idx:end_idx]

        # Load batch from memory-mapped files
        batch_X = self.X[batch_indices].copy()  # Copy to ensure contiguous array
        batch_y = self.y[batch_indices].copy()

        return batch_X, batch_y

    def on_epoch_end(self):
        """Shuffle indices at end of epoch"""
        if self.shuffle:
            np.random.shuffle(self.indices)


def create_data_generator(X, y, batch_size=None, shuffle=True):
    """
    Create a data generator to avoid loading entire dataset in memory
    Legacy function for backward compatibility
    """
    if batch_size is None:
        batch_size = BATCH_SIZE

    num_samples = len(X)
    indices = np.arange(num_samples)

    while True:
        if shuffle:
            np.random.shuffle(indices)

        for start_idx in range(0, num_samples, batch_size):
            end_idx = min(start_idx + batch_size, num_samples)
            batch_indices = indices[start_idx:end_idx]

            batch_x = X[batch_indices]
            batch_y = y[batch_indices]

            yield batch_x, batch_y


def train_model(model, X_train, y_train, X_test, y_test):
    """
    Phase 4: Train the DeepFed model
    """
    print("\n" + "=" * 80)
    print("PHASE 4: TRAINING DEEPFED MODEL")
    print("=" * 80)

    print(f"\nTraining Configuration:")
    print(f"  • Batch size: {BATCH_SIZE}")
    print(f"  • Epochs: {EPOCHS}")
    print(f"  • Learning rate: {LEARNING_RATE}")
    print(f"  • Training samples: {len(X_train):,}")
    print(f"  • Validation samples: {len(X_test):,}")
    print(f"  • Steps per epoch: {len(X_train) // BATCH_SIZE}")

    # Callbacks for training
    model_callbacks = [
        callbacks.ModelCheckpoint(
            filepath=str(Path(MODEL_DIR) / 'deepfed_best.keras'),
            monitor='val_accuracy',  # Monitor validation accuracy for best model
            save_best_only=True,
            save_weights_only=False,  # Save full model (architecture + weights + optimizer state)
            mode='max',  # Maximize validation accuracy
            verbose=1
        ),
        callbacks.EarlyStopping(
            monitor='val_loss',
            patience=15,  # Increased patience for better convergence
            restore_best_weights=True,
            mode='min',  # Minimize validation loss
            verbose=1
        ),
        callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=7,  # Increased to allow more exploration before LR reduction
            min_lr=1e-7,
            mode='min',
            verbose=1
        ),
        callbacks.CSVLogger(
            filename=str(Path(MODEL_DIR) / 'training_log.csv'),
            separator=',',
            append=False
        )
    ]

    print(f"\n{'='*80}")
    print("STARTING TRAINING")
    print(f"{'='*80}\n")

    # Train model
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=model_callbacks,
        verbose=1
    )

    # Plot training history
    print(f"\n{'='*80}")
    print("TRAINING COMPLETED")
    print(f"{'='*80}")

    # Visualize training history
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Accuracy plot
    axes[0].plot(history.history['accuracy'], label='Training')
    axes[0].plot(history.history['val_accuracy'], label='Validation')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Accuracy')
    axes[0].set_title('Model Accuracy', fontweight='bold')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Loss plot
    axes[1].plot(history.history['loss'], label='Training')
    axes[1].plot(history.history['val_loss'], label='Validation')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].set_title('Model Loss', fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'training_history.png', dpi=150, bbox_inches='tight')
    print(f"\n✓ Saved training history: {Path(VIS_DIR) / 'training_history.png'}")
    plt.close()

    return history


def evaluate_model(model, X_test, y_test, label_encoder):
    """
    Phase 5: Evaluate the trained model
    """
    print("\n" + "=" * 80)
    print("PHASE 5: MODEL EVALUATION")
    print("=" * 80)

    print("\nGenerating predictions...")
    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=1)
    y_pred = np.argmax(y_pred_prob, axis=1)

    # Overall metrics
    accuracy = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro')
    f1_weighted = f1_score(y_test, y_pred, average='weighted')

    print(f"\n{'='*80}")
    print("OVERALL METRICS")
    print(f"{'='*80}")
    print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
    print("\nNOTE: Accuracy can be misleading in this highly imbalanced dataset.")
    print("      Focus on metrics like F1-score, precision, and recall,")
    print("      especially for minority classes, as shown in the Classification Report.")
    print("-" * 80)
    print(f"F1-Score (macro):    {f1_macro:.4f}")
    print(f"F1-Score (weighted): {f1_weighted:.4f}")
    print(f"{'='*80}")

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 80)
    # Convert label_encoder.classes_ (which are numpy.int8) to strings
    target_names_str = [str(cls) for cls in label_encoder.classes_]
    print(classification_report(
        y_test, y_pred,
        target_names=target_names_str,
        digits=4
    ))

    # Per-class metrics
    print("\nPER-CLASS ACCURACY")
    print("-" * 80)
    for i, class_name in enumerate(label_encoder.classes_):
        mask = y_test == i
        if mask.sum() > 0:
            class_acc = accuracy_score(y_test[mask], y_pred[mask])
            print(f"{str(class_name):30s}: {class_acc:.4f} ({class_acc*100:5.2f}%) [{mask.sum():>6,} samples]")

    # Confusion matrix
    print("\nCONFUSION MATRIX")
    print("-" * 80)
    cm = confusion_matrix(y_test, y_pred)

    # Visualize confusion matrix
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=label_encoder.classes_,
                yticklabels=label_encoder.classes_,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontweight='bold')
    plt.ylabel('Actual', fontweight='bold')
    plt.title('Confusion Matrix', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'confusion_matrix.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved confusion matrix: {Path(VIS_DIR) / 'confusion_matrix.png'}")
    plt.close()

    # Save confusion matrix
    np.save(Path(MODEL_DIR) / 'confusion_matrix.npy', cm)

    return accuracy, f1_macro, f1_weighted


def save_model_artifacts(model, label_encoder, scaler, metadata):
    """
    Save all model artifacts
    """
    print("\n" + "=" * 80)
    print("SAVING MODEL ARTIFACTS")
    print("=" * 80)

    # Ensure model directory exists
    Path(MODEL_DIR).mkdir(parents=True, exist_ok=True)

    # Save full model
    model_path = Path(MODEL_DIR) / 'deepfed_final.keras'
    model.save(model_path)
    print(f"✓ Model saved: {model_path}")

    # Save label encoder
    le_path = Path(MODEL_DIR) / 'label_encoder.pkl'
    with open(le_path, 'wb') as f:
        pickle.dump(label_encoder, f)
    print(f"✓ Label encoder saved: {le_path}")

    # Save scaler
    scaler_path = Path(MODEL_DIR) / 'scaler.pkl'
    with open(scaler_path, 'wb') as f:
        pickle.dump(scaler, f)
    print(f"✓ Scaler saved: {scaler_path}")

    # Save metadata
    meta_path = Path(MODEL_DIR) / 'model_metadata.json'
    with open(meta_path, 'w') as f:
        json.dump(metadata, f, indent=2)
    print(f"✓ Metadata saved: {meta_path}")

    print(f"\n{'='*80}")
    print("ALL ARTIFACTS SAVED")
    print(f"{'='*80}")


# ============================================================================
# COMPARISON FUNCTIONS FOR IMBALANCED DATASET HANDLING
# ============================================================================

def train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 1: Baseline - No imbalance handling techniques applied.

    This approach trains the DeepFed model on the original imbalanced data
    without any resampling or class weighting. Serves as the baseline for
    comparison with other imbalance handling techniques.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes

    Returns:
        Tuple of (model, training_history, predictions)
    """
    print("\n" + "="*80)
    print("APPROACH 1: BASELINE (No Imbalance Handling)")
    print("="*80)

    print(f"\nFlat training data shape: {X_train_flat.shape}")
    print("Class distribution in flat data:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Create sliding window sequences from flat training data
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes):
    """
    Approach 2: Class Weights - Penalize misclassification of minority classes.

    This approach uses scikit-learn's compute_class_weight with 'balanced' strategy
    to assign higher weights to minority classes during training. The loss function
    penalizes misclassification of rare classes more heavily, encouraging the model
    to pay more attention to them.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes

    Returns:
        Tuple of (model, training_history, predictions)
    """
    print("\n" + "="*80)
    print("APPROACH 2: CLASS WEIGHTS")
    print("="*80)

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from original flat data (no resampling)
    print("\nCreating time-series sequences from original data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_flat, y_train_flat,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")

    # Compute class weights from sequence-level labels
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(y_train_sequences),
        y=y_train_sequences
    )
    class_weight_dict = {i: weight for i, weight in enumerate(class_weights)}

    print("\nComputed class weights (based on sequence labels):")
    for cls, weight in class_weight_dict.items():
        print(f"  Class {cls}: {weight:.4f}")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        class_weight=class_weight_dict,
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 3: SMOTE Oversampling - Generate synthetic samples for minority classes.

    SMOTE (Synthetic Minority Over-sampling Technique) creates synthetic samples
    by interpolating between existing minority class samples. This balances the
    dataset by increasing representation of rare classes. SMOTE is applied to
    flat data first, then sequences are created from the resampled data.

    Note: Synthetic samples may not preserve true temporal patterns, but this
    approach is valuable for comparing imbalance handling techniques.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if SMOTE unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 3: SMOTE OVERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before SMOTE:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE on flat data
    print("\nApplying SMOTE on flat data...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=min(5, counts.min() - 1) if counts.min() > 1 else 1)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter SMOTE: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after SMOTE:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 4: Random Undersampling - Reduce majority class samples to balance dataset.

    RandomUnderSampler randomly removes samples from majority classes until all
    classes have equal representation. This balances the dataset but reduces total
    training data, which may hurt performance if minority classes have very few samples.

    Undersampling is applied to flat data first, then sequences are created from
    the resampled data.

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if undersampling unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 4: RANDOM UNDERSAMPLING")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before undersampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply random undersampling on flat data
    print("\nApplying random undersampling on flat data...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE)
    X_train_resampled, y_train_resampled = rus.fit_resample(X_train_flat, y_train_flat)

    print(f"\nAfter undersampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after undersampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None):
    """
    Approach 5: Combined SMOTE + Random Undersampling - Hybrid resampling strategy.

    This approach combines the benefits of both oversampling and undersampling:
    1. First, SMOTE generates synthetic samples for minority classes
    2. Then, RandomUnderSampler reduces majority class samples

    This hybrid approach often outperforms using either technique alone, as it:
    - Increases minority class representation (via SMOTE)
    - Reduces majority class dominance (via undersampling)
    - Results in a more balanced dataset without excessive synthetic data

    Note: We use RandomUnderSampler instead of Tomek links because Tomek links
    are 10-100x slower with minimal accuracy improvement (<1%).

    Args:
        X_train_flat: Flat training data (samples x features)
        y_train_flat: Training labels (samples,)
        X_test: Test sequences (already sequenced) (sequences x timesteps x features)
        y_test: Test labels (sequences,)
        num_classes: Number of output classes
        target_samples: Optional target number of samples per class (unused)

    Returns:
        Tuple of (model, training_history, predictions) or (None, None, None) if resampling unavailable
    """
    print("\n" + "="*80)
    print("APPROACH 5: COMBINED (SMOTE + Random UnderSampling)")
    print("="*80)

    if not IMBLEARN_AVAILABLE:
        print("ERROR: imbalanced-learn not available. Skipping this approach.")
        return None, None, None

    print(f"\nOriginal flat training data shape: {X_train_flat.shape}")
    print("Class distribution before resampling:")
    unique, counts = np.unique(y_train_flat, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_flat)*100:5.2f}%)")

    # Apply SMOTE first to oversample minority classes
    print("\n[Step 1/2] Applying SMOTE to oversample minority classes...")
    smote = SMOTE(random_state=RANDOM_STATE, k_neighbors=5)
    X_smote, y_smote = smote.fit_resample(X_train_flat, y_train_flat)

    print(f"After SMOTE: {X_smote.shape[0]:,} samples")
    unique, counts = np.unique(y_smote, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_smote)*100:5.2f}%)")

    # Apply undersampling to reduce majority class (faster than Tomek, better balance)
    print("\n[Step 2/2] Applying random undersampling for final balance...")
    rus = RandomUnderSampler(random_state=RANDOM_STATE, sampling_strategy='auto')
    X_train_resampled, y_train_resampled = rus.fit_resample(X_smote, y_smote)

    print(f"\nAfter SMOTE+UnderSampling: {X_train_resampled.shape[0]:,} samples")
    print("Class distribution after combined resampling:")
    unique, counts = np.unique(y_train_resampled, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_resampled)*100:5.2f}%)")

    # 🕐 TIME-SERIES: Create sequences from resampled flat data
    print("\nCreating time-series sequences from resampled data...")
    X_train_sequences, y_train_sequences = create_sequences(
        X_train_resampled, y_train_resampled,
        time_steps=SEQUENCE_LENGTH, stride=WINDOW_STRIDE
    )

    print(f"Created {len(X_train_sequences):,} sequences with shape {X_train_sequences.shape}")
    print("Class distribution in sequences:")
    unique, counts = np.unique(y_train_sequences, return_counts=True)
    for cls, count in zip(unique, counts):
        print(f"  Class {cls}: {count:>7,} ({count/len(y_train_sequences)*100:5.2f}%)")

    model = build_deepfed_model(input_shape=X_train_sequences.shape[1:], num_classes=num_classes)

    early_stop = callbacks.EarlyStopping(
        monitor='val_loss', patience=15, restore_best_weights=True, verbose=1
    )

    history = model.fit(
        X_train_sequences, y_train_sequences,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_test, y_test),
        callbacks=[early_stop],
        verbose=1
    )

    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)

    return model, history, y_pred_prob


def evaluate_and_compare(predictions_list, histories, X_test, y_test, label_encoder, approach_names):
    """
    Evaluate all approaches and create comparison visualizations
    """
    print("\n" + "="*80)
    print("COMPARISON EVALUATION")
    print("="*80)

    results = []

    for i, (y_pred_prob, history, approach) in enumerate(zip(predictions_list, histories, approach_names)):
        if y_pred_prob is None:
            print(f"\nSkipping {approach} (not available)")
            continue

        print(f"\n{'-'*80}")
        print(f"Evaluating: {approach}")
        print(f"{'-'*80}")

        # Predictions (already generated during training)
        y_pred = np.argmax(y_pred_prob, axis=1)

        # Compute metrics
        accuracy = accuracy_score(y_test, y_pred)
        precision, recall, f1, support = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )

        # Aggregate metrics
        macro_f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)
        weighted_f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        macro_precision = np.mean(precision)
        macro_recall = np.mean(recall)

        print(f"Accuracy:          {accuracy:.4f}")
        print(f"Macro F1:          {macro_f1:.4f}")
        print(f"Weighted F1:       {weighted_f1:.4f}")
        print(f"Macro Precision:   {macro_precision:.4f}")
        print(f"Macro Recall:      {macro_recall:.4f}")

        results.append({
            'approach': approach,
            'accuracy': accuracy,
            'macro_f1': macro_f1,
            'weighted_f1': weighted_f1,
            'macro_precision': macro_precision,
            'macro_recall': macro_recall,
            'per_class_f1': f1.tolist(),
            'per_class_precision': precision.tolist(),
            'per_class_recall': recall.tolist(),
            'per_class_support': support.tolist(),
            'confusion_matrix': confusion_matrix(y_test, y_pred).tolist()
        })

    # Create comparison visualizations
    if len(results) > 0:
        create_comparison_plots(results, label_encoder)
        save_comparison_results(results)

    return results


def cleanup_model_memory(model, approach_name, save_model=True):
    """
    Clean up GPU/CPU memory after training to free resources

    For proper GPU memory management in Keras:
    1. Save the model immediately after training
    2. Delete the model object to free GPU memory
    3. Run garbage collection

    Args:
        model: Trained Keras model
        approach_name: Name of the training approach (for logging)
        save_model: Whether to save the full model (default True)
    """
    if model is None:
        return

    try:
        # Save the full model immediately to preserve it
        if save_model:
            model_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_model.keras"
            model_path.parent.mkdir(parents=True, exist_ok=True)
            model.save(str(model_path))
            print(f"✓ Model saved: {model_path}")

        # Clear Keras backend session
        try:
            import keras.backend as K
            K.clear_session()
        except ImportError:
            try:
                import tensorflow as tf
                tf.keras.backend.clear_session()
            except:
                print(f"⚠️  Could not clear Keras session for {approach_name}")

        # Delete the model object to free GPU memory
        del model

        # Force garbage collection
        gc.collect()

        print(f"✓ GPU/CPU memory freed for: {approach_name}")

    except Exception as e:
        print(f"⚠️  Warning during cleanup for {approach_name}: {e}")


def load_saved_model(approach_name):
    """
    Load a previously saved model for evaluation

    Args:
        approach_name: Name of the training approach

    Returns:
        Loaded Keras model or None if not found
    """
    try:
        model_path = Path(MODEL_DIR) / f"{approach_name.lower().replace(' ', '_')}_model.keras"
        if model_path.exists():
            model = keras.models.load_model(str(model_path))
            print(f"✓ Model loaded: {model_path}")
            return model
        else:
            print(f"⚠️  Model not found: {model_path}")
            return None
    except Exception as e:
        print(f"⚠️  Error loading model {approach_name}: {e}")
        return None


def create_comparison_plots(results, label_encoder):
    """
    Create comprehensive comparison visualizations
    """
    print("\n" + "="*80)
    print("CREATING COMPARISON VISUALIZATIONS")
    print("="*80)

    approaches = [r['approach'] for r in results]

    # 1. Overall Metrics Comparison
    fig, axes = plt.subplots(2, 2, figsize=(15, 12))

    metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision']
    metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score', 'Macro Precision']

    for idx, (metric, name) in enumerate(zip(metrics, metric_names)):
        ax = axes[idx // 2, idx % 2]
        values = [r[metric] for r in results]
        bars = ax.bar(approaches, values, color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
        ax.set_ylabel(name, fontweight='bold')
        ax.set_ylim([0, 1.05])
        ax.set_title(f'{name} Comparison', fontweight='bold')
        ax.grid(axis='y', alpha=0.3)

        # Add value labels on bars
        for bar in bars:
            height = bar.get_height()
            ax.text(bar.get_x() + bar.get_width()/2., height,
                   f'{height:.3f}', ha='center', va='bottom', fontsize=9)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_overall_metrics.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_overall_metrics.png'}")
    plt.close()

    # 2. Per-Class F1-Score Comparison
    num_classes = len(results[0]['per_class_f1'])
    x = np.arange(num_classes)
    width = 0.15

    fig, ax = plt.subplots(figsize=(16, 8))

    colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']
    for i, result in enumerate(results):
        offset = width * (i - len(results)/2 + 0.5)
        ax.bar(x + offset, result['per_class_f1'], width,
               label=result['approach'], color=colors[i % len(colors)])

    ax.set_xlabel('Class', fontweight='bold')
    ax.set_ylabel('F1-Score', fontweight='bold')
    ax.set_title('Per-Class F1-Score Comparison', fontweight='bold', fontsize=14)
    ax.set_xticks(x)
    ax.set_xticklabels([str(c) for c in label_encoder.classes_], rotation=45, ha='right')
    ax.legend(loc='upper right')
    ax.grid(axis='y', alpha=0.3)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_per_class_f1.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_per_class_f1.png'}")
    plt.close()

    # 3. Macro Recall Comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    values = [r['macro_recall'] for r in results]
    bars = ax.bar(approaches, values,
                  color=['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd'][:len(approaches)])
    ax.set_ylabel('Macro Recall', fontweight='bold')
    ax.set_ylim([0, 1.05])
    ax.set_title('Macro Recall Comparison (Minority Class Performance)',
                fontweight='bold', fontsize=12)
    ax.grid(axis='y', alpha=0.3)

    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
               f'{height:.3f}', ha='center', va='bottom', fontsize=10)

    plt.tight_layout()
    plt.savefig(Path(VIS_DIR) / 'comparison_macro_recall.png', dpi=150, bbox_inches='tight')
    print(f"✓ Saved: {Path(VIS_DIR) / 'comparison_macro_recall.png'}")
    plt.close()


def save_comparison_results(results):
    """
    Save comparison results to JSON and generate text summary
    """
    # Save JSON
    json_path = Path(MODEL_DIR) / 'comparison_results.json'
    with open(json_path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f"✓ Saved: {json_path}")

    # Create text summary
    summary_path = Path(MODEL_DIR) / 'comparison_summary.txt'
    with open(summary_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("IMBALANCED DATASET HANDLING COMPARISON SUMMARY\n")
        f.write("="*80 + "\n\n")

        for result in results:
            f.write(f"\n{result['approach']}\n")
            f.write("-"*80 + "\n")
            f.write(f"  Accuracy:          {result['accuracy']:.4f}\n")
            f.write(f"  Macro F1-Score:    {result['macro_f1']:.4f}\n")
            f.write(f"  Weighted F1-Score: {result['weighted_f1']:.4f}\n")
            f.write(f"  Macro Precision:   {result['macro_precision']:.4f}\n")
            f.write(f"  Macro Recall:      {result['macro_recall']:.4f}\n")

        # Best approach per metric
        f.write("\n" + "="*80 + "\n")
        f.write("BEST APPROACHES PER METRIC\n")
        f.write("="*80 + "\n")

        metrics = ['accuracy', 'macro_f1', 'weighted_f1', 'macro_precision', 'macro_recall']
        metric_names = ['Accuracy', 'Macro F1-Score', 'Weighted F1-Score',
                       'Macro Precision', 'Macro Recall']

        for metric, name in zip(metrics, metric_names):
            best = max(results, key=lambda x: x[metric])
            f.write(f"\n{name:20s}: {best['approach']:30s} ({best[metric]:.4f})\n")

    print(f"✓ Saved: {summary_path}")


def main():
    """
    Main execution pipeline with efficient data loading and caching
    """
    print("\n" + "=" * 80)
    print("DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM")
    print("Dataset: Edge-IIoTset")
    print("Model: GRU + CNN (Time-Series Architecture)")
    print("=" * 80)

    print(f"\nConfiguration:")
    print(f"  • Classification: {'Multi-class (attack types)' if USE_MULTICLASS else 'Binary (normal vs attack)'}")
    print(f"  • Dataset: {'Full dataset' if SAMPLE_SIZE is None else f'Sampled ({SAMPLE_SIZE:,} rows/file)'}")
    print(f"  • Use cached data: {USE_CACHED_DATA}")
    print(f"  • Sequence length: {SEQUENCE_LENGTH} time steps")
    print(f"  • Window stride: {WINDOW_STRIDE} (for sliding window)")
    print(f"  • Batch size: {BATCH_SIZE} (fixed)")
    print(f"  • Epochs: {EPOCHS} (with early stopping)")
    print(f"  • Learning rate: {LEARNING_RATE}")

    # Check what's already cached
    cached_sequences = all(Path(CACHE_DIR, f).exists() for f in ['X_train.npy', 'X_test.npy', 'y_train.npy', 'y_test.npy'])
    cached_hdf5 = HDF5_DATASET.exists()

    print(f"\nCache Status:")
    print(f"  • HDF5 preprocessed: {'✓ Found' if cached_hdf5 else '✗ Not found'}")
    print(f"  • Sequence arrays: {'✓ Found' if cached_sequences else '✗ Not found'}")
    if cached_sequences and USE_CACHED_DATA:
        print(f"  → Will skip CSV parsing and sequence creation!")

    try:
        # Check if dataset exists
        csv_exists = any(Path(DATA_DIR).rglob("*.csv"))
        if not csv_exists:
            print("\nDataset not found. Downloading...")
            if not download_dataset():
                print("\n✗ Download failed. Please download manually:")
                print(f"   https://www.kaggle.com/datasets/{DATASET_NAME}")
                return 1

        # Phase 1 & 2: Load and preprocess dataset
        X_train_flat, X_test, y_train_flat, y_test, le, scaler, num_classes = prepare_time_series_data()

        print(f"\n✅ Data loaded successfully:")
        print(f"  • Training samples (flat): {X_train_flat.shape[0]:,}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Features: {X_train_flat.shape[1]}")
        print(f"  • Classes: {num_classes}")
        print(f"  • Class names: {le.classes_}")
        print(f"  • Batch size: {BATCH_SIZE}")

        # Phase 3: Train and evaluate all 5 comparison approaches
        print("\n" + "="*80)
        print("PHASE 3-5: COMPARISON STUDY")
        print("Training 5 Different Approaches for Handling Imbalanced Data")
        print("="*80)

        approach_names = [
            "Baseline (No Handling)",
            "Class Weights",
            "SMOTE Oversampling",
            "Random Undersampling",
            "Combined (SMOTE+UnderSample)"
        ]

        # All approaches will work with the same flat training data
        # Each approach will apply its resampling strategy, then create sequences
        print(f"\n📊 Training data prepared:")
        print(f"  • Flat training samples: {X_train_flat.shape[0]:,}")
        print(f"  • Features per sample: {X_train_flat.shape[1]}")
        print(f"  • Test sequences: {X_test.shape[0]:,}")
        print(f"  • Sequence shape: ({X_test.shape[1]}, {X_test.shape[2]})")

        models = []
        histories = []
        predictions_list = []

        # Helper: the exact identifiers we saved models under in cleanup_model_memory
        saved_identifiers = [
            "Baseline",
            "Class_Weights",
            "SMOTE_Oversampling",
            "Random_Undersampling",
            "Combined_SMOTE_Tomek"
        ]

        if USE_PRETRAINED:
            # Load pre-saved models from MODEL_DIR, produce predictions without retraining
            print("\nUsing pre-trained models from disk; skipping training for all approaches.")
            for display_name, ident in zip(approach_names, saved_identifiers):
                print(f"\nLoading model for: {display_name} (id: {ident})")
                model = load_saved_model(ident)
                if model is None:
                    print(f"  ✗ Saved model for {ident} not found. Skipping this approach.")
                    models.append(ident)
                    histories.append(None)
                    predictions_list.append(None)
                    continue

                # Make predictions on test set
                try:
                    y_pred_prob = model.predict(X_test, batch_size=BATCH_SIZE, verbose=0)
                except Exception as e:
                    print(f"  ⚠️  Prediction failed for {ident}: {e}")
                    y_pred_prob = None

                models.append(ident)
                histories.append(None)
                predictions_list.append(y_pred_prob)

                # Clean up the loaded model to free GPU memory
                cleanup_model_memory(model, ident, save_model=False)

        else:
            # Train each approach sequentially, save and delete model after each
            # Approach 1: Baseline
            print("\n" + "█"*80)
            print("█ APPROACH 1/5: BASELINE")
            print("█"*80)
            model1, hist1, pred1 = train_baseline(X_train_flat, y_train_flat, X_test, y_test, num_classes)
            models.append("Baseline")  # Store approach name instead of model object
            histories.append(hist1)
            predictions_list.append(pred1)
            if model1 is not None:
                cleanup_model_memory(model1, "Baseline")  # Saves and deletes model

            # Approach 2: Class Weights
            print("\n" + "█"*80)
            print("█ APPROACH 2/5: CLASS WEIGHTS")
            print("█"*80)
            model2, hist2, pred2 = train_with_class_weights(X_train_flat, y_train_flat, X_test, y_test, num_classes)
            models.append("Class_Weights")  # Store approach name
            histories.append(hist2)
            predictions_list.append(pred2)
            if model2 is not None:
                cleanup_model_memory(model2, "Class_Weights")  # Saves and deletes model

            # Approach 3: SMOTE Oversampling
            print("\n" + "█"*80)
            print("█ APPROACH 3/5: SMOTE OVERSAMPLING")
            print("█"*80)
            model3, hist3, pred3 = train_with_oversampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
            models.append("SMOTE_Oversampling")  # Store approach name
            histories.append(hist3)
            predictions_list.append(pred3)
            if model3 is not None:
                cleanup_model_memory(model3, "SMOTE_Oversampling")  # Saves and deletes model

            # Approach 4: Random Undersampling
            print("\n" + "█"*80)
            print("█ APPROACH 4/5: RANDOM UNDERSAMPLING")
            print("█"*80)
            model4, hist4, pred4 = train_with_undersampling(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
            models.append("Random_Undersampling")  # Store approach name
            histories.append(hist4)
            predictions_list.append(pred4)
            if model4 is not None:
                cleanup_model_memory(model4, "Random_Undersampling")  # Saves and deletes model

            # Approach 5: Combined SMOTE + UnderSampling
            print("\n" + "█"*80)
            print("█ APPROACH 5/5: COMBINED (SMOTE + UNDERSAMPLING)")
            print("█"*80)
            model5, hist5, pred5 = train_with_combined(X_train_flat, y_train_flat, X_test, y_test, num_classes, target_samples=None)
            models.append("Combined_SMOTE_Tomek")  # Store approach name
            histories.append(hist5)
            predictions_list.append(pred5)
            if model5 is not None:
                cleanup_model_memory(model5, "Combined_SMOTE_Tomek")  # Saves and deletes model

        # Evaluate and compare all approaches
        print("\n" + "="*80)
        print("ALL TRAINING COMPLETED - STARTING COMPARISON EVALUATION")
        print("="*80)

        results = evaluate_and_compare(predictions_list, histories, X_test, y_test, le, approach_names)

        # Save metadata
        metadata = {
            'model_name': 'DeepFed Comparison Study',
            'dataset': 'Edge-IIoTset',
            'architecture': 'GRU + CNN + MLP',
            'sequence_length': SEQUENCE_LENGTH,
            'num_features': X_test.shape[2],  # Use test sequences shape
            'num_classes': num_classes,
            'class_names': le.classes_.tolist(),
            'approaches': approach_names,
            'comparison_results': results,
            'batch_size': BATCH_SIZE,
            'learning_rate': LEARNING_RATE,
            'epochs': EPOCHS
        }

        # Save best model (highest macro F1)
        if results:
            best_idx = max(range(len(results)), key=lambda i: results[i]['macro_f1'])
            best_approach_name = models[best_idx]  # Get approach name
            best_model = load_saved_model(best_approach_name)  # Load the saved model
            best_approach = approach_names[best_idx]

            print(f"\n{'='*80}")
            print(f"BEST APPROACH: {best_approach}")
            print(f"  Macro F1-Score: {results[best_idx]['macro_f1']:.4f}")
            print(f"{'='*80}")

            if best_model is not None:
                save_model_artifacts(best_model, le, scaler, metadata)
                # Clean up the reloaded model too
                cleanup_model_memory(best_model, f"Best_{best_approach_name}", save_model=False)
            else:
                print("⚠️  Could not load best model for final saving")

        print("\n" + "=" * 80)
        print("✓ COMPARISON STUDY COMPLETED SUCCESSFULLY")
        print("=" * 80)
        print(f"\nResults Summary:")
        for i, (name, result) in enumerate(zip(approach_names, results)):
            print(f"\n{i+1}. {name}")
            print(f"   Accuracy:   {result['accuracy']:.4f}")
            print(f"   Macro F1:   {result['macro_f1']:.4f}")
            print(f"   Macro Recall: {result['macro_recall']:.4f}")
        print(f"\n  • Results saved in:     {MODEL_DIR}/")
        print(f"  • Visualizations in:    {VIS_DIR}/")
        print("=" * 80)

        return 0

    except Exception as e:
        print(f"\n{'='*80}")
        print(f"✗ ERROR: {e}")
        print(f"{'='*80}")
        import traceback
        traceback.print_exc()
        return 1


if __name__ == "__main__":
    exit_code = main()
    sys.exit(exit_code)

TensorFlow available for GPU detection

DEEPFED: TIME-SERIES INTRUSION DETECTION SYSTEM
Dataset: Edge-IIoTset
Model: GRU + CNN (Time-Series Architecture)

Configuration:
  • Classification: Multi-class (attack types)
  • Dataset: Sampled (100,000 rows/file)
  • Use cached data: True
  • Sequence length: 128 time steps
  • Window stride: 64 (for sliding window)
  • Batch size: 512 (fixed)
  • Epochs: 100 (with early stopping)
  • Learning rate: 0.001

Cache Status:
  • HDF5 preprocessed: ✗ Not found
  • Sequence arrays: ✗ Not found

Dataset not found. Downloading...

DOWNLOADING EDGE-IIOTSET DATASET FROM KAGGLE
Dataset not found locally - proceeding with download...
✓ Running in Google Colab - using secrets
  • Username: pctablet505
  • API Key: ********************************

Extracting edgeiiotset-cyber-security-dataset-of-iot-iiot.zip...
✓ Dataset downloaded and extracted successfully!

PHASE 2: TIME-SERIES DATA PREPARATION

CONVERTING CSV TO EFFICIENT BINARY FORMAT

✓ Found 26 CSV

SystemExit: 0

In [ ]:
!rm -rf /content/balancedfl_repo/cache
!rm -rf /content/balancedfl_repo/data

In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Stage model weights, results, and comparison files
!git add ./models/deepfed/*_weights.h5
!git add ./models/deepfed/comparison_results.json
!git add ./models/deepfed/comparison_summary.txt
!git add ./visualizations/*

# Commit the changes
!git commit -m "Add trained model weights and comparison results"

# Push the changes to the remote repository (e.g., 'origin' branch 'main')
# You might need to configure your Git credentials first.
!git push origin main